In [ ]:
pip install yfinance -q

# ver.1 (Step 2 bugged)

In [ ]:
# =============================================================================
# SHARIAH-COMPLIANT AI ROBO-ADVISOR — FULL PIPELINE (single-file, Colab-ready)
# =============================================================================
# Paste this entire file into one Google Colab cell (or split it across cells
# at the "STAGE" section breaks below) and run.
#
# Pipeline:
#   1. Data Ingestion & Preprocessing (5y prices/volume/fundamentals + technical indicators)
#   2. Shariah Universe Screening (business activity + financial ratios)
#   3. AI Return Forecasting (per-asset ANN via scikit-learn MLPRegressor)
#   4. Investor Profiling (robo-advisor risk questionnaire)
#   5. Constrained Optimisation (mean-CVaR, transaction costs, holding limits, zakat)
#   6. Portfolio Output
#
# NOTE ON DATA: this script tries a live pull via `yfinance` first (per-ticker,
# so one bad symbol won't sink the whole pull) and validates real data came
# back before trusting it. Fundamentals (debt/assets/cash/receivables) are
# read from the balance-sheet statement rather than yfinance's flat `.info`
# dict, since `.info` frequently lacks those fields for non-US listings even
# when it resolves fine otherwise. If yfinance/network isn't available, or too
# few tickers return usable data, it automatically falls back to a realistic
# 5-year SYNTHETIC dataset with an identical schema, so the pipeline still
# runs end-to-end either way.
#
# Bursa Malaysia tickers on Yahoo Finance use NUMERIC stock codes + ".KL"
# (e.g. Maybank = "1155.KL", not "MAYBANK.KL") -- the universe below already
# uses the correct codes; re-verify periodically as codes/listings can change.
# =============================================================================

# --- Run this in Colab if yfinance isn't preinstalled (safe to skip/fail) ---
# !pip install yfinance -q

# --- Shared imports for the whole pipeline ---
from __future__ import annotations
import sys
import warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from scipy.optimize import minimize
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ==============================================================================
# STAGE 1a  |  DATA INGESTION (5-YEAR HORIZON)
# (originally: data_ingestion.py)
# ==============================================================================

HORIZON_YEARS = 5
TRADING_DAYS_PER_YEAR = 252

# A representative universe: large, liquid Bursa Malaysia counters spanning
# sectors that are typically screened (some pass, some fail Shariah screens),
# so downstream Stage 2 has something real to filter.
#
# IMPORTANT: Yahoo Finance (and therefore yfinance) identifies Bursa Malaysia
# counters by their NUMERIC stock code + ".KL", not by the name-style ticker
# used on Bursa's own trading terminals (e.g. Maybank trades as "MAYBANK" on
# Bursa but is "1155.KL" on Yahoo). Using name-style tickers here is what
# causes every download to 404. Codes below verified against Yahoo Finance /
# Bursa Malaysia listings; re-verify periodically as codes/listings can change
# (e.g. after mergers such as Digi + Celcom -> CelcomDigi, same code 6947.KL).
DEFAULT_UNIVERSE = [
    # yahoo_ticker, company, sector  (sector drives the business-activity screen)
    ("1155.KL", "Malayan Banking Bhd (Maybank)", "Conventional Banking"),   # excluded (riba)
    ("1295.KL", "Public Bank Bhd", "Conventional Banking"),                 # excluded (riba)
    ("5347.KL", "Tenaga Nasional Bhd", "Utilities"),
    ("6033.KL", "Petronas Gas Bhd", "Energy"),
    ("5183.KL", "Petronas Chemicals Group Bhd", "Materials"),
    ("1961.KL", "IOI Corp Bhd", "Plantation"),
    ("2445.KL", "Kuala Lumpur Kepong Bhd", "Plantation"),
    ("4707.KL", "Nestle Malaysia Bhd", "Consumer Staples"),
    ("3689.KL", "Fraser & Neave Holdings Bhd", "Consumer Staples"),
    ("3026.KL", "Dutch Lady Milk Industries Bhd", "Consumer Staples"),
    ("7113.KL", "Top Glove Corp Bhd", "Health Care Equipment"),
    ("5168.KL", "Hartalega Holdings Bhd", "Health Care Equipment"),
    ("3182.KL", "Genting Bhd", "Gaming & Casinos"),                         # excluded (maysir)
    ("4715.KL", "Genting Malaysia Bhd", "Gaming & Casinos"),                # excluded (maysir)
    ("6888.KL", "Axiata Group Bhd", "Telecommunications"),
    ("6947.KL", "CelcomDigi Bhd (fka Digi.Com)", "Telecommunications"),
    ("6012.KL", "Maxis Bhd", "Telecommunications"),
    ("5285.KL", "SD Guthrie Bhd (fka Sime Darby Plantation)", "Plantation"),
    ("8869.KL", "Press Metal Aluminium Holdings Bhd", "Materials"),
    ("3816.KL", "MISC Bhd", "Shipping/Logistics"),
    ("2836.KL", "Carlsberg Brewery Malaysia Bhd", "Brewery"),               # excluded (alcohol)
    ("3255.KL", "Heineken Malaysia Bhd", "Brewery"),                       # excluded (alcohol)
    ("4677.KL", "YTL Corp Bhd", "Conglomerate/Utilities"),
    ("5211.KL", "Sunway Bhd", "Property & Construction"),
    ("3336.KL", "IJM Corp Bhd", "Property & Construction"),
]

MIN_VALID_TICKERS = 5  # below this, treat a "live" pull as failed -> fall back to synthetic


@dataclass
class MarketDataset:
    prices: pd.DataFrame          # index=date, columns=ticker -> adjusted close
    volumes: pd.DataFrame         # index=date, columns=ticker -> volume
    fundamentals: pd.DataFrame    # index=ticker -> latest fundamentals snapshot
    meta: pd.DataFrame            # index=ticker -> name, sector
    start_date: datetime = field(default=None)
    end_date: datetime = field(default=None)


# Yahoo's generic "sector" (e.g. "Financial Services") is too coarse for a
# Shariah business-activity screen, so live ingestion instead keyword-matches
# the more specific "industry" string. This is a best-effort classifier for
# demo purposes -- a production system should validate against the SC
# Malaysia List of Shariah-Compliant Securities rather than rely on this alone.
NON_COMPLIANT_INDUSTRY_KEYWORDS = [
    "bank", "insurance", "credit services", "capital markets",  # riba
    "gambling", "resorts & casinos", "casino",                   # maysir
    "brewers", "distillers", "wineries", "beverages - wineries",  # alcohol
    "tobacco",
    "aerospace & defense",
    "adult",  # adult entertainment (rarely tagged distinctly by Yahoo)
]


def _classify_industry(industry: str) -> str:
    """Maps a raw Yahoo `industry` string to a coarse label. If any excluded
    keyword matches, returns that keyword's category name (which is also
    listed in NON_COMPLIANT_SECTORS in shariah_screening.py); otherwise
    passes the original industry through unchanged."""
    ind_lower = (industry or "").lower()
    for kw in NON_COMPLIANT_INDUSTRY_KEYWORDS:
        if kw in ind_lower:
            return f"Excluded ({kw})"
    return industry or "Unknown"


# .info's flat fields (totalDebt/totalAssets/totalCash/netReceivables) are
# frequently just absent for non-US listings even when .info itself resolves
# fine (sector/industry, price, market cap etc. come through normally) --
# that data lives in the balance-sheet statement instead, so we read it from
# there with flexible row-name matching (Yahoo's standardized line-item names
# have shifted over the years and vary slightly by market).
_TOTAL_ASSETS_KEYS = ["Total Assets"]
_TOTAL_DEBT_KEYS = ["Total Debt"]
_LONG_TERM_DEBT_KEYS = ["Long Term Debt"]
_CURRENT_DEBT_KEYS = ["Current Debt", "Current Debt And Capital Lease Obligation"]
_CASH_KEYS = ["Cash Cash Equivalents And Short Term Investments", "Cash And Cash Equivalents"]
_RECEIVABLES_KEYS = ["Receivables", "Accounts Receivable"]


def _bs_lookup(balance_sheet: pd.DataFrame, candidates: list[str]) -> float:
    """Looks up the most recent value of the first matching row-name
    candidate in a yfinance balance_sheet/quarterly_balance_sheet DataFrame
    (index = line items, columns = period-end dates, most recent first)."""
    if balance_sheet is None or balance_sheet.empty:
        return np.nan
    col = balance_sheet.columns[0]
    idx_lower = {str(i).strip().lower(): i for i in balance_sheet.index}
    for cand in candidates:
        key = cand.strip().lower()
        if key in idx_lower:
            val = balance_sheet.loc[idx_lower[key], col]
            if pd.notna(val):
                return float(val)
        for lower_name, orig_name in idx_lower.items():
            if key in lower_name:
                val = balance_sheet.loc[orig_name, col]
                if pd.notna(val):
                    return float(val)
    return np.nan


def _fetch_fundamentals(ticker_obj) -> dict:
    """Pulls total_assets/total_debt/cash/receivables from the balance sheet
    (quarterly preferred for recency, annual as fallback) and market cap from
    the lightweight fast_info (falls back to .info)."""
    balance_sheet = None
    try:
        balance_sheet = ticker_obj.quarterly_balance_sheet
        if balance_sheet is None or balance_sheet.empty:
            balance_sheet = ticker_obj.balance_sheet
    except Exception:
        pass

    total_assets = _bs_lookup(balance_sheet, _TOTAL_ASSETS_KEYS)
    total_debt = _bs_lookup(balance_sheet, _TOTAL_DEBT_KEYS)
    if np.isnan(total_debt):
        ltd = _bs_lookup(balance_sheet, _LONG_TERM_DEBT_KEYS)
        std = _bs_lookup(balance_sheet, _CURRENT_DEBT_KEYS)
        if not (np.isnan(ltd) and np.isnan(std)):
            total_debt = np.nansum([ltd, std])
    cash = _bs_lookup(balance_sheet, _CASH_KEYS)
    receivables = _bs_lookup(balance_sheet, _RECEIVABLES_KEYS)

    market_cap = np.nan
    try:
        market_cap = ticker_obj.fast_info.get("market_cap", np.nan)
    except Exception:
        pass
    if market_cap is None or (isinstance(market_cap, float) and np.isnan(market_cap)):
        try:
            market_cap = ticker_obj.info.get("marketCap", np.nan)
        except Exception:
            market_cap = np.nan

    return {
        "market_cap": market_cap,
        "total_assets": total_assets,
        "total_debt": total_debt,
        "cash_and_interest_securities": cash,
        "receivables": receivables,
    }


def _try_live_ingestion(tickers, start, end) -> MarketDataset | None:
    """Attempt a real pull via yfinance + a fundamentals source. Fetches each
    ticker individually so one bad symbol doesn't sink the whole pull, and
    validates that real data actually came back before trusting it -- yfinance
    logs download errors rather than raising them, so a naive call can appear
    to "succeed" while returning an empty/all-NaN frame. Returns None
    (triggering the synthetic fallback) if too few tickers yield usable data."""
    try:
        import yfinance as yf
    except ImportError:
        print("[data_ingestion] yfinance not installed.")
        return None

    price_series, volume_series, fundamentals_rows = {}, {}, []

    for t in tickers:
        tk = yf.Ticker(t)
        try:
            hist = tk.history(start=start, end=end, auto_adjust=True)
            if hist is None or hist.empty or hist["Close"].dropna().empty:
                print(f"[data_ingestion] No price history for {t}, skipping.")
                continue
            price_series[t] = hist["Close"]
            volume_series[t] = hist["Volume"]
        except Exception as e:
            print(f"[data_ingestion] Price fetch failed for {t}: {e}")
            continue

        try:
            info = tk.info
            row = _fetch_fundamentals(tk)
            row["ticker"] = t
            row["sector"] = _classify_industry(info.get("industry", info.get("sector")))
            fundamentals_rows.append(row)
        except Exception as e:
            print(f"[data_ingestion] Fundamentals fetch failed for {t}: {e} (will be NaN).")
            fundamentals_rows.append({
                "ticker": t, "market_cap": np.nan, "total_debt": np.nan,
                "total_assets": np.nan, "cash_and_interest_securities": np.nan,
                "receivables": np.nan, "sector": "Unknown",
            })

    if len(price_series) < MIN_VALID_TICKERS:
        print(f"[data_ingestion] Only {len(price_series)} tickers returned live price data "
              f"(< {MIN_VALID_TICKERS} minimum) -> treating live pull as failed.")
        return None

    prices = pd.DataFrame(price_series).sort_index()
    volumes = pd.DataFrame(volume_series).sort_index()
    fundamentals = pd.DataFrame(fundamentals_rows).set_index("ticker").reindex(prices.columns)
    meta = fundamentals[["sector"]].copy()

    n_missing_assets = fundamentals["total_assets"].isna().sum()
    n_missing_core = fundamentals[["total_debt", "cash_and_interest_securities", "receivables"]].isna().any(axis=1).sum()
    if n_missing_assets or n_missing_core:
        print(f"[data_ingestion] Note: {n_missing_assets}/{len(fundamentals)} tickers are missing "
              f"'total_assets' from the balance sheet (Stage 2 will fall back to market-cap as the "
              f"screening denominator for those); {n_missing_core}/{len(fundamentals)} are missing "
              "debt/cash/receivables entirely and will fail the financial-ratio screen outright.")

    return MarketDataset(prices, volumes, fundamentals, meta, start, end)


def _synthetic_universe(universe, start, end, seed: int = 42) -> MarketDataset:
    """Generates a realistic 5y OHLCV + fundamentals dataset via GBM + noise,
    used whenever live ingestion isn't reachable (e.g. this sandbox)."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end)
    n = len(dates)

    tickers = [u[0] for u in universe]
    names = {u[0]: u[1] for u in universe}
    sectors = {u[0]: u[2] for u in universe}

    prices, volumes, fundamentals_rows = {}, {}, []

    for i, t in enumerate(tickers):
        mu = rng.uniform(0.04, 0.12) / TRADING_DAYS_PER_YEAR       # annual drift -> daily
        sigma = rng.uniform(0.15, 0.45) / np.sqrt(TRADING_DAYS_PER_YEAR)
        s0 = rng.uniform(1.0, 25.0)
        shocks = rng.normal(mu - 0.5 * sigma ** 2, sigma, n)
        price_path = s0 * np.exp(np.cumsum(shocks))
        prices[t] = price_path

        base_vol = rng.uniform(2e5, 8e6)
        vol_series = np.abs(rng.normal(base_vol, base_vol * 0.3, n)).astype(int)
        volumes[t] = vol_series

        market_cap = price_path[-1] * rng.uniform(2e8, 6e9)
        # Sector drives whether ratios are typically compliant or not, so the
        # synthetic data still gives Stage 2 a mix of passes/fails.
        is_bank_or_brewer = sectors[t] in ("Conventional Banking", "Brewery")
        debt_ratio = rng.uniform(0.35, 0.55) if is_bank_or_brewer else rng.uniform(0.05, 0.30)
        cash_ratio = rng.uniform(0.35, 0.60) if is_bank_or_brewer else rng.uniform(0.05, 0.28)
        recv_ratio = rng.uniform(0.10, 0.30)

        total_assets = market_cap * rng.uniform(0.8, 1.5)
        fundamentals_rows.append({
            "ticker": t,
            "market_cap": market_cap,
            "total_assets": total_assets,
            "total_debt": debt_ratio * total_assets,
            "cash_and_interest_securities": cash_ratio * total_assets,
            "receivables": recv_ratio * total_assets,
            "sector": sectors[t],
        })

    prices_df = pd.DataFrame(prices, index=dates)
    volumes_df = pd.DataFrame(volumes, index=dates)
    fundamentals_df = pd.DataFrame(fundamentals_rows).set_index("ticker")
    meta_df = pd.DataFrame({"name": names, "sector": sectors})

    return MarketDataset(prices_df, volumes_df, fundamentals_df, meta_df, start, end)


def ingest_market_data(universe=None, years: int = HORIZON_YEARS) -> MarketDataset:
    """Public entry point for Stage 1a. Tries live ingestion first, else
    synthesizes a stand-in dataset with an identical schema."""
    universe = universe or DEFAULT_UNIVERSE
    end = datetime.today()
    start = end - timedelta(days=int(years * 365.25))
    tickers = [u[0] for u in universe]

    live = _try_live_ingestion(tickers, start, end)
    if live is not None:
        print(f"[data_ingestion] Live data pulled via yfinance for "
              f"{live.prices.shape[1]}/{len(tickers)} requested tickers.")
        return live

    print("[data_ingestion] No network / yfinance unavailable -> using synthetic "
          f"{years}y dataset for {len(universe)} tickers (schema-identical to live pull).")
    return _synthetic_universe(universe, start, end)


# ==============================================================================
# STAGE 1b  |  TECHNICAL INDICATOR ENGINEERING
# (originally: technical_indicators.py)
# ==============================================================================

def _rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _macd(series: pd.Series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line


def _atr(close: pd.Series, window: int = 14) -> pd.Series:
    # No separate high/low feed in this dataset, so approximate true range
    # from close-to-close moves (common simplification when only close/volume
    # is available; swap in real H/L/C once live OHLC data is wired in).
    tr = close.diff().abs()
    return tr.rolling(window).mean()


def _obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direction = np.sign(close.diff().fillna(0))
    raw_obv = (direction * volume).cumsum()
    # Scale-free: rolling 1y z-score so the feature stays O(1) instead of 1e10
    roll_mean = raw_obv.rolling(252, min_periods=60).mean()
    roll_std = raw_obv.rolling(252, min_periods=60).std()
    return (raw_obv - roll_mean) / roll_std.replace(0, np.nan)


def compute_indicators_for_ticker(close: pd.Series, volume: pd.Series) -> pd.DataFrame:
    log_ret = np.log(close / close.shift(1))

    sma10 = close.rolling(10).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line, signal_line = _macd(close)

    roll_std21 = log_ret.rolling(21).std() * np.sqrt(252)  # annualized realized vol
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_width = (bb_mid + 2 * bb_std - (bb_mid - 2 * bb_std)) / bb_mid

    feats = pd.DataFrame({
        "close": close,
        "ret_1d": log_ret,
        "ret_5d": np.log(close / close.shift(5)),
        "ret_21d": np.log(close / close.shift(21)),
        "sma10": sma10, "sma50": sma50, "sma200": sma200,
        "ema12": ema12, "ema26": ema26,
        "macd": macd_line, "macd_signal": signal_line, "macd_hist": macd_line - signal_line,
        "rsi14": _rsi(close, 14),
        "roc10": close.pct_change(10, fill_method=None) * 100,
        "vol21_ann": roll_std21,
        "bb_width": bb_width,
        "atr14": _atr(close, 14),
        "vol_roc10": volume.pct_change(10, fill_method=None) * 100,
        "obv": _obv(close, volume),
    })
    # Price-relative-to-trend signals (scale-free, better ANN inputs than raw prices)
    feats["px_over_sma50"] = close / sma50 - 1
    feats["sma10_over_sma50"] = sma10 / sma50 - 1
    feats = feats.replace([np.inf, -np.inf], np.nan)
    return feats


def build_feature_panel(prices: pd.DataFrame, volumes: pd.DataFrame) -> pd.DataFrame:
    """Returns a long-format panel: MultiIndex (date, ticker) x features."""
    frames = []
    for ticker in prices.columns:
        f = compute_indicators_for_ticker(prices[ticker], volumes[ticker])
        f["ticker"] = ticker
        frames.append(f)
    panel = pd.concat(frames)
    panel = panel.set_index("ticker", append=True)
    panel.index.names = ["date", "ticker"]
    return panel.sort_index()


# ==============================================================================
# STAGE 2   |  SHARIAH UNIVERSE SCREENING
# (originally: shariah_screening.py)
# ==============================================================================

# Tier 1 — non-permissible business activities (by sector label)
NON_COMPLIANT_SECTORS = {
    "Conventional Banking",
    "Conventional Insurance",
    "Gaming & Casinos",
    "Brewery",
    "Tobacco",
    "Adult Entertainment",
    "Conventional Leasing",
    "Weapons & Defense",
    "Pork / Non-Halal Food",
}

# Tier 2 — SC Malaysia-style thresholds
RATIO_THRESHOLDS = {
    "cash_ratio": 0.33,   # cash & interest-bearing securities / denominator
    "debt_ratio": 0.33,   # interest-bearing debt / denominator
    "receivables_ratio": 0.50,  # (receivables + cash) / denominator
}


def business_activity_screen(meta: pd.DataFrame) -> pd.Series:
    """Tier 1: returns a boolean Series indexed by ticker, True = passes."""
    return ~meta["sector"].isin(NON_COMPLIANT_SECTORS)


def financial_ratio_screen(fundamentals: pd.DataFrame,
                            denominator: str = "assets") -> pd.DataFrame:
    """Tier 2: computes the three ratios and a pass/fail flag per ticker.

    Tickers with missing fundamental inputs (common with free data sources --
    Yahoo doesn't always populate every balance-sheet field, especially for
    non-US listings) explicitly fail with `data_available=False` rather than
    being silently excluded by a NaN comparison with no explanation. If the
    requested denominator (`total_assets` by default) is missing for a given
    ticker but market cap is available, that ticker falls back to the
    market-cap-based (AAOIFI-style) denominator automatically rather than
    being excluded purely for lacking one specific balance-sheet figure --
    `denominator_used` records which was actually applied per ticker."""
    primary = fundamentals["total_assets"] if denominator == "assets" else fundamentals["market_cap"]
    fallback = fundamentals["market_cap"] if denominator == "assets" else fundamentals["total_assets"]
    denom = primary.where(primary.notna() & (primary != 0), fallback)

    ratios = pd.DataFrame(index=fundamentals.index)
    ratios["denominator_used"] = np.where(
        primary.notna() & (primary != 0), denominator,
        np.where(fallback.notna() & (fallback != 0), f"{denominator}_fallback", "unavailable"),
    )

    required = ["cash_and_interest_securities", "total_debt", "receivables"]
    ratios["data_available"] = fundamentals[required].notna().all(axis=1) & denom.notna() & (denom != 0)

    ratios["cash_ratio"] = fundamentals["cash_and_interest_securities"] / denom
    ratios["debt_ratio"] = fundamentals["total_debt"] / denom
    ratios["receivables_ratio"] = (fundamentals["receivables"] + fundamentals["cash_and_interest_securities"]) / denom

    ratios["pass_cash"] = ratios["data_available"] & (ratios["cash_ratio"] < RATIO_THRESHOLDS["cash_ratio"])
    ratios["pass_debt"] = ratios["data_available"] & (ratios["debt_ratio"] < RATIO_THRESHOLDS["debt_ratio"])
    ratios["pass_receivables"] = ratios["data_available"] & (ratios["receivables_ratio"] < RATIO_THRESHOLDS["receivables_ratio"])
    ratios["passes_tier2"] = ratios["data_available"] & ratios[["pass_cash", "pass_debt", "pass_receivables"]].all(axis=1)
    return ratios


def screen_universe(meta: pd.DataFrame, fundamentals: pd.DataFrame,
                     denominator: str = "assets") -> tuple[list[str], pd.DataFrame]:
    """Runs both tiers and returns (compliant_tickers, full_audit_dataframe)."""
    tier1 = business_activity_screen(meta)
    tier2 = financial_ratio_screen(fundamentals, denominator=denominator)

    audit = tier2.copy()
    audit["sector"] = meta["sector"]
    audit["passes_tier1"] = tier1
    audit["is_shariah_compliant"] = audit["passes_tier1"] & audit["passes_tier2"]

    compliant = audit.index[audit["is_shariah_compliant"]].tolist()
    n_missing = (~audit["data_available"]).sum()
    if n_missing:
        print(f"[shariah_screening] {n_missing}/{len(audit)} tickers excluded due to missing "
              "fundamental data (rather than a genuine ratio breach) -- see 'data_available' column.")
    cols = ["sector", "passes_tier1", "data_available", "denominator_used", "cash_ratio", "pass_cash",
            "debt_ratio", "pass_debt", "receivables_ratio", "pass_receivables",
            "passes_tier2", "is_shariah_compliant"]
    return compliant, audit[cols]


def filter_price_panel(feature_panel: pd.DataFrame, compliant_tickers: list[str]) -> pd.DataFrame:
    """Isolates the 5y indicator panel down to the compliant universe only."""
    mask = feature_panel.index.get_level_values("ticker").isin(compliant_tickers)
    return feature_panel.loc[mask]


# ==============================================================================
# STAGE 3   |  AI RETURN FORECASTING (ANN)
# (originally: ann_forecast.py)
# ==============================================================================

FEATURE_COLS = [
    "ret_1d", "ret_5d", "ret_21d",
    "sma10", "sma50", "sma200", "ema12", "ema26",
    "macd", "macd_signal", "macd_hist", "rsi14", "roc10",
    "vol21_ann", "bb_width", "atr14", "vol_roc10",
    "px_over_sma50", "sma10_over_sma50",
]
FORWARD_HORIZON = 21  # trading days ~ 1 month ahead


def _make_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["fwd_return"] = np.log(df["close"].shift(-FORWARD_HORIZON) / df["close"])
    df["fwd_vol"] = df["ret_1d"].rolling(FORWARD_HORIZON).std().shift(-FORWARD_HORIZON) * np.sqrt(252)
    return df


def _fit_ann(X_train, y_train, seed=7) -> MLPRegressor:
    model = MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=800,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=seed,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        model.fit(X_train, y_train)
    return model


# Sane guard-rails for a per-asset annualized forecast. Real fitted models on
# real (non-synthetic) data rarely need this, but any ANN can occasionally
# extrapolate wildly on an out-of-distribution input row, and Stage 5's
# optimizer should never be handed an unbounded expected return.
MAX_ABS_ANNUAL_RETURN = 0.60
MAX_ANNUAL_VOL = 0.90
MIN_ANNUAL_VOL = 0.03


EMPTY_FORECAST_COLUMNS = ["exp_return_ann", "exp_vol_ann", "val_rmse_return", "n_train_obs"]


def forecast_universe(feature_panel: pd.DataFrame, compliant_tickers: list[str],
                       min_history: int = 300) -> pd.DataFrame:
    """Trains a per-ticker ANN and returns a DataFrame indexed by ticker with
    columns: exp_return_ann (annualized), exp_vol_ann (annualized), val_rmse_return.
    Returns an empty (but correctly-shaped) DataFrame if the compliant universe
    is empty or none have enough history -- callers should check `.empty`
    rather than assume at least one row comes back."""
    if not compliant_tickers:
        print("[ann_forecast] No compliant tickers were passed in -- nothing to forecast.")
        return pd.DataFrame(columns=EMPTY_FORECAST_COLUMNS).rename_axis("ticker")

    results = []

    for ticker in compliant_tickers:
        df = feature_panel.xs(ticker, level="ticker").sort_index()
        df = _make_targets(df)
        model_df = df.dropna(subset=FEATURE_COLS + ["fwd_return", "fwd_vol"])
        if len(model_df) < min_history:
            continue  # not enough clean history for a reliable ANN fit

        X = model_df[FEATURE_COLS].values
        y_ret = model_df["fwd_return"].values
        y_vol = model_df["fwd_vol"].values

        # Defensive: drop any residual non-finite rows (belt and braces)
        finite_mask = np.isfinite(X).all(axis=1) & np.isfinite(y_ret) & np.isfinite(y_vol)
        if not finite_mask.all():
            n_bad = (~finite_mask).sum()
            print(f"[ann_forecast] {ticker}: dropping {n_bad} row(s) with non-finite feature/target values.")
            X = X[finite_mask]; y_ret = y_ret[finite_mask]; y_vol = y_vol[finite_mask]

        if len(X) < min_history:
            continue

        latest_row = df.dropna(subset=FEATURE_COLS)[FEATURE_COLS].iloc[[-1]].values
        # ...and then also guard the inference row:
        if not np.isfinite(latest_row).all():
            print(f"[ann_forecast] {ticker}: latest feature row has non-finite values, skipping.")
            continue

        split = int(len(model_df) * 0.8)
        X_train, X_test = X[:split], X[split:]
        y_ret_train, y_ret_test = y_ret[:split], y_ret[split:]
        y_vol_train, y_vol_test = y_vol[:split], y_vol[split:]

        scaler = StandardScaler().fit(X_train)
        X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

        ret_model = _fit_ann(X_train_s, y_ret_train)
        vol_model = _fit_ann(X_train_s, y_vol_train)

        ret_pred_test = ret_model.predict(X_test_s)
        rmse_ret = mean_squared_error(y_ret_test, ret_pred_test) ** 0.5

        # Live forecast: most recent fully-observed feature row (not the
        # target-dependent rows dropped above, so it can include the latest data)
        latest_row = df.dropna(subset=FEATURE_COLS)[FEATURE_COLS].iloc[[-1]].values
        latest_scaled = scaler.transform(latest_row)

        pred_fwd_return = float(ret_model.predict(latest_scaled)[0])
        pred_fwd_vol = float(vol_model.predict(latest_scaled)[0])

        # Annualize the 21-trading-day forecast horizon, then clip to guard-rails
        periods_per_year = 252 / FORWARD_HORIZON
        exp_return_ann = float(np.clip(pred_fwd_return * periods_per_year,
                                        -MAX_ABS_ANNUAL_RETURN, MAX_ABS_ANNUAL_RETURN))
        exp_vol_ann = float(np.clip(pred_fwd_vol, MIN_ANNUAL_VOL, MAX_ANNUAL_VOL))

        results.append({
            "ticker": ticker,
            "exp_return_ann": exp_return_ann,
            "exp_vol_ann": exp_vol_ann,
            "val_rmse_return": rmse_ret,
            "n_train_obs": len(X_train),
        })

    if not results:
        print(f"[ann_forecast] {len(compliant_tickers)} compliant ticker(s) passed in, but none had "
              f">= {min_history} clean observations after indicator/target warm-up -- try a longer "
              "history window or lower `min_history`.")
        return pd.DataFrame(columns=EMPTY_FORECAST_COLUMNS).rename_axis("ticker")

    return pd.DataFrame(results).set_index("ticker")


    ds = ingest_market_data()
    panel = build_feature_panel(ds.prices, ds.volumes)
    compliant, _ = screen_universe(ds.meta, ds.fundamentals)
    forecasts = forecast_universe(panel, compliant)
    print(forecasts.sort_values("exp_return_ann", ascending=False))


# ==============================================================================
# STAGE 4   |  INVESTOR PROFILING (ROBO-ADVISOR)
# (originally: investor_profiling.py)
# ==============================================================================

QUESTIONS = [
    {
        "id": "horizon",
        "text": "What is your investment time horizon?",
        "options": {
            "<1 year": 1, "1-3 years": 2, "3-7 years": 3, "7+ years": 4,
        },
    },
    {
        "id": "loss_reaction",
        "text": "If your portfolio fell 20% in a month, what would you do?",
        "options": {
            "Sell everything immediately": 1,
            "Sell some to reduce risk": 2,
            "Hold and wait it out": 3,
            "Buy more at the lower price": 4,
        },
    },
    {
        "id": "income_stability",
        "text": "How stable is your income / need for liquidity from this portfolio?",
        "options": {
            "I may need this money soon": 1,
            "Stable, but I prefer safety": 2,
            "Stable, comfortable with risk": 3,
            "Very stable / surplus capital": 4,
        },
    },
    {
        "id": "experience",
        "text": "How would you describe your investing experience?",
        "options": {
            "None": 1, "Basic": 2, "Experienced": 3, "Very experienced": 4,
        },
    },
    {
        "id": "goal",
        "text": "What is your primary goal?",
        "options": {
            "Capital preservation": 1, "Income": 2, "Balanced growth": 3, "Maximum growth": 4,
        },
    },
]


@dataclass
class InvestorProfile:
    raw_score: int
    max_score: int
    risk_category: str
    lambda_risk_aversion: float
    cvar_alpha: float
    max_single_holding: float


def score_questionnaire(answers: dict[str, str]) -> InvestorProfile:
    """`answers` maps question id -> the chosen option label."""
    total, max_total = 0, 0
    for q in QUESTIONS:
        max_total += max(q["options"].values())
        chosen = answers.get(q["id"])
        if chosen not in q["options"]:
            raise ValueError(f"Missing/invalid answer for '{q['id']}': {chosen!r}")
        total += q["options"][chosen]

    pct = total / max_total  # 0..1

    if pct < 0.40:
        category, lam, alpha, cap = "Conservative", 8.0, 0.99, 0.10
    elif pct < 0.60:
        category, lam, alpha, cap = "Moderate", 4.0, 0.97, 0.15
    elif pct < 0.80:
        category, lam, alpha, cap = "Growth", 2.0, 0.95, 0.20
    else:
        category, lam, alpha, cap = "Aggressive", 1.0, 0.90, 0.30

    return InvestorProfile(
        raw_score=total, max_score=max_total, risk_category=category,
        lambda_risk_aversion=lam, cvar_alpha=alpha, max_single_holding=cap,
    )


def run_cli_questionnaire() -> InvestorProfile:
    """Interactive terminal version of the robo-advisor elicitation flow."""
    answers = {}
    for q in QUESTIONS:
        print(f"\n{q['text']}")
        opts = list(q["options"].keys())
        for i, opt in enumerate(opts, 1):
            print(f"  {i}. {opt}")
        choice = int(input("Choose an option number: "))
        answers[q["id"]] = opts[choice - 1]
    return score_questionnaire(answers)


# ==============================================================================
# STAGE 5   |  CONSTRAINED OPTIMISATION (MEAN-CVAR)
# (originally: portfolio_optimization.py)
# ==============================================================================

ZAKAT_RATE = 0.025          # 2.5% annual zakat on eligible wealth
TRANSACTION_COST_BPS = 15   # 15 bps per unit of turnover (buy or sell)
N_SCENARIOS = 5000
RANDOM_SEED = 11


@dataclass
class OptimizationResult:
    weights: pd.Series
    expected_return_gross: float
    expected_return_net_of_costs_and_zakat: float
    cvar: float
    turnover: float
    transaction_cost: float
    zakat_due: float


def _simulate_return_scenarios(forecasts: pd.DataFrame, correlation: pd.DataFrame | None,
                                n_scenarios: int = N_SCENARIOS, seed: int = RANDOM_SEED) -> np.ndarray:
    """Monte-Carlo scenarios of asset returns from ANN (mu, sigma) forecasts.
    If a historical correlation matrix is supplied, scenarios are correlated
    (Cholesky); otherwise assets are simulated independently (conservative
    fallback, tends to understate diversification benefit)."""
    rng = np.random.default_rng(seed)
    mu = forecasts["exp_return_ann"].values
    sigma = forecasts["exp_vol_ann"].values
    n_assets = len(mu)

    z = rng.standard_normal((n_scenarios, n_assets))
    if correlation is not None:
        corr = correlation.loc[forecasts.index, forecasts.index].values
        # Ensure positive semi-definite before Cholesky (numerical safety)
        corr = (corr + corr.T) / 2
        eigvals, eigvecs = np.linalg.eigh(corr)
        eigvals = np.clip(eigvals, 1e-8, None)
        corr_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
        L = np.linalg.cholesky(corr_psd)
        z = z @ L.T

    scenarios = mu + z * sigma  # shape (n_scenarios, n_assets)
    return scenarios


def _portfolio_cvar(weights: np.ndarray, scenario_returns: np.ndarray, alpha: float) -> float:
    """Historical/scenario CVaR (Expected Shortfall) of portfolio losses at
    confidence alpha (e.g. 0.95 -> worst 5% tail averaged)."""
    port_returns = scenario_returns @ weights
    var_threshold = np.percentile(port_returns, (1 - alpha) * 100)
    tail = port_returns[port_returns <= var_threshold]
    if len(tail) == 0:
        tail = np.array([var_threshold])
    cvar_loss = -tail.mean()  # positive number = expected tail loss
    return float(cvar_loss)


def optimize_portfolio(forecasts: pd.DataFrame,
                        lambda_risk_aversion: float,
                        cvar_alpha: float,
                        max_single_holding: float,
                        current_weights: pd.Series | None = None,
                        correlation: pd.DataFrame | None = None) -> OptimizationResult:
    tickers = forecasts.index.tolist()
    n = len(tickers)
    mu = forecasts["exp_return_ann"].values

    if current_weights is None:
        current_weights = pd.Series(0.0, index=tickers)
    else:
        current_weights = current_weights.reindex(tickers).fillna(0.0)
    w0_current = current_weights.values

    scenarios = _simulate_return_scenarios(forecasts, correlation)

    def objective(w):
        exp_return = mu @ w
        cvar = _portfolio_cvar(w, scenarios, cvar_alpha)
        turnover = np.sum(np.abs(w - w0_current))
        tc = (TRANSACTION_COST_BPS / 10_000) * turnover
        # Mean-CVaR utility, net of transaction costs; zakat handled post-hoc
        # as a portfolio-level drag (see below) so it doesn't bias relative
        # asset weighting, matching how zakat is actually assessed (on total
        # eligible wealth, not per-security).
        utility = exp_return - lambda_risk_aversion * cvar - tc
        return -utility  # minimize negative utility

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    ]
    bounds = [(0.0, max_single_holding) for _ in range(n)]
    w_start = np.full(n, 1.0 / n)

    result = minimize(objective, w_start, method="SLSQP", bounds=bounds,
                       constraints=constraints, options={"maxiter": 500, "ftol": 1e-9})

    if not result.success:
        # Fall back to a projected-gradient-style renormalization of the
        # best-found point rather than failing the whole pipeline.
        w_final = np.clip(result.x, 0, max_single_holding)
        w_final = w_final / w_final.sum()
    else:
        w_final = result.x
        w_final = np.clip(w_final, 0, max_single_holding)
        w_final = w_final / w_final.sum()

    weights = pd.Series(w_final, index=tickers, name="weight")
    exp_return_gross = float(mu @ w_final)
    cvar_final = _portfolio_cvar(w_final, scenarios, cvar_alpha)
    turnover = float(np.sum(np.abs(w_final - w0_current)))
    tc_final = (TRANSACTION_COST_BPS / 10_000) * turnover
    zakat_due = ZAKAT_RATE  # expressed as an annual rate drag on portfolio value
    exp_return_net = exp_return_gross - tc_final - zakat_due

    return OptimizationResult(
        weights=weights.sort_values(ascending=False),
        expected_return_gross=exp_return_gross,
        expected_return_net_of_costs_and_zakat=exp_return_net,
        cvar=cvar_final,
        turnover=turnover,
        transaction_cost=tc_final,
        zakat_due=zakat_due,
    )


    ds = ingest_market_data()
    panel = build_feature_panel(ds.prices, ds.volumes)
    compliant, _ = screen_universe(ds.meta, ds.fundamentals)
    forecasts = forecast_universe(panel, compliant)

    demo_answers = {
        "horizon": "3-7 years", "loss_reaction": "Sell some to reduce risk",
        "income_stability": "Stable, but I prefer safety", "experience": "Basic",
        "goal": "Balanced growth",
    }
    profile = score_questionnaire(demo_answers)

    hist_returns = ds.prices[forecasts.index].pct_change().dropna()
    corr = hist_returns.corr()

    result = optimize_portfolio(
        forecasts, profile.lambda_risk_aversion, profile.cvar_alpha,
        profile.max_single_holding, correlation=corr,
    )
    print(profile)
    print("\nFinal weights:\n", result.weights[result.weights > 0.001])
    print(f"\nExpected return (gross, annualized): {result.expected_return_gross:.2%}")
    print(f"Expected return (net of TC + zakat):  {result.expected_return_net_of_costs_and_zakat:.2%}")
    print(f"Portfolio CVaR@{profile.cvar_alpha:.0%}: {result.cvar:.2%}")
    print(f"Turnover: {result.turnover:.2%}, Transaction cost: {result.transaction_cost:.4%}")


# ==============================================================================
# STAGE 6   |  PIPELINE ORCHESTRATION & PORTFOLIO OUTPUT
# (originally: main.py)
# ==============================================================================

warnings.filterwarnings("ignore", category=ConvergenceWarning)

DEMO_ANSWERS = {
    "horizon": "7+ years",
    "loss_reaction": "Hold and wait it out",
    "income_stability": "Stable, comfortable with risk",
    "experience": "Experienced",
    "goal": "Balanced growth",
}


def run_pipeline(interactive: bool = False) -> None:
    print("=" * 70)
    print("STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)")
    print("=" * 70)
    dataset = ingest_market_data()
    feature_panel = build_feature_panel(dataset.prices, dataset.volumes)
    print(f"Ingested {dataset.prices.shape[1]} tickers x {dataset.prices.shape[0]} trading days.")
    print(f"Feature panel: {feature_panel.shape[0]} (date,ticker) rows x {feature_panel.shape[1]} indicators.\n")

    print("=" * 70)
    print("STAGE 2 — Shariah Universe Screening")
    print("=" * 70)
    compliant_tickers, audit = screen_universe(dataset.meta, dataset.fundamentals)
    print(f"{len(compliant_tickers)} / {len(dataset.meta)} tickers pass business-activity + "
          f"financial-ratio screens:")
    print(compliant_tickers, "\n")
    if not compliant_tickers:
        print("No tickers passed the Shariah screen -- stopping here. Common causes: (1) live "
              "fundamentals data was missing for most/all tickers (see any '[data_ingestion] "
              "Warning' above), or (2) the universe genuinely contains no compliant names. "
              "Full audit trail:\n", audit)
        return

    print("=" * 70)
    print("STAGE 3 — AI Return Forecasting (ANN)")
    print("=" * 70)
    forecasts = forecast_universe(feature_panel, compliant_tickers)
    if forecasts.empty:
        print("No forecasts were produced (not enough clean history per ticker) -- stopping here.")
        return
    print(forecasts.sort_values("exp_return_ann", ascending=False).round(4), "\n")

    print("=" * 70)
    print("STAGE 4 — Investor Profiling (Robo-Advisor)")
    print("=" * 70)
    profile = run_cli_questionnaire() if interactive else score_questionnaire(DEMO_ANSWERS)
    print(f"Risk category: {profile.risk_category}")
    print(f"  lambda (risk aversion)   = {profile.lambda_risk_aversion}")
    print(f"  CVaR confidence (alpha)  = {profile.cvar_alpha:.0%}")
    print(f"  Max single holding cap   = {profile.max_single_holding:.0%}\n")

    print("=" * 70)
    print("STAGE 5 — Constrained Optimisation (Mean-CVaR)")
    print("=" * 70)
    hist_returns = dataset.prices[forecasts.index].pct_change().dropna()
    correlation = hist_returns.corr()
    result = optimize_portfolio(
        forecasts,
        lambda_risk_aversion=profile.lambda_risk_aversion,
        cvar_alpha=profile.cvar_alpha,
        max_single_holding=profile.max_single_holding,
        correlation=correlation,
    )

    print("=" * 70)
    print("STAGE 6 — Portfolio Output")
    print("=" * 70)
    final_weights = result.weights[result.weights > 0.005]
    report = final_weights.to_frame("weight")
    report["sector"] = dataset.meta.loc[report.index, "sector"]
    report["exp_return_ann"] = forecasts.loc[report.index, "exp_return_ann"]
    report["exp_vol_ann"] = forecasts.loc[report.index, "exp_vol_ann"]
    print(report.round(4))
    print(f"\nPortfolio-level expected return (gross, annualized): {result.expected_return_gross:.2%}")
    print(f"Portfolio-level expected return (net of TC + zakat) : "
          f"{result.expected_return_net_of_costs_and_zakat:.2%}")
    print(f"Portfolio CVaR @ {profile.cvar_alpha:.0%} confidence          : {result.cvar:.2%}")
    print(f"Turnover from current holdings                       : {result.turnover:.2%}")
    print(f"Transaction cost drag                                : {result.transaction_cost:.4%}")
    print(f"Annual zakat obligation (2.5% of eligible wealth)    : {result.zakat_due:.2%}")

    report.to_csv("/tmp/final_portfolio.csv")
    print("\nSaved final portfolio to /tmp/final_portfolio.csv")


if __name__ == "__main__":
    run_pipeline(interactive=False)  # set True to answer the questionnaire yourself in the Colab cell prompt

STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)
[data_ingestion] Live data pulled via yfinance for 25/25 requested tickers.
Ingested 25 tickers x 1227 trading days.
Feature panel: 30675 (date,ticker) rows x 21 indicators.

STAGE 2 — Shariah Universe Screening
17 / 25 tickers pass business-activity + financial-ratio screens:
['1155.KL', '1295.KL', '6033.KL', '5183.KL', '1961.KL', '4707.KL', '3689.KL', '3026.KL', '7113.KL', '5168.KL', '5285.KL', '8869.KL', '3816.KL', '2836.KL', '3255.KL', '5211.KL', '3336.KL'] 

STAGE 3 — AI Return Forecasting (ANN)
         exp_return_ann  exp_vol_ann  val_rmse_return  n_train_obs
ticker                                                            
2836.KL          0.6000       0.5158           0.1211          802
5168.KL          0.6000       0.8720           0.4545          784
3255.KL          0.6000       0.5798           0.1359          801
8869.KL          0.6000       0.2806           0.3976          802
5211.KL          0.6000       0.16

/tmp/ipykernel_1641/3659351008.py:995: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  hist_returns = dataset.prices[forecasts.index].pct_change().dropna()


# fix compliance

In [ ]:
# =============================================================================
# SHARIAH-COMPLIANT AI ROBO-ADVISOR — FULL PIPELINE (single-file, Colab-ready)
# =============================================================================
# Paste this entire file into one Google Colab cell and run.
# =============================================================================

# !pip install yfinance -q

from __future__ import annotations
import sys
import warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from scipy.optimize import minimize
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ==============================================================================
# STAGE 1a  |  DATA INGESTION (5-YEAR HORIZON)
# ==============================================================================

HORIZON_YEARS = 5
TRADING_DAYS_PER_YEAR = 252

DEFAULT_UNIVERSE = [
    ("1155.KL", "Malayan Banking Bhd (Maybank)", "Conventional Banking"),
    ("1295.KL", "Public Bank Bhd", "Conventional Banking"),
    ("5347.KL", "Tenaga Nasional Bhd", "Utilities"),
    ("6033.KL", "Petronas Gas Bhd", "Energy"),
    ("5183.KL", "Petronas Chemicals Group Bhd", "Materials"),
    ("1961.KL", "IOI Corp Bhd", "Plantation"),
    ("2445.KL", "Kuala Lumpur Kepong Bhd", "Plantation"),
    ("4707.KL", "Nestle Malaysia Bhd", "Consumer Staples"),
    ("3689.KL", "Fraser & Neave Holdings Bhd", "Consumer Staples"),
    ("3026.KL", "Dutch Lady Milk Industries Bhd", "Consumer Staples"),
    ("7113.KL", "Top Glove Corp Bhd", "Health Care Equipment"),
    ("5168.KL", "Hartalega Holdings Bhd", "Health Care Equipment"),
    ("3182.KL", "Genting Bhd", "Gaming & Casinos"),
    ("4715.KL", "Genting Malaysia Bhd", "Gaming & Casinos"),
    ("6888.KL", "Axiata Group Bhd", "Telecommunications"),
    ("6947.KL", "CelcomDigi Bhd (fka Digi.Com)", "Telecommunications"),
    ("6012.KL", "Maxis Bhd", "Telecommunications"),
    ("5285.KL", "SD Guthrie Bhd (fka Sime Darby Plantation)", "Plantation"),
    ("8869.KL", "Press Metal Aluminium Holdings Bhd", "Materials"),
    ("3816.KL", "MISC Bhd", "Shipping/Logistics"),
    ("2836.KL", "Carlsberg Brewery Malaysia Bhd", "Brewery"),
    ("3255.KL", "Heineken Malaysia Bhd", "Brewery"),
    ("4677.KL", "YTL Corp Bhd", "Conglomerate/Utilities"),
    ("5211.KL", "Sunway Bhd", "Property & Construction"),
    ("3336.KL", "IJM Corp Bhd", "Property & Construction"),
]

MIN_VALID_TICKERS = 5


@dataclass
class MarketDataset:
    prices: pd.DataFrame
    volumes: pd.DataFrame
    fundamentals: pd.DataFrame
    meta: pd.DataFrame
    start_date: datetime = field(default=None)
    end_date: datetime = field(default=None)


# -----------------------------------------------------------------------------
# FIX (Stage 2 Tier 1 vocabulary): the live ingestion path re-labels each
# ticker using Yahoo's `industry` string, while the Shariah screen matches on
# the canonical labels in NON_COMPLIANT_SECTORS. Previously the classifier
# emitted "Excluded (brewers)" etc., which never intersected NON_COMPLIANT_SECTORS
# — so Tier 1 passed EVERY live-ingested ticker unconditionally (Maybank and
# Carlsberg were silently passing the screen). We now map industry keywords
# directly onto the canonical sector labels, and assert consistency at import
# time so this class of bug can't silently reappear.
# -----------------------------------------------------------------------------
NON_COMPLIANT_SECTORS = {
    "Conventional Banking",
    "Conventional Insurance",
    "Gaming & Casinos",
    "Brewery",
    "Tobacco",
    "Adult Entertainment",
    "Conventional Leasing",
    "Weapons & Defense",
    "Pork / Non-Halal Food",
}

NON_COMPLIANT_INDUSTRY_KEYWORDS: dict[str, str] = {
    # Yahoo industry substring (lowercase)  ->  canonical NON_COMPLIANT_SECTORS label
    "bank": "Conventional Banking",
    "insurance": "Conventional Insurance",
    "credit services": "Conventional Banking",
    "capital markets": "Conventional Banking",
    "gambling": "Gaming & Casinos",
    "resorts & casinos": "Gaming & Casinos",
    "casino": "Gaming & Casinos",
    "brewers": "Brewery",
    "distillers": "Brewery",
    "wineries": "Brewery",
    "beverages - wineries": "Brewery",
    "tobacco": "Tobacco",
    "aerospace & defense": "Weapons & Defense",
    "adult": "Adult Entertainment",
}


def _assert_vocabulary_consistency() -> None:
    orphans = {v for v in NON_COMPLIANT_INDUSTRY_KEYWORDS.values()
               if v not in NON_COMPLIANT_SECTORS}
    assert not orphans, (
        f"Vocabulary mismatch: {orphans} are used as canonical labels in "
        "NON_COMPLIANT_INDUSTRY_KEYWORDS but are missing from "
        "NON_COMPLIANT_SECTORS. Tier 1 would silently fail to exclude them."
    )


_assert_vocabulary_consistency()


def _classify_industry(industry: str) -> str:
    """Maps a raw Yahoo `industry` string onto a canonical sector label.
    Excluded industries map directly onto a label in NON_COMPLIANT_SECTORS;
    everything else passes through unchanged."""
    ind_lower = (industry or "").lower()
    # Longer keywords first so e.g. "beverages - wineries" wins over "wineries"
    for kw in sorted(NON_COMPLIANT_INDUSTRY_KEYWORDS, key=len, reverse=True):
        if kw in ind_lower:
            return NON_COMPLIANT_INDUSTRY_KEYWORDS[kw]
    return industry or "Unknown"


_TOTAL_ASSETS_KEYS = ["Total Assets"]
_TOTAL_DEBT_KEYS = ["Total Debt"]
_LONG_TERM_DEBT_KEYS = ["Long Term Debt"]
_CURRENT_DEBT_KEYS = ["Current Debt", "Current Debt And Capital Lease Obligation"]
_CASH_KEYS = ["Cash Cash Equivalents And Short Term Investments", "Cash And Cash Equivalents"]
_RECEIVABLES_KEYS = ["Receivables", "Accounts Receivable"]


def _bs_lookup(balance_sheet: pd.DataFrame, candidates: list[str]) -> float:
    if balance_sheet is None or balance_sheet.empty:
        return np.nan
    col = balance_sheet.columns[0]
    idx_lower = {str(i).strip().lower(): i for i in balance_sheet.index}
    for cand in candidates:
        key = cand.strip().lower()
        if key in idx_lower:
            val = balance_sheet.loc[idx_lower[key], col]
            if pd.notna(val):
                return float(val)
        for lower_name, orig_name in idx_lower.items():
            if key in lower_name:
                val = balance_sheet.loc[orig_name, col]
                if pd.notna(val):
                    return float(val)
    return np.nan


def _fetch_fundamentals(ticker_obj) -> dict:
    balance_sheet = None
    try:
        balance_sheet = ticker_obj.quarterly_balance_sheet
        if balance_sheet is None or balance_sheet.empty:
            balance_sheet = ticker_obj.balance_sheet
    except Exception:
        pass

    total_assets = _bs_lookup(balance_sheet, _TOTAL_ASSETS_KEYS)
    total_debt = _bs_lookup(balance_sheet, _TOTAL_DEBT_KEYS)
    if np.isnan(total_debt):
        ltd = _bs_lookup(balance_sheet, _LONG_TERM_DEBT_KEYS)
        std = _bs_lookup(balance_sheet, _CURRENT_DEBT_KEYS)
        if not (np.isnan(ltd) and np.isnan(std)):
            total_debt = np.nansum([ltd, std])
    cash = _bs_lookup(balance_sheet, _CASH_KEYS)
    receivables = _bs_lookup(balance_sheet, _RECEIVABLES_KEYS)

    market_cap = np.nan
    try:
        market_cap = ticker_obj.fast_info.get("market_cap", np.nan)
    except Exception:
        pass
    if market_cap is None or (isinstance(market_cap, float) and np.isnan(market_cap)):
        try:
            market_cap = ticker_obj.info.get("marketCap", np.nan)
        except Exception:
            market_cap = np.nan

    return {
        "market_cap": market_cap,
        "total_assets": total_assets,
        "total_debt": total_debt,
        "cash_and_interest_securities": cash,
        "receivables": receivables,
    }


def _try_live_ingestion(tickers, start, end) -> MarketDataset | None:
    try:
        import yfinance as yf
    except ImportError:
        print("[data_ingestion] yfinance not installed.")
        return None

    price_series, volume_series, fundamentals_rows = {}, {}, []

    for t in tickers:
        tk = yf.Ticker(t)
        try:
            hist = tk.history(start=start, end=end, auto_adjust=True)
            if hist is None or hist.empty or hist["Close"].dropna().empty:
                print(f"[data_ingestion] No price history for {t}, skipping.")
                continue
            price_series[t] = hist["Close"]
            volume_series[t] = hist["Volume"]
        except Exception as e:
            print(f"[data_ingestion] Price fetch failed for {t}: {e}")
            continue

        try:
            info = tk.info
            row = _fetch_fundamentals(tk)
            row["ticker"] = t
            row["sector"] = _classify_industry(info.get("industry", info.get("sector")))
            fundamentals_rows.append(row)
        except Exception as e:
            print(f"[data_ingestion] Fundamentals fetch failed for {t}: {e} (will be NaN).")
            fundamentals_rows.append({
                "ticker": t, "market_cap": np.nan, "total_debt": np.nan,
                "total_assets": np.nan, "cash_and_interest_securities": np.nan,
                "receivables": np.nan, "sector": "Unknown",
            })

    if len(price_series) < MIN_VALID_TICKERS:
        print(f"[data_ingestion] Only {len(price_series)} tickers returned live price data "
              f"(< {MIN_VALID_TICKERS} minimum) -> treating live pull as failed.")
        return None

    prices = pd.DataFrame(price_series).sort_index()
    volumes = pd.DataFrame(volume_series).sort_index()
    fundamentals = pd.DataFrame(fundamentals_rows).set_index("ticker").reindex(prices.columns)
    meta = fundamentals[["sector"]].copy()

    n_missing_assets = fundamentals["total_assets"].isna().sum()
    n_missing_core = fundamentals[["total_debt", "cash_and_interest_securities", "receivables"]].isna().any(axis=1).sum()
    if n_missing_assets or n_missing_core:
        print(f"[data_ingestion] Note: {n_missing_assets}/{len(fundamentals)} tickers are missing "
              f"'total_assets' from the balance sheet (Stage 2 will fall back to market-cap as the "
              f"screening denominator for those); {n_missing_core}/{len(fundamentals)} are missing "
              "debt/cash/receivables entirely and will fail the financial-ratio screen outright.")

    return MarketDataset(prices, volumes, fundamentals, meta, start, end)


def _synthetic_universe(universe, start, end, seed: int = 42) -> MarketDataset:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end)
    n = len(dates)

    tickers = [u[0] for u in universe]
    names = {u[0]: u[1] for u in universe}
    sectors = {u[0]: u[2] for u in universe}

    prices, volumes, fundamentals_rows = {}, {}, []

    for i, t in enumerate(tickers):
        mu = rng.uniform(0.04, 0.12) / TRADING_DAYS_PER_YEAR
        sigma = rng.uniform(0.15, 0.45) / np.sqrt(TRADING_DAYS_PER_YEAR)
        s0 = rng.uniform(1.0, 25.0)
        shocks = rng.normal(mu - 0.5 * sigma ** 2, sigma, n)
        price_path = s0 * np.exp(np.cumsum(shocks))
        prices[t] = price_path

        base_vol = rng.uniform(2e5, 8e6)
        vol_series = np.abs(rng.normal(base_vol, base_vol * 0.3, n)).astype(int)
        volumes[t] = vol_series

        market_cap = price_path[-1] * rng.uniform(2e8, 6e9)
        is_bank_or_brewer = sectors[t] in ("Conventional Banking", "Brewery")
        debt_ratio = rng.uniform(0.35, 0.55) if is_bank_or_brewer else rng.uniform(0.05, 0.30)
        cash_ratio = rng.uniform(0.35, 0.60) if is_bank_or_brewer else rng.uniform(0.05, 0.28)
        recv_ratio = rng.uniform(0.10, 0.30)

        total_assets = market_cap * rng.uniform(0.8, 1.5)
        fundamentals_rows.append({
            "ticker": t,
            "market_cap": market_cap,
            "total_assets": total_assets,
            "total_debt": debt_ratio * total_assets,
            "cash_and_interest_securities": cash_ratio * total_assets,
            "receivables": recv_ratio * total_assets,
            "sector": sectors[t],
        })

    prices_df = pd.DataFrame(prices, index=dates)
    volumes_df = pd.DataFrame(volumes, index=dates)
    fundamentals_df = pd.DataFrame(fundamentals_rows).set_index("ticker")
    meta_df = pd.DataFrame({"name": names, "sector": sectors})

    return MarketDataset(prices_df, volumes_df, fundamentals_df, meta_df, start, end)


def ingest_market_data(universe=None, years: int = HORIZON_YEARS) -> MarketDataset:
    universe = universe or DEFAULT_UNIVERSE
    end = datetime.today()
    start = end - timedelta(days=int(years * 365.25))
    tickers = [u[0] for u in universe]

    live = _try_live_ingestion(tickers, start, end)
    if live is not None:
        print(f"[data_ingestion] Live data pulled via yfinance for "
              f"{live.prices.shape[1]}/{len(tickers)} requested tickers.")
        return live

    print("[data_ingestion] No network / yfinance unavailable -> using synthetic "
          f"{years}y dataset for {len(universe)} tickers (schema-identical to live pull).")
    return _synthetic_universe(universe, start, end)


# ==============================================================================
# STAGE 1b  |  TECHNICAL INDICATOR ENGINEERING
# ==============================================================================

def _rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _macd(series: pd.Series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line


def _atr(close: pd.Series, window: int = 14) -> pd.Series:
    tr = close.diff().abs()
    return tr.rolling(window).mean()


def _obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    """FIX: scale-free OBV (rolling 1y z-score) instead of raw cumulative sum.
    Raw cumulative OBV grows into the billions, which is a pathological input
    for StandardScaler + MLP; a rolling z-score keeps the feature O(1)."""
    direction = np.sign(close.diff().fillna(0))
    raw_obv = (direction * volume).cumsum()
    roll_mean = raw_obv.rolling(252, min_periods=60).mean()
    roll_std = raw_obv.rolling(252, min_periods=60).std()
    return (raw_obv - roll_mean) / roll_std.replace(0, np.nan)


def compute_indicators_for_ticker(close: pd.Series, volume: pd.Series) -> pd.DataFrame:
    log_ret = np.log(close / close.shift(1))

    sma10 = close.rolling(10).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line, signal_line = _macd(close)

    roll_std21 = log_ret.rolling(21).std() * np.sqrt(252)
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_width = (bb_mid + 2 * bb_std - (bb_mid - 2 * bb_std)) / bb_mid

    feats = pd.DataFrame({
        "close": close,
        "ret_1d": log_ret,
        "ret_5d": np.log(close / close.shift(5)),
        "ret_21d": np.log(close / close.shift(21)),
        "sma10": sma10, "sma50": sma50, "sma200": sma200,
        "ema12": ema12, "ema26": ema26,
        "macd": macd_line, "macd_signal": signal_line, "macd_hist": macd_line - signal_line,
        "rsi14": _rsi(close, 14),
        "roc10": close.pct_change(10, fill_method=None) * 100,
        "vol21_ann": roll_std21,
        "bb_width": bb_width,
        "atr14": _atr(close, 14),
        "vol_roc10": volume.pct_change(10, fill_method=None) * 100,
        "obv": _obv(close, volume),
    })
    feats["px_over_sma50"] = close / sma50 - 1
    feats["sma10_over_sma50"] = sma10 / sma50 - 1

    # FIX: pct_change on zero volume (halted/thin Bursa counters) produces ±inf.
    # Convert to NaN so downstream dropna() excludes those rows cleanly instead
    # of feeding inf into StandardScaler / the ANN.
    feats = feats.replace([np.inf, -np.inf], np.nan)
    return feats


def build_feature_panel(prices: pd.DataFrame, volumes: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for ticker in prices.columns:
        f = compute_indicators_for_ticker(prices[ticker], volumes[ticker])
        f["ticker"] = ticker
        frames.append(f)
    panel = pd.concat(frames)
    panel = panel.set_index("ticker", append=True)
    panel.index.names = ["date", "ticker"]
    return panel.sort_index()


# ==============================================================================
# STAGE 2   |  SHARIAH UNIVERSE SCREENING
# ==============================================================================

RATIO_THRESHOLDS = {
    "cash_ratio": 0.33,
    "debt_ratio": 0.33,
    "receivables_ratio": 0.50,
}


def business_activity_screen(meta: pd.DataFrame) -> pd.Series:
    """Tier 1: returns a boolean Series indexed by ticker, True = passes."""
    return ~meta["sector"].isin(NON_COMPLIANT_SECTORS)


def financial_ratio_screen(fundamentals: pd.DataFrame,
                            denominator: str = "assets") -> pd.DataFrame:
    primary = fundamentals["total_assets"] if denominator == "assets" else fundamentals["market_cap"]
    fallback = fundamentals["market_cap"] if denominator == "assets" else fundamentals["total_assets"]
    denom = primary.where(primary.notna() & (primary != 0), fallback)

    ratios = pd.DataFrame(index=fundamentals.index)
    ratios["denominator_used"] = np.where(
        primary.notna() & (primary != 0), denominator,
        np.where(fallback.notna() & (fallback != 0), f"{denominator}_fallback", "unavailable"),
    )

    required = ["cash_and_interest_securities", "total_debt", "receivables"]
    ratios["data_available"] = fundamentals[required].notna().all(axis=1) & denom.notna() & (denom != 0)

    ratios["cash_ratio"] = fundamentals["cash_and_interest_securities"] / denom
    ratios["debt_ratio"] = fundamentals["total_debt"] / denom
    ratios["receivables_ratio"] = (fundamentals["receivables"] + fundamentals["cash_and_interest_securities"]) / denom

    ratios["pass_cash"] = ratios["data_available"] & (ratios["cash_ratio"] < RATIO_THRESHOLDS["cash_ratio"])
    ratios["pass_debt"] = ratios["data_available"] & (ratios["debt_ratio"] < RATIO_THRESHOLDS["debt_ratio"])
    ratios["pass_receivables"] = ratios["data_available"] & (ratios["receivables_ratio"] < RATIO_THRESHOLDS["receivables_ratio"])
    ratios["passes_tier2"] = ratios["data_available"] & ratios[["pass_cash", "pass_debt", "pass_receivables"]].all(axis=1)
    return ratios


def screen_universe(meta: pd.DataFrame, fundamentals: pd.DataFrame,
                     denominator: str = "assets") -> tuple[list[str], pd.DataFrame]:
    tier1 = business_activity_screen(meta)
    tier2 = financial_ratio_screen(fundamentals, denominator=denominator)

    audit = tier2.copy()
    audit["sector"] = meta["sector"]
    audit["passes_tier1"] = tier1
    audit["is_shariah_compliant"] = audit["passes_tier1"] & audit["passes_tier2"]

    # FIX (Stage 2 sanity): loudly report Tier-1 failures so that a silent
    # vocabulary mismatch (the previous bug) can never go unnoticed again.
    n_tier1_fail = int((~tier1).sum())
    if n_tier1_fail:
        excluded_names = audit.index[~tier1].tolist()
        print(f"[shariah_screening] Tier 1 excluded {n_tier1_fail}/{len(audit)} tickers "
              f"on business activity: {excluded_names}")

    compliant = audit.index[audit["is_shariah_compliant"]].tolist()
    n_missing = (~audit["data_available"]).sum()
    if n_missing:
        print(f"[shariah_screening] {n_missing}/{len(audit)} tickers excluded due to missing "
              "fundamental data (rather than a genuine ratio breach) -- see 'data_available' column.")
    cols = ["sector", "passes_tier1", "data_available", "denominator_used", "cash_ratio", "pass_cash",
            "debt_ratio", "pass_debt", "receivables_ratio", "pass_receivables",
            "passes_tier2", "is_shariah_compliant"]
    return compliant, audit[cols]


def filter_price_panel(feature_panel: pd.DataFrame, compliant_tickers: list[str]) -> pd.DataFrame:
    mask = feature_panel.index.get_level_values("ticker").isin(compliant_tickers)
    return feature_panel.loc[mask]


# ==============================================================================
# STAGE 3   |  AI RETURN FORECASTING (ANN)
# ==============================================================================

FEATURE_COLS = [
    "ret_1d", "ret_5d", "ret_21d",
    "sma10", "sma50", "sma200", "ema12", "ema26",
    "macd", "macd_signal", "macd_hist", "rsi14", "roc10",
    "vol21_ann", "bb_width", "atr14", "vol_roc10",
    "px_over_sma50", "sma10_over_sma50",
]
FORWARD_HORIZON = 21

MAX_ABS_ANNUAL_RETURN = 0.60
MAX_ANNUAL_VOL = 0.90
MIN_ANNUAL_VOL = 0.03
# FIX: cap on how much we ever trust the raw ANN forecast. The ANN on this
# universe (19 collinear technical features, ~800 training rows) is largely
# noise-fitting, so we blend with the historical mean using an OOS skill score.
MAX_ANN_TRUST = 0.7

EMPTY_FORECAST_COLUMNS = [
    "exp_return_ann", "exp_vol_ann", "raw_ann_return_ann",
    "hist_mean_return_ann", "shrink_applied",
    "val_rmse_return", "naive_rmse_return", "n_train_obs",
]


def _make_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["fwd_return"] = np.log(df["close"].shift(-FORWARD_HORIZON) / df["close"])
    df["fwd_vol"] = df["ret_1d"].rolling(FORWARD_HORIZON).std().shift(-FORWARD_HORIZON) * np.sqrt(252)
    return df


def _fit_ann(X_train, y_train, seed=7) -> MLPRegressor:
    model = MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=800,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=seed,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        model.fit(X_train, y_train)
    return model


def forecast_universe(feature_panel: pd.DataFrame, compliant_tickers: list[str],
                       min_history: int = 300,
                       max_ann_trust: float = MAX_ANN_TRUST) -> pd.DataFrame:
    """Trains a per-ticker ANN and returns forecasts, shrunk toward the ticker's
    historical mean return/vol using an OOS skill score vs a naive baseline.
    Columns:
      exp_return_ann      — shrunk + clipped annualized expected return (used by Stage 5)
      exp_vol_ann         — shrunk + clipped annualized expected vol   (used by Stage 5)
      raw_ann_return_ann  — raw pre-shrinkage, pre-clip ANN output (for diagnostics)
      hist_mean_return_ann— historical annualized mean return (the prior)
      shrink_applied      — blend weight actually placed on the ANN (0..max_ann_trust)
      val_rmse_return     — ANN OOS RMSE on the test split
      naive_rmse_return   — naive baseline OOS RMSE (predict training mean)
      n_train_obs         — number of training rows used
    """
    if not compliant_tickers:
        print("[ann_forecast] No compliant tickers were passed in -- nothing to forecast.")
        return pd.DataFrame(columns=EMPTY_FORECAST_COLUMNS).rename_axis("ticker")

    results = []

    for ticker in compliant_tickers:
        df = feature_panel.xs(ticker, level="ticker").sort_index()
        df = _make_targets(df)
        model_df = df.dropna(subset=FEATURE_COLS + ["fwd_return", "fwd_vol"])
        if len(model_df) < min_history:
            continue

        X = model_df[FEATURE_COLS].values
        y_ret = model_df["fwd_return"].values
        y_vol = model_df["fwd_vol"].values

        # FIX: belt-and-braces — drop any residual non-finite rows before fit.
        finite = np.isfinite(X).all(axis=1) & np.isfinite(y_ret) & np.isfinite(y_vol)
        if not finite.all():
            n_bad = int((~finite).sum())
            print(f"[ann_forecast] {ticker}: dropping {n_bad} non-finite row(s).")
            X, y_ret, y_vol = X[finite], y_ret[finite], y_vol[finite]
        if len(X) < min_history:
            continue

        split = int(len(X) * 0.8)
        X_train, X_test = X[:split], X[split:]
        y_ret_train, y_ret_test = y_ret[:split], y_ret[split:]
        y_vol_train, y_vol_test = y_vol[:split], y_vol[split:]

        scaler = StandardScaler().fit(X_train)
        X_train_s = scaler.transform(X_train)
        X_test_s = scaler.transform(X_test)

        ret_model = _fit_ann(X_train_s, y_ret_train)
        vol_model = _fit_ann(X_train_s, y_vol_train)

        ret_pred_test = ret_model.predict(X_test_s)
        rmse_ret = float(mean_squared_error(y_ret_test, ret_pred_test) ** 0.5)

        # FIX: compute the OOS skill of the ANN vs the naive "always predict
        # training mean" baseline. If the ANN is no better than naive, we
        # should effectively not trust its raw output.
        naive_pred_test = np.full_like(y_ret_test, float(y_ret_train.mean()))
        naive_rmse = float(mean_squared_error(y_ret_test, naive_pred_test) ** 0.5)
        skill = 1.0 - rmse_ret / max(naive_rmse, 1e-9)
        shrink = float(np.clip(skill, 0.0, max_ann_trust))

        # Historical prior (annualized). fwd_return and fwd_vol are already
        # the model's target variables, so these are directly comparable.
        periods_per_year = 252 / FORWARD_HORIZON
        hist_mu_ann = float(model_df["fwd_return"].mean() * periods_per_year)
        hist_vol_ann = float(model_df["fwd_vol"].mean())

        # Latest fully-observed feature row for live inference
        latest_row = df.dropna(subset=FEATURE_COLS)[FEATURE_COLS].iloc[[-1]].values
        if not np.isfinite(latest_row).all():
            print(f"[ann_forecast] {ticker}: latest feature row has non-finite values, skipping.")
            continue
        latest_scaled = scaler.transform(latest_row)

        pred_fwd_return = float(ret_model.predict(latest_scaled)[0])
        pred_fwd_vol = float(vol_model.predict(latest_scaled)[0])

        raw_ann_ann = float(pred_fwd_return * periods_per_year)

        # Shrink raw ANN output toward historical prior, then clip.
        ret_pre_clip = shrink * raw_ann_ann + (1.0 - shrink) * hist_mu_ann
        exp_return_ann = float(np.clip(ret_pre_clip, -MAX_ABS_ANNUAL_RETURN, MAX_ABS_ANNUAL_RETURN))

        vol_pre_clip = shrink * pred_fwd_vol + (1.0 - shrink) * hist_vol_ann
        exp_vol_ann = float(np.clip(vol_pre_clip, MIN_ANNUAL_VOL, MAX_ANNUAL_VOL))

        if abs(ret_pre_clip) > MAX_ABS_ANNUAL_RETURN:
            print(f"[ann_forecast] {ticker}: shrunk forecast {ret_pre_clip:+.1%} hit the clip "
                  f"(raw ANN {raw_ann_ann:+.1%}, prior {hist_mu_ann:+.1%}, shrink {shrink:.2f})")

        results.append({
            "ticker": ticker,
            "exp_return_ann": exp_return_ann,
            "exp_vol_ann": exp_vol_ann,
            "raw_ann_return_ann": raw_ann_ann,
            "hist_mean_return_ann": hist_mu_ann,
            "shrink_applied": shrink,
            "val_rmse_return": rmse_ret,
            "naive_rmse_return": naive_rmse,
            "n_train_obs": len(X_train),
        })

    if not results:
        print(f"[ann_forecast] {len(compliant_tickers)} compliant ticker(s) passed in, but none had "
              f">= {min_history} clean observations after indicator/target warm-up.")
        return pd.DataFrame(columns=EMPTY_FORECAST_COLUMNS).rename_axis("ticker")

    out = pd.DataFrame(results).set_index("ticker")

    # Diagnostic summary so saturation is impossible to miss.
    n_clipped = int((out["exp_return_ann"].abs() >= MAX_ABS_ANNUAL_RETURN - 1e-9).sum())
    mean_shrink = float(out["shrink_applied"].mean())
    print(f"[ann_forecast] Mean ANN trust weight: {mean_shrink:.2f} "
          f"(0 = ignore ANN, {max_ann_trust:.2f} = cap). "
          f"{n_clipped}/{len(out)} forecasts hit the ±{MAX_ABS_ANNUAL_RETURN:.0%} clip.")
    if n_clipped > 0.3 * len(out):
        print("[ann_forecast] NOTE: >30% of forecasts are clipped. The ANN is not "
              "informative on this universe; consider lowering `max_ann_trust`, adding "
              "features, or increasing history.")

    return out


# ==============================================================================
# STAGE 4   |  INVESTOR PROFILING (ROBO-ADVISOR)
# ==============================================================================

QUESTIONS = [
    {"id": "horizon", "text": "What is your investment time horizon?",
     "options": {"<1 year": 1, "1-3 years": 2, "3-7 years": 3, "7+ years": 4}},
    {"id": "loss_reaction", "text": "If your portfolio fell 20% in a month, what would you do?",
     "options": {"Sell everything immediately": 1, "Sell some to reduce risk": 2,
                 "Hold and wait it out": 3, "Buy more at the lower price": 4}},
    {"id": "income_stability", "text": "How stable is your income / need for liquidity from this portfolio?",
     "options": {"I may need this money soon": 1, "Stable, but I prefer safety": 2,
                 "Stable, comfortable with risk": 3, "Very stable / surplus capital": 4}},
    {"id": "experience", "text": "How would you describe your investing experience?",
     "options": {"None": 1, "Basic": 2, "Experienced": 3, "Very experienced": 4}},
    {"id": "goal", "text": "What is your primary goal?",
     "options": {"Capital preservation": 1, "Income": 2, "Balanced growth": 3, "Maximum growth": 4}},
]


@dataclass
class InvestorProfile:
    raw_score: int
    max_score: int
    risk_category: str
    lambda_risk_aversion: float
    cvar_alpha: float
    max_single_holding: float


def score_questionnaire(answers: dict[str, str]) -> InvestorProfile:
    total, max_total = 0, 0
    for q in QUESTIONS:
        max_total += max(q["options"].values())
        chosen = answers.get(q["id"])
        if chosen not in q["options"]:
            raise ValueError(f"Missing/invalid answer for '{q['id']}': {chosen!r}")
        total += q["options"][chosen]

    pct = total / max_total

    if pct < 0.40:
        category, lam, alpha, cap = "Conservative", 8.0, 0.99, 0.10
    elif pct < 0.60:
        category, lam, alpha, cap = "Moderate", 4.0, 0.97, 0.15
    elif pct < 0.80:
        category, lam, alpha, cap = "Growth", 2.0, 0.95, 0.20
    else:
        category, lam, alpha, cap = "Aggressive", 1.0, 0.90, 0.30

    return InvestorProfile(
        raw_score=total, max_score=max_total, risk_category=category,
        lambda_risk_aversion=lam, cvar_alpha=alpha, max_single_holding=cap,
    )


def run_cli_questionnaire() -> InvestorProfile:
    answers = {}
    for q in QUESTIONS:
        print(f"\n{q['text']}")
        opts = list(q["options"].keys())
        for i, opt in enumerate(opts, 1):
            print(f"  {i}. {opt}")
        choice = int(input("Choose an option number: "))
        answers[q["id"]] = opts[choice - 1]
    return score_questionnaire(answers)


# ==============================================================================
# STAGE 5   |  CONSTRAINED OPTIMISATION (MEAN-CVAR)
# ==============================================================================

ZAKAT_RATE = 0.025
TRANSACTION_COST_BPS = 15
N_SCENARIOS = 5000
RANDOM_SEED = 11


@dataclass
class OptimizationResult:
    weights: pd.Series
    expected_return_gross: float
    expected_return_net_of_costs_and_zakat: float
    cvar: float
    turnover: float
    transaction_cost: float
    zakat_due: float


def _simulate_return_scenarios(forecasts: pd.DataFrame, correlation: pd.DataFrame | None,
                                n_scenarios: int = N_SCENARIOS, seed: int = RANDOM_SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    mu = forecasts["exp_return_ann"].values
    sigma = forecasts["exp_vol_ann"].values
    n_assets = len(mu)

    z = rng.standard_normal((n_scenarios, n_assets))
    if correlation is not None:
        corr = correlation.loc[forecasts.index, forecasts.index].values
        corr = (corr + corr.T) / 2
        eigvals, eigvecs = np.linalg.eigh(corr)
        eigvals = np.clip(eigvals, 1e-8, None)
        corr_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
        L = np.linalg.cholesky(corr_psd)
        z = z @ L.T

    scenarios = mu + z * sigma
    return scenarios


def _portfolio_cvar(weights: np.ndarray, scenario_returns: np.ndarray, alpha: float) -> float:
    port_returns = scenario_returns @ weights
    var_threshold = np.percentile(port_returns, (1 - alpha) * 100)
    tail = port_returns[port_returns <= var_threshold]
    if len(tail) == 0:
        tail = np.array([var_threshold])
    cvar_loss = -tail.mean()
    return float(cvar_loss)


def optimize_portfolio(forecasts: pd.DataFrame,
                        lambda_risk_aversion: float,
                        cvar_alpha: float,
                        max_single_holding: float,
                        current_weights: pd.Series | None = None,
                        correlation: pd.DataFrame | None = None) -> OptimizationResult:
    tickers = forecasts.index.tolist()
    n = len(tickers)
    mu = forecasts["exp_return_ann"].values

    if current_weights is None:
        current_weights = pd.Series(0.0, index=tickers)
    else:
        current_weights = current_weights.reindex(tickers).fillna(0.0)
    w0_current = current_weights.values

    scenarios = _simulate_return_scenarios(forecasts, correlation)

    def objective(w):
        exp_return = mu @ w
        cvar = _portfolio_cvar(w, scenarios, cvar_alpha)
        turnover = np.sum(np.abs(w - w0_current))
        tc = (TRANSACTION_COST_BPS / 10_000) * turnover
        utility = exp_return - lambda_risk_aversion * cvar - tc
        return -utility

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, max_single_holding) for _ in range(n)]
    w_start = np.full(n, 1.0 / n)

    result = minimize(objective, w_start, method="SLSQP", bounds=bounds,
                       constraints=constraints, options={"maxiter": 500, "ftol": 1e-9})

    if not result.success:
        w_final = np.clip(result.x, 0, max_single_holding)
        w_final = w_final / w_final.sum()
    else:
        w_final = result.x
        w_final = np.clip(w_final, 0, max_single_holding)
        w_final = w_final / w_final.sum()

    weights = pd.Series(w_final, index=tickers, name="weight")
    exp_return_gross = float(mu @ w_final)
    cvar_final = _portfolio_cvar(w_final, scenarios, cvar_alpha)
    turnover = float(np.sum(np.abs(w_final - w0_current)))
    tc_final = (TRANSACTION_COST_BPS / 10_000) * turnover
    zakat_due = ZAKAT_RATE
    exp_return_net = exp_return_gross - tc_final - zakat_due

    # FIX: CVaR sanity check. A negative CVaR means the worst (1-alpha) tail of
    # simulated scenarios is still profitable — almost always a symptom of
    # saturated/over-optimistic return forecasts, not a genuinely great portfolio.
    if cvar_final < 0:
        print(f"[optimization] WARNING: CVaR @ {cvar_alpha:.0%} = {cvar_final:+.2%} is NEGATIVE "
              "(worst tail of scenarios is profitable). This usually means the return "
              "forecasts are still too optimistic even after shrinkage — treat the "
              "headline expected return with scepticism.")

    return OptimizationResult(
        weights=weights.sort_values(ascending=False),
        expected_return_gross=exp_return_gross,
        expected_return_net_of_costs_and_zakat=exp_return_net,
        cvar=cvar_final,
        turnover=turnover,
        transaction_cost=tc_final,
        zakat_due=zakat_due,
    )


# ==============================================================================
# STAGE 6   |  PIPELINE ORCHESTRATION & PORTFOLIO OUTPUT
# ==============================================================================

warnings.filterwarnings("ignore", category=ConvergenceWarning)

DEMO_ANSWERS = {
    "horizon": "7+ years",
    "loss_reaction": "Hold and wait it out",
    "income_stability": "Stable, comfortable with risk",
    "experience": "Experienced",
    "goal": "Balanced growth",
}


def run_pipeline(interactive: bool = False) -> None:
    print("=" * 70)
    print("STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)")
    print("=" * 70)
    dataset = ingest_market_data()
    feature_panel = build_feature_panel(dataset.prices, dataset.volumes)
    print(f"Ingested {dataset.prices.shape[1]} tickers x {dataset.prices.shape[0]} trading days.")
    print(f"Feature panel: {feature_panel.shape[0]} (date,ticker) rows x {feature_panel.shape[1]} indicators.\n")

    print("=" * 70)
    print("STAGE 2 — Shariah Universe Screening")
    print("=" * 70)
    compliant_tickers, audit = screen_universe(dataset.meta, dataset.fundamentals)
    print(f"{len(compliant_tickers)} / {len(dataset.meta)} tickers pass business-activity + "
          f"financial-ratio screens:")
    print(compliant_tickers, "\n")
    if not compliant_tickers:
        print("No tickers passed the Shariah screen -- stopping here. Full audit trail:\n", audit)
        return

    print("=" * 70)
    print("STAGE 3 — AI Return Forecasting (ANN)")
    print("=" * 70)
    forecasts = forecast_universe(feature_panel, compliant_tickers)
    if forecasts.empty:
        print("No forecasts were produced -- stopping here.")
        return
    display_cols = ["exp_return_ann", "exp_vol_ann", "raw_ann_return_ann",
                    "hist_mean_return_ann", "shrink_applied",
                    "val_rmse_return", "naive_rmse_return"]
    print(forecasts.sort_values("exp_return_ann", ascending=False)[display_cols].round(4), "\n")

    print("=" * 70)
    print("STAGE 4 — Investor Profiling (Robo-Advisor)")
    print("=" * 70)
    profile = run_cli_questionnaire() if interactive else score_questionnaire(DEMO_ANSWERS)
    print(f"Risk category: {profile.risk_category}")
    print(f"  lambda (risk aversion)   = {profile.lambda_risk_aversion}")
    print(f"  CVaR confidence (alpha)  = {profile.cvar_alpha:.0%}")
    print(f"  Max single holding cap   = {profile.max_single_holding:.0%}\n")

    print("=" * 70)
    print("STAGE 5 — Constrained Optimisation (Mean-CVaR)")
    print("=" * 70)
    # FIX: pct_change(fill_method=None) to silence the pandas FutureWarning.
    hist_returns = dataset.prices[forecasts.index].pct_change(fill_method=None).dropna()
    correlation = hist_returns.corr()
    result = optimize_portfolio(
        forecasts,
        lambda_risk_aversion=profile.lambda_risk_aversion,
        cvar_alpha=profile.cvar_alpha,
        max_single_holding=profile.max_single_holding,
        correlation=correlation,
    )

    print("=" * 70)
    print("STAGE 6 — Portfolio Output")
    print("=" * 70)
    final_weights = result.weights[result.weights > 0.005]
    report = final_weights.to_frame("weight")
    report["sector"] = dataset.meta.loc[report.index, "sector"]
    report["exp_return_ann"] = forecasts.loc[report.index, "exp_return_ann"]
    report["exp_vol_ann"] = forecasts.loc[report.index, "exp_vol_ann"]

    # FIX: hard compliance sanity check on the FINAL portfolio.
    bad_in_final = report.index[report["sector"].isin(NON_COMPLIANT_SECTORS)].tolist()
    if bad_in_final:
        print(f"[pipeline] *** COMPLIANCE FAILURE *** non-compliant tickers ended up in "
              f"the final portfolio: {bad_in_final}. This should be impossible and "
              "indicates an upstream screening bug.")

    print(report.round(4))
    print(f"\nPortfolio-level expected return (gross, annualized): {result.expected_return_gross:.2%}")
    print(f"Portfolio-level expected return (net of TC + zakat) : "
          f"{result.expected_return_net_of_costs_and_zakat:.2%}")
    print(f"Portfolio CVaR @ {profile.cvar_alpha:.0%} confidence          : {result.cvar:.2%}")
    print(f"Turnover from current holdings                       : {result.turnover:.2%}")
    print(f"Transaction cost drag                                : {result.transaction_cost:.4%}")
    print(f"Annual zakat obligation (2.5% of eligible wealth)    : {result.zakat_due:.2%}")

    report.to_csv("/tmp/final_portfolio.csv")
    print("\nSaved final portfolio to /tmp/final_portfolio.csv")


if __name__ == "__main__":
    run_pipeline(interactive=False)

STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)
[data_ingestion] Live data pulled via yfinance for 25/25 requested tickers.
Ingested 25 tickers x 1227 trading days.
Feature panel: 30675 (date,ticker) rows x 21 indicators.

STAGE 2 — Shariah Universe Screening
[shariah_screening] Tier 1 excluded 6/25 tickers on business activity: ['1155.KL', '1295.KL', '3182.KL', '4715.KL', '2836.KL', '3255.KL']
13 / 25 tickers pass business-activity + financial-ratio screens:
['6033.KL', '5183.KL', '1961.KL', '4707.KL', '3689.KL', '3026.KL', '7113.KL', '5168.KL', '5285.KL', '8869.KL', '3816.KL', '5211.KL', '3336.KL'] 

STAGE 3 — AI Return Forecasting (ANN)
[ann_forecast] Mean ANN trust weight: 0.00 (0 = ignore ANN, 0.70 = cap). 0/13 forecasts hit the ±60% clip.
         exp_return_ann  exp_vol_ann  raw_ann_return_ann  \
ticker                                                     
5211.KL          0.3135       0.2573              0.8908   
3336.KL          0.1607       0.3102             -1.537

# improve stage 3

In [ ]:
# =============================================================================
# SHARIAH-COMPLIANT AI ROBO-ADVISOR — FULL PIPELINE (single-file, Colab-ready)
# =============================================================================
# Paste this entire file into one Google Colab cell and run.
# =============================================================================

# !pip install yfinance -q

from __future__ import annotations
import sys
import warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from scipy.optimize import minimize
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ==============================================================================
# STAGE 1a  |  DATA INGESTION (5-YEAR HORIZON)
# ==============================================================================

HORIZON_YEARS = 5
TRADING_DAYS_PER_YEAR = 252

DEFAULT_UNIVERSE = [
    ("1155.KL", "Malayan Banking Bhd (Maybank)", "Conventional Banking"),
    ("1295.KL", "Public Bank Bhd", "Conventional Banking"),
    ("5347.KL", "Tenaga Nasional Bhd", "Utilities"),
    ("6033.KL", "Petronas Gas Bhd", "Energy"),
    ("5183.KL", "Petronas Chemicals Group Bhd", "Materials"),
    ("1961.KL", "IOI Corp Bhd", "Plantation"),
    ("2445.KL", "Kuala Lumpur Kepong Bhd", "Plantation"),
    ("4707.KL", "Nestle Malaysia Bhd", "Consumer Staples"),
    ("3689.KL", "Fraser & Neave Holdings Bhd", "Consumer Staples"),
    ("3026.KL", "Dutch Lady Milk Industries Bhd", "Consumer Staples"),
    ("7113.KL", "Top Glove Corp Bhd", "Health Care Equipment"),
    ("5168.KL", "Hartalega Holdings Bhd", "Health Care Equipment"),
    ("3182.KL", "Genting Bhd", "Gaming & Casinos"),
    ("4715.KL", "Genting Malaysia Bhd", "Gaming & Casinos"),
    ("6888.KL", "Axiata Group Bhd", "Telecommunications"),
    ("6947.KL", "CelcomDigi Bhd (fka Digi.Com)", "Telecommunications"),
    ("6012.KL", "Maxis Bhd", "Telecommunications"),
    ("5285.KL", "SD Guthrie Bhd (fka Sime Darby Plantation)", "Plantation"),
    ("8869.KL", "Press Metal Aluminium Holdings Bhd", "Materials"),
    ("3816.KL", "MISC Bhd", "Shipping/Logistics"),
    ("2836.KL", "Carlsberg Brewery Malaysia Bhd", "Brewery"),
    ("3255.KL", "Heineken Malaysia Bhd", "Brewery"),
    ("4677.KL", "YTL Corp Bhd", "Conglomerate/Utilities"),
    ("5211.KL", "Sunway Bhd", "Property & Construction"),
    ("3336.KL", "IJM Corp Bhd", "Property & Construction"),
]

MIN_VALID_TICKERS = 5


@dataclass
class MarketDataset:
    prices: pd.DataFrame
    volumes: pd.DataFrame
    fundamentals: pd.DataFrame
    meta: pd.DataFrame
    start_date: datetime = field(default=None)
    end_date: datetime = field(default=None)


# -----------------------------------------------------------------------------
# FIX (Stage 2 Tier 1 vocabulary): the live ingestion path re-labels each
# ticker using Yahoo's `industry` string, while the Shariah screen matches on
# the canonical labels in NON_COMPLIANT_SECTORS. Previously the classifier
# emitted "Excluded (brewers)" etc., which never intersected NON_COMPLIANT_SECTORS
# — so Tier 1 passed EVERY live-ingested ticker unconditionally (Maybank and
# Carlsberg were silently passing the screen). We now map industry keywords
# directly onto the canonical sector labels, and assert consistency at import
# time so this class of bug can't silently reappear.
# -----------------------------------------------------------------------------
NON_COMPLIANT_SECTORS = {
    "Conventional Banking",
    "Conventional Insurance",
    "Gaming & Casinos",
    "Brewery",
    "Tobacco",
    "Adult Entertainment",
    "Conventional Leasing",
    "Weapons & Defense",
    "Pork / Non-Halal Food",
}

NON_COMPLIANT_INDUSTRY_KEYWORDS: dict[str, str] = {
    # Yahoo industry substring (lowercase)  ->  canonical NON_COMPLIANT_SECTORS label
    "bank": "Conventional Banking",
    "insurance": "Conventional Insurance",
    "credit services": "Conventional Banking",
    "capital markets": "Conventional Banking",
    "gambling": "Gaming & Casinos",
    "resorts & casinos": "Gaming & Casinos",
    "casino": "Gaming & Casinos",
    "brewers": "Brewery",
    "distillers": "Brewery",
    "wineries": "Brewery",
    "beverages - wineries": "Brewery",
    "tobacco": "Tobacco",
    "aerospace & defense": "Weapons & Defense",
    "adult": "Adult Entertainment",
}


def _assert_vocabulary_consistency() -> None:
    orphans = {v for v in NON_COMPLIANT_INDUSTRY_KEYWORDS.values()
               if v not in NON_COMPLIANT_SECTORS}
    assert not orphans, (
        f"Vocabulary mismatch: {orphans} are used as canonical labels in "
        "NON_COMPLIANT_INDUSTRY_KEYWORDS but are missing from "
        "NON_COMPLIANT_SECTORS. Tier 1 would silently fail to exclude them."
    )


_assert_vocabulary_consistency()


def _classify_industry(industry: str) -> str:
    """Maps a raw Yahoo `industry` string onto a canonical sector label.
    Excluded industries map directly onto a label in NON_COMPLIANT_SECTORS;
    everything else passes through unchanged."""
    ind_lower = (industry or "").lower()
    # Longer keywords first so e.g. "beverages - wineries" wins over "wineries"
    for kw in sorted(NON_COMPLIANT_INDUSTRY_KEYWORDS, key=len, reverse=True):
        if kw in ind_lower:
            return NON_COMPLIANT_INDUSTRY_KEYWORDS[kw]
    return industry or "Unknown"


_TOTAL_ASSETS_KEYS = ["Total Assets"]
_TOTAL_DEBT_KEYS = ["Total Debt"]
_LONG_TERM_DEBT_KEYS = ["Long Term Debt"]
_CURRENT_DEBT_KEYS = ["Current Debt", "Current Debt And Capital Lease Obligation"]
_CASH_KEYS = ["Cash Cash Equivalents And Short Term Investments", "Cash And Cash Equivalents"]
_RECEIVABLES_KEYS = ["Receivables", "Accounts Receivable"]


def _bs_lookup(balance_sheet: pd.DataFrame, candidates: list[str]) -> float:
    if balance_sheet is None or balance_sheet.empty:
        return np.nan
    col = balance_sheet.columns[0]
    idx_lower = {str(i).strip().lower(): i for i in balance_sheet.index}
    for cand in candidates:
        key = cand.strip().lower()
        if key in idx_lower:
            val = balance_sheet.loc[idx_lower[key], col]
            if pd.notna(val):
                return float(val)
        for lower_name, orig_name in idx_lower.items():
            if key in lower_name:
                val = balance_sheet.loc[orig_name, col]
                if pd.notna(val):
                    return float(val)
    return np.nan


def _fetch_fundamentals(ticker_obj) -> dict:
    balance_sheet = None
    try:
        balance_sheet = ticker_obj.quarterly_balance_sheet
        if balance_sheet is None or balance_sheet.empty:
            balance_sheet = ticker_obj.balance_sheet
    except Exception:
        pass

    total_assets = _bs_lookup(balance_sheet, _TOTAL_ASSETS_KEYS)
    total_debt = _bs_lookup(balance_sheet, _TOTAL_DEBT_KEYS)
    if np.isnan(total_debt):
        ltd = _bs_lookup(balance_sheet, _LONG_TERM_DEBT_KEYS)
        std = _bs_lookup(balance_sheet, _CURRENT_DEBT_KEYS)
        if not (np.isnan(ltd) and np.isnan(std)):
            total_debt = np.nansum([ltd, std])
    cash = _bs_lookup(balance_sheet, _CASH_KEYS)
    receivables = _bs_lookup(balance_sheet, _RECEIVABLES_KEYS)

    market_cap = np.nan
    try:
        market_cap = ticker_obj.fast_info.get("market_cap", np.nan)
    except Exception:
        pass
    if market_cap is None or (isinstance(market_cap, float) and np.isnan(market_cap)):
        try:
            market_cap = ticker_obj.info.get("marketCap", np.nan)
        except Exception:
            market_cap = np.nan

    return {
        "market_cap": market_cap,
        "total_assets": total_assets,
        "total_debt": total_debt,
        "cash_and_interest_securities": cash,
        "receivables": receivables,
    }


def _try_live_ingestion(tickers, start, end) -> MarketDataset | None:
    try:
        import yfinance as yf
    except ImportError:
        print("[data_ingestion] yfinance not installed.")
        return None

    price_series, volume_series, fundamentals_rows = {}, {}, []

    for t in tickers:
        tk = yf.Ticker(t)
        try:
            hist = tk.history(start=start, end=end, auto_adjust=True)
            if hist is None or hist.empty or hist["Close"].dropna().empty:
                print(f"[data_ingestion] No price history for {t}, skipping.")
                continue
            price_series[t] = hist["Close"]
            volume_series[t] = hist["Volume"]
        except Exception as e:
            print(f"[data_ingestion] Price fetch failed for {t}: {e}")
            continue

        try:
            info = tk.info
            row = _fetch_fundamentals(tk)
            row["ticker"] = t
            row["sector"] = _classify_industry(info.get("industry", info.get("sector")))
            fundamentals_rows.append(row)
        except Exception as e:
            print(f"[data_ingestion] Fundamentals fetch failed for {t}: {e} (will be NaN).")
            fundamentals_rows.append({
                "ticker": t, "market_cap": np.nan, "total_debt": np.nan,
                "total_assets": np.nan, "cash_and_interest_securities": np.nan,
                "receivables": np.nan, "sector": "Unknown",
            })

    if len(price_series) < MIN_VALID_TICKERS:
        print(f"[data_ingestion] Only {len(price_series)} tickers returned live price data "
              f"(< {MIN_VALID_TICKERS} minimum) -> treating live pull as failed.")
        return None

    prices = pd.DataFrame(price_series).sort_index()
    volumes = pd.DataFrame(volume_series).sort_index()
    fundamentals = pd.DataFrame(fundamentals_rows).set_index("ticker").reindex(prices.columns)
    meta = fundamentals[["sector"]].copy()

    n_missing_assets = fundamentals["total_assets"].isna().sum()
    n_missing_core = fundamentals[["total_debt", "cash_and_interest_securities", "receivables"]].isna().any(axis=1).sum()
    if n_missing_assets or n_missing_core:
        print(f"[data_ingestion] Note: {n_missing_assets}/{len(fundamentals)} tickers are missing "
              f"'total_assets' from the balance sheet (Stage 2 will fall back to market-cap as the "
              f"screening denominator for those); {n_missing_core}/{len(fundamentals)} are missing "
              "debt/cash/receivables entirely and will fail the financial-ratio screen outright.")

    return MarketDataset(prices, volumes, fundamentals, meta, start, end)


def _synthetic_universe(universe, start, end, seed: int = 42) -> MarketDataset:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end)
    n = len(dates)

    tickers = [u[0] for u in universe]
    names = {u[0]: u[1] for u in universe}
    sectors = {u[0]: u[2] for u in universe}

    prices, volumes, fundamentals_rows = {}, {}, []

    for i, t in enumerate(tickers):
        mu = rng.uniform(0.04, 0.12) / TRADING_DAYS_PER_YEAR
        sigma = rng.uniform(0.15, 0.45) / np.sqrt(TRADING_DAYS_PER_YEAR)
        s0 = rng.uniform(1.0, 25.0)
        shocks = rng.normal(mu - 0.5 * sigma ** 2, sigma, n)
        price_path = s0 * np.exp(np.cumsum(shocks))
        prices[t] = price_path

        base_vol = rng.uniform(2e5, 8e6)
        vol_series = np.abs(rng.normal(base_vol, base_vol * 0.3, n)).astype(int)
        volumes[t] = vol_series

        market_cap = price_path[-1] * rng.uniform(2e8, 6e9)
        is_bank_or_brewer = sectors[t] in ("Conventional Banking", "Brewery")
        debt_ratio = rng.uniform(0.35, 0.55) if is_bank_or_brewer else rng.uniform(0.05, 0.30)
        cash_ratio = rng.uniform(0.35, 0.60) if is_bank_or_brewer else rng.uniform(0.05, 0.28)
        recv_ratio = rng.uniform(0.10, 0.30)

        total_assets = market_cap * rng.uniform(0.8, 1.5)
        fundamentals_rows.append({
            "ticker": t,
            "market_cap": market_cap,
            "total_assets": total_assets,
            "total_debt": debt_ratio * total_assets,
            "cash_and_interest_securities": cash_ratio * total_assets,
            "receivables": recv_ratio * total_assets,
            "sector": sectors[t],
        })

    prices_df = pd.DataFrame(prices, index=dates)
    volumes_df = pd.DataFrame(volumes, index=dates)
    fundamentals_df = pd.DataFrame(fundamentals_rows).set_index("ticker")
    meta_df = pd.DataFrame({"name": names, "sector": sectors})

    return MarketDataset(prices_df, volumes_df, fundamentals_df, meta_df, start, end)


def ingest_market_data(universe=None, years: int = HORIZON_YEARS) -> MarketDataset:
    universe = universe or DEFAULT_UNIVERSE
    end = datetime.today()
    start = end - timedelta(days=int(years * 365.25))
    tickers = [u[0] for u in universe]

    live = _try_live_ingestion(tickers, start, end)
    if live is not None:
        print(f"[data_ingestion] Live data pulled via yfinance for "
              f"{live.prices.shape[1]}/{len(tickers)} requested tickers.")
        return live

    print("[data_ingestion] No network / yfinance unavailable -> using synthetic "
          f"{years}y dataset for {len(universe)} tickers (schema-identical to live pull).")
    return _synthetic_universe(universe, start, end)


# ==============================================================================
# STAGE 1b  |  TECHNICAL INDICATOR ENGINEERING
# ==============================================================================

def _rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _macd(series: pd.Series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line


def _atr(close: pd.Series, window: int = 14) -> pd.Series:
    tr = close.diff().abs()
    return tr.rolling(window).mean()


def _obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    """FIX: scale-free OBV (rolling 1y z-score) instead of raw cumulative sum.
    Raw cumulative OBV grows into the billions, which is a pathological input
    for StandardScaler + MLP; a rolling z-score keeps the feature O(1)."""
    direction = np.sign(close.diff().fillna(0))
    raw_obv = (direction * volume).cumsum()
    roll_mean = raw_obv.rolling(252, min_periods=60).mean()
    roll_std = raw_obv.rolling(252, min_periods=60).std()
    return (raw_obv - roll_mean) / roll_std.replace(0, np.nan)


def compute_indicators_for_ticker(close: pd.Series, volume: pd.Series) -> pd.DataFrame:
    log_ret = np.log(close / close.shift(1))

    sma10 = close.rolling(10).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line, signal_line = _macd(close)

    roll_std21 = log_ret.rolling(21).std() * np.sqrt(252)
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_width = (bb_mid + 2 * bb_std - (bb_mid - 2 * bb_std)) / bb_mid

    feats = pd.DataFrame({
        "close": close,
        "ret_1d": log_ret,
        "ret_5d": np.log(close / close.shift(5)),
        "ret_21d": np.log(close / close.shift(21)),
        "sma10": sma10, "sma50": sma50, "sma200": sma200,
        "ema12": ema12, "ema26": ema26,
        "macd": macd_line, "macd_signal": signal_line, "macd_hist": macd_line - signal_line,
        "rsi14": _rsi(close, 14),
        "roc10": close.pct_change(10, fill_method=None) * 100,
        "vol21_ann": roll_std21,
        "bb_width": bb_width,
        "atr14": _atr(close, 14),
        "vol_roc10": volume.pct_change(10, fill_method=None) * 100,
        "obv": _obv(close, volume),
    })
    feats["px_over_sma50"] = close / sma50 - 1
    feats["sma10_over_sma50"] = sma10 / sma50 - 1

    # FIX: pct_change on zero volume (halted/thin Bursa counters) produces ±inf.
    # Convert to NaN so downstream dropna() excludes those rows cleanly instead
    # of feeding inf into StandardScaler / the ANN.
    feats = feats.replace([np.inf, -np.inf], np.nan)
    return feats


def build_feature_panel(prices: pd.DataFrame, volumes: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for ticker in prices.columns:
        f = compute_indicators_for_ticker(prices[ticker], volumes[ticker])
        f["ticker"] = ticker
        frames.append(f)
    panel = pd.concat(frames)
    panel = panel.set_index("ticker", append=True)
    panel.index.names = ["date", "ticker"]
    return panel.sort_index()


# ==============================================================================
# STAGE 2   |  SHARIAH UNIVERSE SCREENING
# ==============================================================================

RATIO_THRESHOLDS = {
    "cash_ratio": 0.33,
    "debt_ratio": 0.33,
    "receivables_ratio": 0.50,
}


def business_activity_screen(meta: pd.DataFrame) -> pd.Series:
    """Tier 1: returns a boolean Series indexed by ticker, True = passes."""
    return ~meta["sector"].isin(NON_COMPLIANT_SECTORS)


def financial_ratio_screen(fundamentals: pd.DataFrame,
                            denominator: str = "assets") -> pd.DataFrame:
    primary = fundamentals["total_assets"] if denominator == "assets" else fundamentals["market_cap"]
    fallback = fundamentals["market_cap"] if denominator == "assets" else fundamentals["total_assets"]
    denom = primary.where(primary.notna() & (primary != 0), fallback)

    ratios = pd.DataFrame(index=fundamentals.index)
    ratios["denominator_used"] = np.where(
        primary.notna() & (primary != 0), denominator,
        np.where(fallback.notna() & (fallback != 0), f"{denominator}_fallback", "unavailable"),
    )

    required = ["cash_and_interest_securities", "total_debt", "receivables"]
    ratios["data_available"] = fundamentals[required].notna().all(axis=1) & denom.notna() & (denom != 0)

    ratios["cash_ratio"] = fundamentals["cash_and_interest_securities"] / denom
    ratios["debt_ratio"] = fundamentals["total_debt"] / denom
    ratios["receivables_ratio"] = (fundamentals["receivables"] + fundamentals["cash_and_interest_securities"]) / denom

    ratios["pass_cash"] = ratios["data_available"] & (ratios["cash_ratio"] < RATIO_THRESHOLDS["cash_ratio"])
    ratios["pass_debt"] = ratios["data_available"] & (ratios["debt_ratio"] < RATIO_THRESHOLDS["debt_ratio"])
    ratios["pass_receivables"] = ratios["data_available"] & (ratios["receivables_ratio"] < RATIO_THRESHOLDS["receivables_ratio"])
    ratios["passes_tier2"] = ratios["data_available"] & ratios[["pass_cash", "pass_debt", "pass_receivables"]].all(axis=1)
    return ratios


def screen_universe(meta: pd.DataFrame, fundamentals: pd.DataFrame,
                     denominator: str = "assets") -> tuple[list[str], pd.DataFrame]:
    tier1 = business_activity_screen(meta)
    tier2 = financial_ratio_screen(fundamentals, denominator=denominator)

    audit = tier2.copy()
    audit["sector"] = meta["sector"]
    audit["passes_tier1"] = tier1
    audit["is_shariah_compliant"] = audit["passes_tier1"] & audit["passes_tier2"]

    # FIX (Stage 2 sanity): loudly report Tier-1 failures so that a silent
    # vocabulary mismatch (the previous bug) can never go unnoticed again.
    n_tier1_fail = int((~tier1).sum())
    if n_tier1_fail:
        excluded_names = audit.index[~tier1].tolist()
        print(f"[shariah_screening] Tier 1 excluded {n_tier1_fail}/{len(audit)} tickers "
              f"on business activity: {excluded_names}")

    compliant = audit.index[audit["is_shariah_compliant"]].tolist()
    n_missing = (~audit["data_available"]).sum()
    if n_missing:
        print(f"[shariah_screening] {n_missing}/{len(audit)} tickers excluded due to missing "
              "fundamental data (rather than a genuine ratio breach) -- see 'data_available' column.")
    cols = ["sector", "passes_tier1", "data_available", "denominator_used", "cash_ratio", "pass_cash",
            "debt_ratio", "pass_debt", "receivables_ratio", "pass_receivables",
            "passes_tier2", "is_shariah_compliant"]
    return compliant, audit[cols]


def filter_price_panel(feature_panel: pd.DataFrame, compliant_tickers: list[str]) -> pd.DataFrame:
    mask = feature_panel.index.get_level_values("ticker").isin(compliant_tickers)
    return feature_panel.loc[mask]


# =============================================================================
# STAGE 3  |  RETURN FORECASTING (CNN-LSTM + FUNDAMENTAL MOMENTUM) + STRESS TEST
# =============================================================================
# Replaces the previous MLPRegressor-based ann_forecast stage.
#
# Design notes
# ------------
# * Architecture: 1-D CNN over the feature axis followed by an LSTM over the
#   time axis. The CNN learns local interactions between features (e.g. RSI
#   crossed with a positive earnings revision), the LSTM learns how those
#   interactions evolve over the lookback window. This is the standard
#   CNN-LSTM hybrid used in the literature (Lu et al. 2020; Xu 2022).
# * Features: price-based technicals from Stage 1b, PLUS forward-looking
#   fundamental features (earnings revision momentum, surprise history,
#   recommendation drift) fetched from yfinance where available. If a ticker
#   has no fundamental coverage, the fundamental block is zero-filled and a
#   flag column records that so downstream consumers know.
# * Shrinkage is retained: the CNN-LSTM is blended with the ticker's
#   historical mean using an OOS skill score vs a naive baseline, so if the
#   model is not informative on a given universe it cannot dominate.
# * Output columns are explicitly labelled (`forecast_source`) so that a
#   report cannot accidentally present a backtested mean as an AI forecast.
# * Stress testing is a separate, composable step (`stress_test_forecasts`)
#   that runs after forecasting and before optimisation.
# =============================================================================

import math
from typing import Protocol, Iterable

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# -----------------------------------------------------------------------------
# Feature configuration
# -----------------------------------------------------------------------------

# Price/technical features from Stage 1b (unchanged)
TECHNICAL_FEATURE_COLS = [
    "ret_1d", "ret_5d", "ret_21d",
    "sma10", "sma50", "sma200", "ema12", "ema26",
    "macd", "macd_signal", "macd_hist", "rsi14", "roc10",
    "vol21_ann", "bb_width", "atr14", "vol_roc10",
    "px_over_sma50", "sma10_over_sma50",
]

# Forward-looking fundamental features. These are *not* derivable from price
# history; they must be fetched from an external source. yfinance provides
# several of them via Ticker.earnings_estimate / eps_revisions /
# recommendations / upgrades_downgrades.
FUNDAMENTAL_FEATURE_COLS = [
    "eps_revision_90d",       # % change in FY1 EPS estimate over trailing 90d
    "eps_revision_30d",       # % change over trailing 30d (acceleration signal)
    "revision_breadth",       # (up - down) / (up + down) over trailing 90d
    "recommendation_drift",   # change in mean recommendation score, 90d
    "surprise_history",       # mean EPS surprise over last 4 quarters
    "est_growth_fy1",         # consensus FY1 YoY growth estimate
]

ALL_FEATURE_COLS = TECHNICAL_FEATURE_COLS + FUNDAMENTAL_FEATURE_COLS

FORWARD_HORIZON = 21            # trading days
LOOKBACK = 60                   # CNN-LSTM input sequence length
MAX_ABS_ANNUAL_RETURN = 0.60
MAX_ANNUAL_VOL = 0.90
MIN_ANNUAL_VOL = 0.03
MAX_MODEL_TRUST = 0.70
MIN_HISTORY = 300


EMPTY_FORECAST_COLUMNS = [
    "exp_return_ann", "exp_vol_ann",
    "raw_model_return_ann", "hist_mean_return_ann",
    "model_trust", "forecast_source", "model_used",
    "fundamental_coverage",
    "val_rmse_return", "naive_rmse_return", "n_train_obs",
]


# -----------------------------------------------------------------------------
# External feature provider protocol
# -----------------------------------------------------------------------------
# Anything that can produce a DataFrame indexed by (date, ticker) with the
# FUNDAMENTAL_FEATURE_COLS columns is a valid provider. This lets you swap
# yfinance for IBES / Bloomberg / a local CSV without touching model code.

class FundamentalFeatureProvider(Protocol):
    def fetch(self, tickers: list[str], start, end) -> pd.DataFrame:
        ...


class YFinanceFundamentalProvider:
    """Best-effort fetch of forward-looking features from yfinance.

    yfinance's coverage of analyst estimates is uneven outside US large caps.
    For each ticker we try to build the six fundamental features from
    Ticker.earnings_estimate, Ticker.eps_revisions, Ticker.recommendations and
    Ticker.earnings_history. Any feature we cannot build is left NaN and
    imputed later. The provider never raises on a single-ticker failure.
    """

    def __init__(self, verbose: bool = True):
        self.verbose = verbose

    def _safe(self, fn, default=None):
        try:
            v = fn()
            return v if v is not None else default
        except Exception:
            return default

    def _one(self, ticker: str) -> dict | None:
        try:
            import yfinance as yf
        except ImportError:
            return None

        tk = yf.Ticker(ticker)

        eps_est = self._safe(lambda: tk.earnings_estimate)
        eps_rev = self._safe(lambda: tk.eps_revisions)
        recs = self._safe(lambda: tk.recommendations)
        hist = self._safe(lambda: tk.earnings_history)

        def _pct(a, b):
            try:
                a, b = float(a), float(b)
                if b == 0 or math.isnan(a) or math.isnan(b):
                    return np.nan
                return (a - b) / abs(b)
            except Exception:
                return np.nan

        out = {c: np.nan for c in FUNDAMENTAL_FEATURE_COLS}

        # eps_revision_90d / _30d: growth in FY1 EPS estimate across periods.
        # yfinance exposes these as a small frame indexed by period
        # ("0q", "+1q", "0y", "+1y") with current / 7d ago / 30d ago /
        # 60d ago / 90d ago columns.
        if eps_est is not None and not eps_est.empty:
            try:
                row = eps_est.loc["0y"] if "0y" in eps_est.index else eps_est.iloc[0]
                out["eps_revision_90d"] = _pct(row.get("avg"), row.get("90daysAgo"))
                out["eps_revision_30d"] = _pct(row.get("avg"), row.get("30daysAgo"))
                out["est_growth_fy1"] = _pct(row.get("avg"),
                                             row.get("yearAgoEps"))
            except Exception:
                pass

        # revision_breadth: up / (up + down) over trailing 90d
        if eps_rev is not None and not eps_rev.empty:
            try:
                row = eps_rev.loc["0y"] if "0y" in eps_rev.index else eps_rev.iloc[0]
                up = float(row.get("upLast90days", 0) or 0)
                down = float(row.get("downLast90days", 0) or 0)
                if up + down > 0:
                    out["revision_breadth"] = up / (up + down)
            except Exception:
                pass

        # recommendation_drift: mean rec score now vs 90d ago.
        # yfinance's `recommendations` summary frame has rows like
        # "strongBuy" / "buy" / "hold" ... with current / 1m / 2m / 3m columns.
        if recs is not None and not recs.empty:
            try:
                weights = {"strongBuy": 1.0, "buy": 2.0, "hold": 3.0,
                           "sell": 4.0, "strongSell": 5.0}
                def _mean_score(col):
                    num = den = 0.0
                    for k, w in weights.items():
                        if k in recs.index:
                            n = float(recs.loc[k, col] or 0)
                            num += w * n
                            den += n
                    return num / den if den > 0 else np.nan
                now = _mean_score("current")
                ago = _mean_score("3m Ago") if "3m Ago" in recs.columns else np.nan
                if not (math.isnan(now) or math.isnan(ago)):
                    out["recommendation_drift"] = now - ago
            except Exception:
                pass

        # surprise_history: mean EPS surprise over last 4 quarters (fractional)
        if hist is not None and not hist.empty:
            try:
                col = "surprisePercent"
                if col in hist.columns:
                    vals = hist[col].dropna().tail(4)
                    if len(vals):
                        out["surprise_history"] = float(vals.mean()) / 100.0
            except Exception:
                pass

        if all(np.isnan(v) for v in out.values()):
            return None
        return out

    def fetch(self, tickers: list[str], start, end) -> pd.DataFrame:
        rows = []
        for t in tickers:
            feats = self._one(t)
            if feats is None:
                if self.verbose:
                    print(f"[fundamentals] No analyst/fundamental coverage for {t}")
                continue
            feats["ticker"] = t
            rows.append(feats)
        if not rows:
            return pd.DataFrame(columns=FUNDAMENTAL_FEATURE_COLS + ["ticker"])
        return pd.DataFrame(rows)


def build_forward_looking_panel(
    technical_panel: pd.DataFrame,
    tickers: list[str],
    provider: FundamentalFeatureProvider | None = None,
) -> tuple[pd.DataFrame, pd.Series]:
    """Merge fundamental features into the technical panel.

    Returns (panel_with_fundamentals, coverage_series).

    Forward-fill semantics: analyst estimate snapshots are point-in-time, not
    a time series. We treat each ticker's latest snapshot as valid from the
    start of the sample. This is an approximation — a production system would
    use a point-in-time database with proper as-of joins. The approximation
    is honest about itself via `coverage_series`, which downstream code uses
    to decide whether to trust the fundamental block.
    """
    provider = provider or YFinanceFundamentalProvider()
    fund = provider.fetch(tickers, None, None)

    if fund.empty:
        print("[stage3] No fundamental features available — the CNN-LSTM will "
              "run on technicals only. Consider supplying a custom provider.")
        for c in FUNDAMENTAL_FEATURE_COLS:
            technical_panel[c] = np.nan
        coverage = pd.Series(False, index=tickers, name="fundamental_coverage")
        return technical_panel, coverage

    fund = fund.set_index("ticker").reindex(tickers)

    # Broadcast each ticker's snapshot across its date index (documented
    # approximation — see docstring).
    for c in FUNDAMENTAL_FEATURE_COLS:
        if c not in fund.columns:
            technical_panel[c] = np.nan
            continue
        snap = fund[c].to_dict()
        idx = technical_panel.index
        technical_panel[c] = pd.Series(
            [snap.get(t, np.nan) for _, t in idx], index=idx
        )

    coverage = fund[FUNDAMENTAL_FEATURE_COLS].notna().any(axis=1)
    print(f"[stage3] Fundamental coverage: {int(coverage.sum())}/{len(tickers)} tickers")
    return technical_panel, coverage


# -----------------------------------------------------------------------------
# CNN-LSTM model
# -----------------------------------------------------------------------------
# Input  shape: (batch, seq_len=LOOKBACK, n_features)
# Conv1d expects (batch, channels, length); we treat the feature axis as
# channels and the time axis as length, so the CNN sees, for each timestep in
# the window, the local interaction pattern across features. The LSTM then
# processes the CNN's output sequence over time.

class CNNLSTMRegressor(nn.Module):
    def __init__(self, n_features: int, hidden_cnn: int = 32,
                 kernel_size: int = 3, hidden_lstm: int = 32,
                 dropout: float = 0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, hidden_cnn, kernel_size, padding=kernel_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(
            input_size=hidden_cnn,
            hidden_size=hidden_lstm,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_lstm, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        # x: (batch, seq, n_feat)
        x = x.transpose(1, 2)             # -> (batch, n_feat, seq)
        x = self.conv(x)                  # -> (batch, hidden_cnn, seq)
        x = x.transpose(1, 2)             # -> (batch, seq, hidden_cnn)
        out, _ = self.lstm(x)             # -> (batch, seq, hidden_lstm)
        last = out[:, -1, :]              # last timestep
        return self.head(last).squeeze(-1)


def _make_sequences(X: np.ndarray, y: np.ndarray, lookback: int):
    """Sliding-window sequence builder. Returns (X_seq, y_seq)."""
    n = len(X)
    if n <= lookback:
        return None, None
    Xs = np.stack([X[i - lookback:i] for i in range(lookback, n)])
    ys = y[lookback:]
    return Xs, ys


def _fit_cnn_lstm(X_train, y_train, n_features, seed=7,
                  epochs=60, batch_size=64, lr=1e-3,
                  device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    model = CNNLSTMRegressor(n_features=n_features).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    )
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
    return model


@torch.no_grad()
def _predict(model, X, device=None):
    device = device or next(model.parameters()).device
    model.eval()
    xb = torch.tensor(X, dtype=torch.float32).to(device)
    return model(xb).cpu().numpy()


# -----------------------------------------------------------------------------
# Forecast driver
# -----------------------------------------------------------------------------

def _make_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["fwd_return"] = np.log(df["close"].shift(-FORWARD_HORIZON) / df["close"])
    df["fwd_vol"] = (df["ret_1d"].rolling(FORWARD_HORIZON).std()
                     .shift(-FORWARD_HORIZON) * np.sqrt(252))
    return df


def forecast_universe(
    feature_panel: pd.DataFrame,
    compliant_tickers: list[str],
    fundamental_provider: FundamentalFeatureProvider | None = None,
    min_history: int = MIN_HISTORY,
    max_model_trust: float = MAX_MODEL_TRUST,
) -> pd.DataFrame:
    """Trains a per-ticker CNN-LSTM, shrinks toward the historical mean using
    an OOS skill score, and returns the forecast table used by Stage 5.

    Every row carries an explicit `forecast_source` label:
      * "cnn_lstm"           — the model beat the naive baseline and its
                               output dominates the blend
      * "historical_mean"    — the model had no OOS skill; the prior is used
      * "blend"              — partial trust, weight recorded in `model_trust`
    """
    if not compliant_tickers:
        print("[stage3] No compliant tickers — nothing to forecast.")
        return pd.DataFrame(columns=EMPTY_FORECAST_COLUMNS).rename_axis("ticker")

    # Merge in forward-looking features
    panel, coverage = build_forward_looking_panel(
        feature_panel.copy(), compliant_tickers, fundamental_provider,
    )

    results = []
    device = "cuda" if torch.cuda.is_available() else "cpu"

    for ticker in compliant_tickers:
        try:
            df = panel.xs(ticker, level="ticker").sort_index()
        except KeyError:
            continue
        df = _make_targets(df)

        # Fundamental NaNs are not fatal — we impute with the cross-sectional
        # median for that ticker (which, if the ticker has no coverage at all,
        # is itself NaN → the whole fundamental block is dropped for it).
        for c in FUNDAMENTAL_FEATURE_COLS:
            if df[c].isna().all():
                df[c] = 0.0
            else:
                df[c] = df[c].fillna(df[c].median())

        # Only require technicals to be fully observed; fundamentals were
        # zero/median-imputed above.
        model_df = df.dropna(subset=TECHNICAL_FEATURE_COLS + ["fwd_return", "fwd_vol"])
        model_df = model_df[np.isfinite(model_df[ALL_FEATURE_COLS]).all(axis=1)]
        if len(model_df) < min_history:
            continue

        X = model_df[ALL_FEATURE_COLS].values
        y_ret = model_df["fwd_return"].values

        # Build sequences for the CNN-LSTM
        X_seq, y_seq = _make_sequences(X, y_ret, LOOKBACK)
        if X_seq is None or len(X_seq) < min_history:
            continue

        split = int(len(X_seq) * 0.8)
        if split < 50 or len(X_seq) - split < 20:
            continue

        # Fit a scaler on the training folds' raw features. We flatten the
        # sequence axis so the scaler sees all timesteps.
        flat_train = X_seq[:split].reshape(-1, len(ALL_FEATURE_COLS))
        mu = flat_train.mean(axis=0)
        sd = flat_train.std(axis=0)
        sd[sd == 0] = 1.0

        def _scale(seq):
            return (seq - mu) / sd

        X_train_s = _scale(X_seq[:split])
        X_test_s = _scale(X_seq[split:])
        y_train, y_test = y_seq[:split], y_seq[split:]

        model = _fit_cnn_lstm(X_train_s, y_train, n_features=len(ALL_FEATURE_COLS),
                               device=device)

        pred_test = _predict(model, X_test_s, device)
        rmse = float(np.sqrt(mean_squared_error(y_test, pred_test)))
        naive_pred = np.full_like(y_test, float(y_train.mean()))
        naive_rmse = float(np.sqrt(mean_squared_error(y_test, naive_pred)))

        # OOS skill vs naive baseline; 0 => do not trust the model at all.
        skill = 1.0 - rmse / max(naive_rmse, 1e-9)
        trust = float(np.clip(skill, 0.0, max_model_trust))

        # Latest sequence for live inference
        last_seq = _scale(X[-LOOKBACK:][None, :, :])
        raw_pred = float(_predict(model, last_seq, device)[0])

        periods_per_year = 252 / FORWARD_HORIZON
        raw_ann = raw_pred * periods_per_year
        hist_mu_ann = float(model_df["fwd_return"].mean() * periods_per_year)
        hist_vol_ann = float(model_df["fwd_vol"].mean())

        ret_pre = trust * raw_ann + (1.0 - trust) * hist_mu_ann
        exp_ret = float(np.clip(ret_pre, -MAX_ABS_ANNUAL_RETURN, MAX_ABS_ANNUAL_RETURN))
        exp_vol = float(np.clip(hist_vol_ann, MIN_ANNUAL_VOL, MAX_ANNUAL_VOL))

        if trust < 0.05:
            source = "historical_mean"
        elif trust < max_model_trust - 0.05:
            source = "blend"
        else:
            source = "cnn_lstm"

        results.append({
            "ticker": ticker,
            "exp_return_ann": exp_ret,
            "exp_vol_ann": exp_vol,
            "raw_model_return_ann": raw_ann,
            "hist_mean_return_ann": hist_mu_ann,
            "model_trust": trust,
            "forecast_source": source,
            "model_used": "cnn_lstm",
            "fundamental_coverage": bool(coverage.get(ticker, False)),
            "val_rmse_return": rmse,
            "naive_rmse_return": naive_rmse,
            "n_train_obs": int(len(X_train_s)),
        })

    if not results:
        print(f"[stage3] {len(compliant_tickers)} tickers passed in, none had enough "
              f"clean history (>= {min_history} sequences after warm-up).")
        return pd.DataFrame(columns=EMPTY_FORECAST_COLUMNS).rename_axis("ticker")

    out = pd.DataFrame(results).set_index("ticker")

    # Diagnostic summary — impossible to miss now.
    n_hist = int((out["forecast_source"] == "historical_mean").sum())
    n_blend = int((out["forecast_source"] == "blend").sum())
    n_model = int((out["forecast_source"] == "cnn_lstm").sum())
    print(f"[stage3] Forecast sources: {n_model} cnn_lstm / {n_blend} blend / "
          f"{n_hist} historical_mean. "
          f"Mean model trust: {out['model_trust'].mean():.2f}.")
    if n_model == 0:
        print("[stage3] NOTE: no ticker cleared the skill bar. All expected returns "
              "in this run are historical means, not model forecasts. Report them "
              "as such.")

    return out


# -----------------------------------------------------------------------------
# Stress testing
# -----------------------------------------------------------------------------

# Historical crisis windows. Each entry maps to (start, end, label) and is
# applied as a return/vol haircut to the forecasts. Values are approximate
# peak-to-trough equity drawdowns, expressed as multipliers on the *forecast*
# annualized return and additive adjustments to vol.
#
# These are deliberately conservative. A robo-advisor should never quote a
# forecast that has not been shown to survive at least one crisis.
HISTORICAL_STRESS_SCENARIOS = {
    "2008_gfc":        {"return_mult": 0.05, "vol_add": 0.35},
    "2020_covid":      {"return_mult": 0.30, "vol_add": 0.25},
    "2000_dotcom":     {"return_mult": 0.10, "vol_add": 0.30},
    "2022_rate_shock": {"return_mult": 0.45, "vol_add": 0.20},
}


@dataclass
class StressTestResult:
    scenario: str
    stressed_returns: pd.Series       # annualized, per ticker
    stressed_vols: pd.Series
    portfolio_return_under_stress: float


def stress_test_forecasts(
    forecasts: pd.DataFrame,
    scenarios: dict | None = None,
) -> list[StressTestResult]:
    """Applies each historical scenario as a multiplicative haircut on the
    expected return and an additive shock to vol. Returns one result per
    scenario, per ticker, plus the equal-weighted portfolio return under
    stress (which is the headline number a client actually cares about).
    """
    scenarios = scenarios or HISTORICAL_STRESS_SCENARIOS
    out = []
    for name, s in scenarios.items():
        r = forecasts["exp_return_ann"] * s["return_mult"]
        v = (forecasts["exp_vol_ann"] + s["vol_add"]).clip(upper=MAX_ANNUAL_VOL)
        port_r = float(r.mean())  # equal-weighted, deliberately crude
        out.append(StressTestResult(
            scenario=name,
            stressed_returns=r,
            stressed_vols=v,
            portfolio_return_under_stress=port_r,
        ))
    return out


def summarise_stress_tests(results: list[StressTestResult]) -> pd.DataFrame:
    return pd.DataFrame({
        r.scenario: {
            "mean_stressed_return_ann": r.stressed_returns.mean(),
            "worst_ticker_return_ann": r.stressed_returns.min(),
            "mean_stressed_vol_ann": r.stressed_vols.mean(),
            "equal_weight_portfolio_return_ann": r.portfolio_return_under_stress,
        } for r in results
    }).T

# ==============================================================================
# STAGE 4   |  INVESTOR PROFILING (ROBO-ADVISOR)
# ==============================================================================

QUESTIONS = [
    {"id": "horizon", "text": "What is your investment time horizon?",
     "options": {"<1 year": 1, "1-3 years": 2, "3-7 years": 3, "7+ years": 4}},
    {"id": "loss_reaction", "text": "If your portfolio fell 20% in a month, what would you do?",
     "options": {"Sell everything immediately": 1, "Sell some to reduce risk": 2,
                 "Hold and wait it out": 3, "Buy more at the lower price": 4}},
    {"id": "income_stability", "text": "How stable is your income / need for liquidity from this portfolio?",
     "options": {"I may need this money soon": 1, "Stable, but I prefer safety": 2,
                 "Stable, comfortable with risk": 3, "Very stable / surplus capital": 4}},
    {"id": "experience", "text": "How would you describe your investing experience?",
     "options": {"None": 1, "Basic": 2, "Experienced": 3, "Very experienced": 4}},
    {"id": "goal", "text": "What is your primary goal?",
     "options": {"Capital preservation": 1, "Income": 2, "Balanced growth": 3, "Maximum growth": 4}},
]


@dataclass
class InvestorProfile:
    raw_score: int
    max_score: int
    risk_category: str
    lambda_risk_aversion: float
    cvar_alpha: float
    max_single_holding: float


def score_questionnaire(answers: dict[str, str]) -> InvestorProfile:
    total, max_total = 0, 0
    for q in QUESTIONS:
        max_total += max(q["options"].values())
        chosen = answers.get(q["id"])
        if chosen not in q["options"]:
            raise ValueError(f"Missing/invalid answer for '{q['id']}': {chosen!r}")
        total += q["options"][chosen]

    pct = total / max_total

    if pct < 0.40:
        category, lam, alpha, cap = "Conservative", 8.0, 0.99, 0.10
    elif pct < 0.60:
        category, lam, alpha, cap = "Moderate", 4.0, 0.97, 0.15
    elif pct < 0.80:
        category, lam, alpha, cap = "Growth", 2.0, 0.95, 0.20
    else:
        category, lam, alpha, cap = "Aggressive", 1.0, 0.90, 0.30

    return InvestorProfile(
        raw_score=total, max_score=max_total, risk_category=category,
        lambda_risk_aversion=lam, cvar_alpha=alpha, max_single_holding=cap,
    )


def run_cli_questionnaire() -> InvestorProfile:
    answers = {}
    for q in QUESTIONS:
        print(f"\n{q['text']}")
        opts = list(q["options"].keys())
        for i, opt in enumerate(opts, 1):
            print(f"  {i}. {opt}")
        choice = int(input("Choose an option number: "))
        answers[q["id"]] = opts[choice - 1]
    return score_questionnaire(answers)


# ==============================================================================
# STAGE 5   |  CONSTRAINED OPTIMISATION (MEAN-CVAR)
# ==============================================================================

ZAKAT_RATE = 0.025
TRANSACTION_COST_BPS = 15
N_SCENARIOS = 5000
RANDOM_SEED = 11


@dataclass
class OptimizationResult:
    weights: pd.Series
    expected_return_gross: float
    expected_return_net_of_costs_and_zakat: float
    cvar: float
    turnover: float
    transaction_cost: float
    zakat_due: float


def _simulate_return_scenarios(forecasts: pd.DataFrame, correlation: pd.DataFrame | None,
                                n_scenarios: int = N_SCENARIOS, seed: int = RANDOM_SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    mu = forecasts["exp_return_ann"].values
    sigma = forecasts["exp_vol_ann"].values
    n_assets = len(mu)

    z = rng.standard_normal((n_scenarios, n_assets))
    if correlation is not None:
        corr = correlation.loc[forecasts.index, forecasts.index].values
        corr = (corr + corr.T) / 2
        eigvals, eigvecs = np.linalg.eigh(corr)
        eigvals = np.clip(eigvals, 1e-8, None)
        corr_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
        L = np.linalg.cholesky(corr_psd)
        z = z @ L.T

    scenarios = mu + z * sigma
    return scenarios


def _portfolio_cvar(weights: np.ndarray, scenario_returns: np.ndarray, alpha: float) -> float:
    port_returns = scenario_returns @ weights
    var_threshold = np.percentile(port_returns, (1 - alpha) * 100)
    tail = port_returns[port_returns <= var_threshold]
    if len(tail) == 0:
        tail = np.array([var_threshold])
    cvar_loss = -tail.mean()
    return float(cvar_loss)


def optimize_portfolio(forecasts: pd.DataFrame,
                        lambda_risk_aversion: float,
                        cvar_alpha: float,
                        max_single_holding: float,
                        current_weights: pd.Series | None = None,
                        correlation: pd.DataFrame | None = None) -> OptimizationResult:
    tickers = forecasts.index.tolist()
    n = len(tickers)
    mu = forecasts["exp_return_ann"].values

    if current_weights is None:
        current_weights = pd.Series(0.0, index=tickers)
    else:
        current_weights = current_weights.reindex(tickers).fillna(0.0)
    w0_current = current_weights.values

    scenarios = _simulate_return_scenarios(forecasts, correlation)

    def objective(w):
        exp_return = mu @ w
        cvar = _portfolio_cvar(w, scenarios, cvar_alpha)
        turnover = np.sum(np.abs(w - w0_current))
        tc = (TRANSACTION_COST_BPS / 10_000) * turnover
        utility = exp_return - lambda_risk_aversion * cvar - tc
        return -utility

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, max_single_holding) for _ in range(n)]
    w_start = np.full(n, 1.0 / n)

    result = minimize(objective, w_start, method="SLSQP", bounds=bounds,
                       constraints=constraints, options={"maxiter": 500, "ftol": 1e-9})

    if not result.success:
        w_final = np.clip(result.x, 0, max_single_holding)
        w_final = w_final / w_final.sum()
    else:
        w_final = result.x
        w_final = np.clip(w_final, 0, max_single_holding)
        w_final = w_final / w_final.sum()

    weights = pd.Series(w_final, index=tickers, name="weight")
    exp_return_gross = float(mu @ w_final)
    cvar_final = _portfolio_cvar(w_final, scenarios, cvar_alpha)
    turnover = float(np.sum(np.abs(w_final - w0_current)))
    tc_final = (TRANSACTION_COST_BPS / 10_000) * turnover
    zakat_due = ZAKAT_RATE
    exp_return_net = exp_return_gross - tc_final - zakat_due

    # FIX: CVaR sanity check. A negative CVaR means the worst (1-alpha) tail of
    # simulated scenarios is still profitable — almost always a symptom of
    # saturated/over-optimistic return forecasts, not a genuinely great portfolio.
    if cvar_final < 0:
        print(f"[optimization] WARNING: CVaR @ {cvar_alpha:.0%} = {cvar_final:+.2%} is NEGATIVE "
              "(worst tail of scenarios is profitable). This usually means the return "
              "forecasts are still too optimistic even after shrinkage — treat the "
              "headline expected return with scepticism.")

    return OptimizationResult(
        weights=weights.sort_values(ascending=False),
        expected_return_gross=exp_return_gross,
        expected_return_net_of_costs_and_zakat=exp_return_net,
        cvar=cvar_final,
        turnover=turnover,
        transaction_cost=tc_final,
        zakat_due=zakat_due,
    )


# ==============================================================================
# STAGE 6   |  PIPELINE ORCHESTRATION & PORTFOLIO OUTPUT
# ==============================================================================

warnings.filterwarnings("ignore", category=ConvergenceWarning)

DEMO_ANSWERS = {
    "horizon": "7+ years",
    "loss_reaction": "Hold and wait it out",
    "income_stability": "Stable, comfortable with risk",
    "experience": "Experienced",
    "goal": "Balanced growth",
}


def run_pipeline(interactive: bool = False) -> None:
    print("=" * 70)
    print("STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)")
    print("=" * 70)
    dataset = ingest_market_data()
    feature_panel = build_feature_panel(dataset.prices, dataset.volumes)
    print(f"Ingested {dataset.prices.shape[1]} tickers x {dataset.prices.shape[0]} trading days.")
    print(f"Feature panel: {feature_panel.shape[0]} (date,ticker) rows x {feature_panel.shape[1]} indicators.\n")

    print("=" * 70)
    print("STAGE 2 — Shariah Universe Screening")
    print("=" * 70)
    compliant_tickers, audit = screen_universe(dataset.meta, dataset.fundamentals)
    print(f"{len(compliant_tickers)} / {len(dataset.meta)} tickers pass business-activity + "
          f"financial-ratio screens:")
    print(compliant_tickers, "\n")
    if not compliant_tickers:
        print("No tickers passed the Shariah screen -- stopping here. Full audit trail:\n", audit)
        return

    print("=" * 70)
    print("=" * 70)
    print("STAGE 3 — Return Forecasting (CNN-LSTM + fundamentals)")
    print("=" * 70)
    forecasts = forecast_universe(feature_panel, compliant_tickers)

    if forecasts.empty:
        print("No forecasts produced — stopping here.")
        return

    display_cols = ["exp_return_ann", "exp_vol_ann", "forecast_source",
                    "model_trust", "raw_model_return_ann",
                    "hist_mean_return_ann", "fundamental_coverage",
                    "val_rmse_return", "naive_rmse_return"]
    print(forecasts.sort_values("exp_return_ann", ascending=False)[display_cols].round(4), "\n")

    # NEW: stress test before optimisation
    print("=" * 70)
    print("STAGE 3b — Stress Testing")
    print("=" * 70)
    stress_results = stress_test_forecasts(forecasts)
    print(summarise_stress_tests(stress_results).round(4))
    print()

    print("=" * 70)
    print("STAGE 4 — Investor Profiling (Robo-Advisor)")
    print("=" * 70)
    profile = run_cli_questionnaire() if interactive else score_questionnaire(DEMO_ANSWERS)
    print(f"Risk category: {profile.risk_category}")
    print(f"  lambda (risk aversion)   = {profile.lambda_risk_aversion}")
    print(f"  CVaR confidence (alpha)  = {profile.cvar_alpha:.0%}")
    print(f"  Max single holding cap   = {profile.max_single_holding:.0%}\n")

    print("=" * 70)
    print("STAGE 5 — Constrained Optimisation (Mean-CVaR)")
    print("=" * 70)
    # FIX: pct_change(fill_method=None) to silence the pandas FutureWarning.
    hist_returns = dataset.prices[forecasts.index].pct_change(fill_method=None).dropna()
    correlation = hist_returns.corr()
    result = optimize_portfolio(
        forecasts,
        lambda_risk_aversion=profile.lambda_risk_aversion,
        cvar_alpha=profile.cvar_alpha,
        max_single_holding=profile.max_single_holding,
        correlation=correlation,
    )

    print("=" * 70)
    print("STAGE 6 — Portfolio Output")
    print("=" * 70)
    final_weights = result.weights[result.weights > 0.005]
    report = final_weights.to_frame("weight")
    report["sector"] = dataset.meta.loc[report.index, "sector"]
    report["exp_return_ann"] = forecasts.loc[report.index, "exp_return_ann"]
    report["exp_vol_ann"] = forecasts.loc[report.index, "exp_vol_ann"]

    # FIX: hard compliance sanity check on the FINAL portfolio.
    bad_in_final = report.index[report["sector"].isin(NON_COMPLIANT_SECTORS)].tolist()
    if bad_in_final:
        print(f"[pipeline] *** COMPLIANCE FAILURE *** non-compliant tickers ended up in "
              f"the final portfolio: {bad_in_final}. This should be impossible and "
              "indicates an upstream screening bug.")

    print(report.round(4))
    print(f"\nPortfolio-level expected return (gross, annualized): {result.expected_return_gross:.2%}")
    print(f"Portfolio-level expected return (net of TC + zakat) : "
          f"{result.expected_return_net_of_costs_and_zakat:.2%}")
    print(f"Portfolio CVaR @ {profile.cvar_alpha:.0%} confidence          : {result.cvar:.2%}")
    print(f"Turnover from current holdings                       : {result.turnover:.2%}")
    print(f"Transaction cost drag                                : {result.transaction_cost:.4%}")
    print(f"Annual zakat obligation (2.5% of eligible wealth)    : {result.zakat_due:.2%}")

    report.to_csv("/tmp/final_portfolio.csv")
    print("\nSaved final portfolio to /tmp/final_portfolio.csv")


if __name__ == "__main__":
    run_pipeline(interactive=False)

STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)
[data_ingestion] Live data pulled via yfinance for 25/25 requested tickers.
Ingested 25 tickers x 1227 trading days.
Feature panel: 30675 (date,ticker) rows x 21 indicators.

STAGE 2 — Shariah Universe Screening
[shariah_screening] Tier 1 excluded 6/25 tickers on business activity: ['1155.KL', '1295.KL', '3182.KL', '4715.KL', '2836.KL', '3255.KL']
13 / 25 tickers pass business-activity + financial-ratio screens:
['6033.KL', '5183.KL', '1961.KL', '4707.KL', '3689.KL', '3026.KL', '7113.KL', '5168.KL', '5285.KL', '8869.KL', '3816.KL', '5211.KL', '3336.KL'] 

STAGE 3 — Return Forecasting (CNN-LSTM + fundamentals)
[stage3] Fundamental coverage: 13/13 tickers
[stage3] Forecast sources: 0 cnn_lstm / 3 blend / 10 historical_mean. Mean model trust: 0.03.
[stage3] NOTE: no ticker cleared the skill bar. All expected returns in this run are historical means, not model forecasts. Report them as such.
         exp_return_ann  exp_vol_ann  for

# fix stage 3, again

In [ ]:
# =============================================================================
# SHARIAH-COMPLIANT AI ROBO-ADVISOR — FULL PIPELINE (single-file, Colab-ready)
# =============================================================================

# !pip install yfinance -q

from __future__ import annotations
import sys
import math
import warnings
from typing import Protocol

import numpy as np
import pandas as pd
import scipy.stats as sps
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# ==============================================================================
# STAGE 1a  |  DATA INGESTION (5-YEAR HORIZON)
# ==============================================================================

HORIZON_YEARS = 5
TRADING_DAYS_PER_YEAR = 252

DEFAULT_UNIVERSE = [
    ("1155.KL", "Malayan Banking Bhd (Maybank)", "Conventional Banking"),
    ("1295.KL", "Public Bank Bhd", "Conventional Banking"),
    ("5347.KL", "Tenaga Nasional Bhd", "Utilities"),
    ("6033.KL", "Petronas Gas Bhd", "Energy"),
    ("5183.KL", "Petronas Chemicals Group Bhd", "Materials"),
    ("1961.KL", "IOI Corp Bhd", "Plantation"),
    ("2445.KL", "Kuala Lumpur Kepong Bhd", "Plantation"),
    ("4707.KL", "Nestle Malaysia Bhd", "Consumer Staples"),
    ("3689.KL", "Fraser & Neave Holdings Bhd", "Consumer Staples"),
    ("3026.KL", "Dutch Lady Milk Industries Bhd", "Consumer Staples"),
    ("7113.KL", "Top Glove Corp Bhd", "Health Care Equipment"),
    ("5168.KL", "Hartalega Holdings Bhd", "Health Care Equipment"),
    ("3182.KL", "Genting Bhd", "Gaming & Casinos"),
    ("4715.KL", "Genting Malaysia Bhd", "Gaming & Casinos"),
    ("6888.KL", "Axiata Group Bhd", "Telecommunications"),
    ("6947.KL", "CelcomDigi Bhd (fka Digi.Com)", "Telecommunications"),
    ("6012.KL", "Maxis Bhd", "Telecommunications"),
    ("5285.KL", "SD Guthrie Bhd (fka Sime Darby Plantation)", "Plantation"),
    ("8869.KL", "Press Metal Aluminium Holdings Bhd", "Materials"),
    ("3816.KL", "MISC Bhd", "Shipping/Logistics"),
    ("2836.KL", "Carlsberg Brewery Malaysia Bhd", "Brewery"),
    ("3255.KL", "Heineken Malaysia Bhd", "Brewery"),
    ("4677.KL", "YTL Corp Bhd", "Conglomerate/Utilities"),
    ("5211.KL", "Sunway Bhd", "Property & Construction"),
    ("3336.KL", "IJM Corp Bhd", "Property & Construction"),
]

MIN_VALID_TICKERS = 5


@dataclass
class MarketDataset:
    prices: pd.DataFrame
    volumes: pd.DataFrame
    fundamentals: pd.DataFrame
    meta: pd.DataFrame
    start_date: datetime = field(default=None)
    end_date: datetime = field(default=None)


NON_COMPLIANT_SECTORS = {
    "Conventional Banking",
    "Conventional Insurance",
    "Gaming & Casinos",
    "Brewery",
    "Tobacco",
    "Adult Entertainment",
    "Conventional Leasing",
    "Weapons & Defense",
    "Pork / Non-Halal Food",
}

NON_COMPLIANT_INDUSTRY_KEYWORDS: dict[str, str] = {
    "bank": "Conventional Banking",
    "insurance": "Conventional Insurance",
    "credit services": "Conventional Banking",
    "capital markets": "Conventional Banking",
    "gambling": "Gaming & Casinos",
    "resorts & casinos": "Gaming & Casinos",
    "casino": "Gaming & Casinos",
    "brewers": "Brewery",
    "distillers": "Brewery",
    "wineries": "Brewery",
    "beverages - wineries": "Brewery",
    "tobacco": "Tobacco",
    "aerospace & defense": "Weapons & Defense",
    "adult": "Adult Entertainment",
}


def _assert_vocabulary_consistency() -> None:
    orphans = {v for v in NON_COMPLIANT_INDUSTRY_KEYWORDS.values()
               if v not in NON_COMPLIANT_SECTORS}
    assert not orphans, (
        f"Vocabulary mismatch: {orphans} used as canonical labels but missing "
        "from NON_COMPLIANT_SECTORS. Tier 1 would silently fail to exclude them."
    )


_assert_vocabulary_consistency()


def _classify_industry(industry: str) -> str:
    ind_lower = (industry or "").lower()
    for kw in sorted(NON_COMPLIANT_INDUSTRY_KEYWORDS, key=len, reverse=True):
        if kw in ind_lower:
            return NON_COMPLIANT_INDUSTRY_KEYWORDS[kw]
    return industry or "Unknown"


_TOTAL_ASSETS_KEYS = ["Total Assets"]
_TOTAL_DEBT_KEYS = ["Total Debt"]
_LONG_TERM_DEBT_KEYS = ["Long Term Debt"]
_CURRENT_DEBT_KEYS = ["Current Debt", "Current Debt And Capital Lease Obligation"]
_CASH_KEYS = ["Cash Cash Equivalents And Short Term Investments", "Cash And Cash Equivalents"]
_RECEIVABLES_KEYS = ["Receivables", "Accounts Receivable"]


def _bs_lookup(balance_sheet: pd.DataFrame, candidates: list[str]) -> float:
    if balance_sheet is None or balance_sheet.empty:
        return np.nan
    col = balance_sheet.columns[0]
    idx_lower = {str(i).strip().lower(): i for i in balance_sheet.index}
    for cand in candidates:
        key = cand.strip().lower()
        if key in idx_lower:
            val = balance_sheet.loc[idx_lower[key], col]
            if pd.notna(val):
                return float(val)
        for lower_name, orig_name in idx_lower.items():
            if key in lower_name:
                val = balance_sheet.loc[orig_name, col]
                if pd.notna(val):
                    return float(val)
    return np.nan


def _fetch_fundamentals(ticker_obj) -> dict:
    balance_sheet = None
    try:
        balance_sheet = ticker_obj.quarterly_balance_sheet
        if balance_sheet is None or balance_sheet.empty:
            balance_sheet = ticker_obj.balance_sheet
    except Exception:
        pass

    total_assets = _bs_lookup(balance_sheet, _TOTAL_ASSETS_KEYS)
    total_debt = _bs_lookup(balance_sheet, _TOTAL_DEBT_KEYS)
    if np.isnan(total_debt):
        ltd = _bs_lookup(balance_sheet, _LONG_TERM_DEBT_KEYS)
        std = _bs_lookup(balance_sheet, _CURRENT_DEBT_KEYS)
        if not (np.isnan(ltd) and np.isnan(std)):
            total_debt = np.nansum([ltd, std])
    cash = _bs_lookup(balance_sheet, _CASH_KEYS)
    receivables = _bs_lookup(balance_sheet, _RECEIVABLES_KEYS)

    market_cap = np.nan
    try:
        market_cap = ticker_obj.fast_info.get("market_cap", np.nan)
    except Exception:
        pass
    if market_cap is None or (isinstance(market_cap, float) and np.isnan(market_cap)):
        try:
            market_cap = ticker_obj.info.get("marketCap", np.nan)
        except Exception:
            market_cap = np.nan

    return {
        "market_cap": market_cap,
        "total_assets": total_assets,
        "total_debt": total_debt,
        "cash_and_interest_securities": cash,
        "receivables": receivables,
    }


def _try_live_ingestion(tickers, start, end) -> MarketDataset | None:
    try:
        import yfinance as yf
    except ImportError:
        print("[data_ingestion] yfinance not installed.")
        return None

    price_series, volume_series, fundamentals_rows = {}, {}, []

    for t in tickers:
        tk = yf.Ticker(t)
        try:
            hist = tk.history(start=start, end=end, auto_adjust=True)
            if hist is None or hist.empty or hist["Close"].dropna().empty:
                print(f"[data_ingestion] No price history for {t}, skipping.")
                continue
            price_series[t] = hist["Close"]
            volume_series[t] = hist["Volume"]
        except Exception as e:
            print(f"[data_ingestion] Price fetch failed for {t}: {e}")
            continue

        try:
            info = tk.info
            row = _fetch_fundamentals(tk)
            row["ticker"] = t
            row["sector"] = _classify_industry(info.get("industry", info.get("sector")))
            fundamentals_rows.append(row)
        except Exception as e:
            print(f"[data_ingestion] Fundamentals fetch failed for {t}: {e} (will be NaN).")
            fundamentals_rows.append({
                "ticker": t, "market_cap": np.nan, "total_debt": np.nan,
                "total_assets": np.nan, "cash_and_interest_securities": np.nan,
                "receivables": np.nan, "sector": "Unknown",
            })

    if len(price_series) < MIN_VALID_TICKERS:
        print(f"[data_ingestion] Only {len(price_series)} tickers returned live price data "
              f"(< {MIN_VALID_TICKERS} minimum) -> treating live pull as failed.")
        return None

    prices = pd.DataFrame(price_series).sort_index()
    volumes = pd.DataFrame(volume_series).sort_index()
    fundamentals = pd.DataFrame(fundamentals_rows).set_index("ticker").reindex(prices.columns)
    meta = fundamentals[["sector"]].copy()

    n_missing_assets = fundamentals["total_assets"].isna().sum()
    n_missing_core = fundamentals[["total_debt", "cash_and_interest_securities", "receivables"]].isna().any(axis=1).sum()
    if n_missing_assets or n_missing_core:
        print(f"[data_ingestion] Note: {n_missing_assets}/{len(fundamentals)} tickers missing "
              f"'total_assets'; {n_missing_core}/{len(fundamentals)} missing debt/cash/receivables "
              "entirely (will fail the ratio screen).")

    return MarketDataset(prices, volumes, fundamentals, meta, start, end)


def _synthetic_universe(universe, start, end, seed: int = 42) -> MarketDataset:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end)
    n = len(dates)

    tickers = [u[0] for u in universe]
    names = {u[0]: u[1] for u in universe}
    sectors = {u[0]: u[2] for u in universe}

    prices, volumes, fundamentals_rows = {}, {}, []

    for i, t in enumerate(tickers):
        mu = rng.uniform(0.04, 0.12) / TRADING_DAYS_PER_YEAR
        sigma = rng.uniform(0.15, 0.45) / np.sqrt(TRADING_DAYS_PER_YEAR)
        s0 = rng.uniform(1.0, 25.0)
        shocks = rng.normal(mu - 0.5 * sigma ** 2, sigma, n)
        price_path = s0 * np.exp(np.cumsum(shocks))
        prices[t] = price_path

        base_vol = rng.uniform(2e5, 8e6)
        vol_series = np.abs(rng.normal(base_vol, base_vol * 0.3, n)).astype(int)
        volumes[t] = vol_series

        market_cap = price_path[-1] * rng.uniform(2e8, 6e9)
        is_bank_or_brewer = sectors[t] in ("Conventional Banking", "Brewery")
        debt_ratio = rng.uniform(0.35, 0.55) if is_bank_or_brewer else rng.uniform(0.05, 0.30)
        cash_ratio = rng.uniform(0.35, 0.60) if is_bank_or_brewer else rng.uniform(0.05, 0.28)
        recv_ratio = rng.uniform(0.10, 0.30)

        total_assets = market_cap * rng.uniform(0.8, 1.5)
        fundamentals_rows.append({
            "ticker": t,
            "market_cap": market_cap,
            "total_assets": total_assets,
            "total_debt": debt_ratio * total_assets,
            "cash_and_interest_securities": cash_ratio * total_assets,
            "receivables": recv_ratio * total_assets,
            "sector": sectors[t],
        })

    prices_df = pd.DataFrame(prices, index=dates)
    volumes_df = pd.DataFrame(volumes, index=dates)
    fundamentals_df = pd.DataFrame(fundamentals_rows).set_index("ticker")
    meta_df = pd.DataFrame({"name": names, "sector": sectors})

    return MarketDataset(prices_df, volumes_df, fundamentals_df, meta_df, start, end)


def ingest_market_data(universe=None, years: int = HORIZON_YEARS) -> MarketDataset:
    universe = universe or DEFAULT_UNIVERSE
    end = datetime.today()
    start = end - timedelta(days=int(years * 365.25))
    tickers = [u[0] for u in universe]

    live = _try_live_ingestion(tickers, start, end)
    if live is not None:
        print(f"[data_ingestion] Live data pulled via yfinance for "
              f"{live.prices.shape[1]}/{len(tickers)} requested tickers.")
        return live

    print("[data_ingestion] No network / yfinance unavailable -> using synthetic "
          f"{years}y dataset for {len(universe)} tickers.")
    return _synthetic_universe(universe, start, end)


# ==============================================================================
# STAGE 1b  |  TECHNICAL INDICATOR ENGINEERING
# ==============================================================================

def _rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _macd(series: pd.Series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line


def _atr(close: pd.Series, window: int = 14) -> pd.Series:
    tr = close.diff().abs()
    return tr.rolling(window).mean()


def _obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direction = np.sign(close.diff().fillna(0))
    raw_obv = (direction * volume).cumsum()
    roll_mean = raw_obv.rolling(252, min_periods=60).mean()
    roll_std = raw_obv.rolling(252, min_periods=60).std()
    return (raw_obv - roll_mean) / roll_std.replace(0, np.nan)


def compute_indicators_for_ticker(close: pd.Series, volume: pd.Series) -> pd.DataFrame:
    log_ret = np.log(close / close.shift(1))

    sma10 = close.rolling(10).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line, signal_line = _macd(close)

    roll_std21 = log_ret.rolling(21).std() * np.sqrt(252)
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_width = (bb_mid + 2 * bb_std - (bb_mid - 2 * bb_std)) / bb_mid

    feats = pd.DataFrame({
        "close": close,
        "ret_1d": log_ret,
        "ret_5d": np.log(close / close.shift(5)),
        "ret_21d": np.log(close / close.shift(21)),
        "sma10": sma10, "sma50": sma50, "sma200": sma200,
        "ema12": ema12, "ema26": ema26,
        "macd": macd_line, "macd_signal": signal_line, "macd_hist": macd_line - signal_line,
        "rsi14": _rsi(close, 14),
        "roc10": close.pct_change(10, fill_method=None) * 100,
        "vol21_ann": roll_std21,
        "bb_width": bb_width,
        "atr14": _atr(close, 14),
        "vol_roc10": volume.pct_change(10, fill_method=None) * 100,
        "obv": _obv(close, volume),
    })
    feats["px_over_sma50"] = close / sma50 - 1
    feats["sma10_over_sma50"] = sma10 / sma50 - 1

    feats = feats.replace([np.inf, -np.inf], np.nan)
    return feats


def build_feature_panel(prices: pd.DataFrame, volumes: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for ticker in prices.columns:
        f = compute_indicators_for_ticker(prices[ticker], volumes[ticker])
        f["ticker"] = ticker
        frames.append(f)
    panel = pd.concat(frames)
    panel = panel.set_index("ticker", append=True)
    panel.index.names = ["date", "ticker"]
    return panel.sort_index()


# ==============================================================================
# STAGE 2   |  SHARIAH UNIVERSE SCREENING
# ==============================================================================

RATIO_THRESHOLDS = {
    "cash_ratio": 0.33,
    "debt_ratio": 0.33,
    "receivables_ratio": 0.50,
}


def business_activity_screen(meta: pd.DataFrame) -> pd.Series:
    return ~meta["sector"].isin(NON_COMPLIANT_SECTORS)


def financial_ratio_screen(fundamentals: pd.DataFrame,
                            denominator: str = "assets") -> pd.DataFrame:
    primary = fundamentals["total_assets"] if denominator == "assets" else fundamentals["market_cap"]
    fallback = fundamentals["market_cap"] if denominator == "assets" else fundamentals["total_assets"]
    denom = primary.where(primary.notna() & (primary != 0), fallback)

    ratios = pd.DataFrame(index=fundamentals.index)
    ratios["denominator_used"] = np.where(
        primary.notna() & (primary != 0), denominator,
        np.where(fallback.notna() & (fallback != 0), f"{denominator}_fallback", "unavailable"),
    )

    required = ["cash_and_interest_securities", "total_debt", "receivables"]
    ratios["data_available"] = fundamentals[required].notna().all(axis=1) & denom.notna() & (denom != 0)

    ratios["cash_ratio"] = fundamentals["cash_and_interest_securities"] / denom
    ratios["debt_ratio"] = fundamentals["total_debt"] / denom
    ratios["receivables_ratio"] = (fundamentals["receivables"] + fundamentals["cash_and_interest_securities"]) / denom

    ratios["pass_cash"] = ratios["data_available"] & (ratios["cash_ratio"] < RATIO_THRESHOLDS["cash_ratio"])
    ratios["pass_debt"] = ratios["data_available"] & (ratios["debt_ratio"] < RATIO_THRESHOLDS["debt_ratio"])
    ratios["pass_receivables"] = ratios["data_available"] & (ratios["receivables_ratio"] < RATIO_THRESHOLDS["receivables_ratio"])
    ratios["passes_tier2"] = ratios["data_available"] & ratios[["pass_cash", "pass_debt", "pass_receivables"]].all(axis=1)
    return ratios


def screen_universe(meta: pd.DataFrame, fundamentals: pd.DataFrame,
                     denominator: str = "assets") -> tuple[list[str], pd.DataFrame]:
    tier1 = business_activity_screen(meta)
    tier2 = financial_ratio_screen(fundamentals, denominator=denominator)

    audit = tier2.copy()
    audit["sector"] = meta["sector"]
    audit["passes_tier1"] = tier1
    audit["is_shariah_compliant"] = audit["passes_tier1"] & audit["passes_tier2"]

    n_tier1_fail = int((~tier1).sum())
    if n_tier1_fail:
        excluded_names = audit.index[~tier1].tolist()
        print(f"[shariah_screening] Tier 1 excluded {n_tier1_fail}/{len(audit)} tickers "
              f"on business activity: {excluded_names}")

    compliant = audit.index[audit["is_shariah_compliant"]].tolist()
    n_missing = (~audit["data_available"]).sum()
    if n_missing:
        print(f"[shariah_screening] {n_missing}/{len(audit)} tickers excluded due to missing "
              "fundamental data -- see 'data_available' column.")
    cols = ["sector", "passes_tier1", "data_available", "denominator_used", "cash_ratio", "pass_cash",
            "debt_ratio", "pass_debt", "receivables_ratio", "pass_receivables",
            "passes_tier2", "is_shariah_compliant"]
    return compliant, audit[cols]


def filter_price_panel(feature_panel: pd.DataFrame, compliant_tickers: list[str]) -> pd.DataFrame:
    mask = feature_panel.index.get_level_values("ticker").isin(compliant_tickers)
    return feature_panel.loc[mask]


# =============================================================================
# STAGE 3  |  CROSS-SECTIONAL RETURN FORECASTING
# =============================================================================

TECHNICAL_FEATURE_COLS = [
    "ret_1d", "ret_5d", "ret_21d",
    "sma10", "sma50", "sma200", "ema12", "ema26",
    "macd", "macd_signal", "macd_hist", "rsi14", "roc10",
    "vol21_ann", "bb_width", "atr14", "vol_roc10",
    "px_over_sma50", "sma10_over_sma50",
]
FUNDAMENTAL_FEATURE_COLS = [
    "eps_revision_90d", "eps_revision_30d", "revision_breadth",
    "recommendation_drift", "surprise_history", "est_growth_fy1",
]
ALL_FEATURE_COLS = TECHNICAL_FEATURE_COLS + FUNDAMENTAL_FEATURE_COLS

FORWARD_HORIZON = 21
MAX_ABS_ANNUAL_RETURN = 0.60
MAX_ANNUAL_VOL = 0.90
MIN_ANNUAL_VOL = 0.03

MIN_FUNDAMENTAL_FEATURES = 3
TRAIN_FRACTION = 0.60
EMBARGO_DAYS = FORWARD_HORIZON
MIN_IC_TSTAT = 2.0
MIN_TRAIN_DATES = 250
RAW_FLAG_MULTIPLIER = 3.0


EMPTY_FORECAST_COLUMNS = [
    "exp_return_ann", "exp_vol_ann",
    "cs_score", "hist_mean_return_ann",
    "forecast_source", "model_used",
    "ic_tstat", "raw_flag", "fundamental_coverage",
    "n_train_dates", "n_test_dates",
]


@dataclass
class CrossSectionalDiagnostics:
    ic_series: pd.Series
    ic_mean: float
    ic_std: float
    ic_tstat: float
    ic_tstat_naive: float
    n_ic_obs: int
    n_ic_eff: float
    n_train_dates: int
    n_test_dates: int
    feature_importances: pd.Series


class FundamentalFeatureProvider(Protocol):
    def fetch(self, tickers: list[str], start, end) -> pd.DataFrame: ...


class YFinanceFundamentalProvider:
    def __init__(self, verbose: bool = True):
        self.verbose = verbose

    def _safe(self, fn, default=None):
        try:
            v = fn()
            return v if v is not None else default
        except Exception:
            return default

    def _pct(self, a, b):
        try:
            a, b = float(a), float(b)
            if b == 0 or math.isnan(a) or math.isnan(b):
                return np.nan
            return (a - b) / abs(b)
        except Exception:
            return np.nan

    def _one(self, ticker: str):
        try:
            import yfinance as yf
        except ImportError:
            return None
        tk = yf.Ticker(ticker)
        eps_est = self._safe(lambda: tk.earnings_estimate)
        eps_rev = self._safe(lambda: tk.eps_revisions)
        recs = self._safe(lambda: tk.recommendations)
        hist = self._safe(lambda: tk.earnings_history)

        out = {c: np.nan for c in FUNDAMENTAL_FEATURE_COLS}

        if eps_est is not None and not eps_est.empty:
            try:
                row = eps_est.loc["0y"] if "0y" in eps_est.index else eps_est.iloc[0]
                out["eps_revision_90d"] = self._pct(row.get("avg"), row.get("90daysAgo"))
                out["eps_revision_30d"] = self._pct(row.get("avg"), row.get("30daysAgo"))
                out["est_growth_fy1"] = self._pct(row.get("avg"), row.get("yearAgoEps"))
            except Exception:
                pass

        if eps_rev is not None and not eps_rev.empty:
            try:
                row = eps_rev.loc["0y"] if "0y" in eps_rev.index else eps_rev.iloc[0]
                up = float(row.get("upLast90days", 0) or 0)
                down = float(row.get("downLast90days", 0) or 0)
                if up + down > 0:
                    out["revision_breadth"] = up / (up + down)
            except Exception:
                pass

        if recs is not None and not recs.empty:
            try:
                weights = {"strongBuy": 1.0, "buy": 2.0, "hold": 3.0,
                           "sell": 4.0, "strongSell": 5.0}
                def _mean_score(col):
                    num = den = 0.0
                    for k, w in weights.items():
                        if k in recs.index:
                            n = float(recs.loc[k, col] or 0)
                            num += w * n
                            den += n
                    return num / den if den > 0 else np.nan
                now = _mean_score("current")
                ago = _mean_score("3m Ago") if "3m Ago" in recs.columns else np.nan
                if not (math.isnan(now) or math.isnan(ago)):
                    out["recommendation_drift"] = now - ago
            except Exception:
                pass

        if hist is not None and not hist.empty:
            try:
                if "surprisePercent" in hist.columns:
                    vals = hist["surprisePercent"].dropna().tail(4)
                    if len(vals):
                        out["surprise_history"] = float(vals.mean()) / 100.0
            except Exception:
                pass

        if all(np.isnan(v) for v in out.values()):
            return None
        return out

    def fetch(self, tickers, start, end) -> pd.DataFrame:
        rows = []
        for t in tickers:
            feats = self._one(t)
            if feats is None:
                if self.verbose:
                    print(f"[fundamentals] No analyst coverage for {t}")
                continue
            feats["ticker"] = t
            rows.append(feats)
        if not rows:
            return pd.DataFrame(columns=FUNDAMENTAL_FEATURE_COLS + ["ticker"])
        return pd.DataFrame(rows)


def _merge_fundamentals(technical_panel, tickers, provider):
    provider = provider or YFinanceFundamentalProvider()
    fund = provider.fetch(tickers, None, None)
    panel = technical_panel.copy()

    if fund.empty:
        print("[stage3] No fundamental coverage at all.")
        for c in FUNDAMENTAL_FEATURE_COLS:
            panel[c] = np.nan
        return panel, pd.Series(False, index=tickers)

    fund = fund.set_index("ticker").reindex(tickers)

    fill = (fund[FUNDAMENTAL_FEATURE_COLS].notna().mean() * 100).round(1)
    print("[stage3] Fundamental fill rate (% of universe):")
    for c, v in fill.items():
        print(f"    {c:<24s} {v:5.1f}%")

    n_features_ok = fund[FUNDAMENTAL_FEATURE_COLS].notna().sum(axis=1)
    coverage = n_features_ok >= MIN_FUNDAMENTAL_FEATURES
    print(f"[stage3] Tickers with >= {MIN_FUNDAMENTAL_FEATURES}/"
          f"{len(FUNDAMENTAL_FEATURE_COLS)} features: "
          f"{int(coverage.sum())}/{len(tickers)}")

    for c in FUNDAMENTAL_FEATURE_COLS:
        if c not in fund.columns:
            panel[c] = np.nan
            continue
        snap = fund[c].to_dict()
        idx = panel.index
        panel[c] = pd.Series([snap.get(t, np.nan) for _, t in idx], index=idx)

    return panel, coverage


def _add_forward_return(panel: pd.DataFrame) -> pd.DataFrame:
    panel = panel.sort_index().copy()
    panel["fwd_log_return"] = (
        panel.groupby(level="ticker", sort=False)["close"]
             .transform(lambda s: np.log(s.shift(-FORWARD_HORIZON) / s))
    )
    return panel


def _vanderwaerden_z(s: pd.Series) -> pd.Series:
    valid = s.dropna()
    n = len(valid)
    if n < 3:
        return pd.Series(np.nan, index=s.index)
    ranks = valid.rank(method="average")
    u = (ranks - 0.5) / n
    z = pd.Series(sps.norm.ppf(u), index=valid.index)
    return z.reindex(s.index)


def _cross_sectional_transform(panel: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    panel = panel.copy()
    for c in cols:
        if c not in panel.columns:
            continue
        panel[c] = panel.groupby(level="date")[c].transform(_vanderwaerden_z)
    return panel


def _make_model(seed: int = 7) -> HistGradientBoostingRegressor:
    return HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=4,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=seed,
    )


def _walk_forward_diagnostics(panel: pd.DataFrame,
                               feature_cols: list[str],
                               target_col: str = "fwd_log_return_cs",
                               seed: int = 7) -> CrossSectionalDiagnostics:
    dates = panel.index.get_level_values("date").unique().sort_values()
    split_idx = int(len(dates) * TRAIN_FRACTION)

    test_start = dates[split_idx]
    train_cutoff = test_start - timedelta(days=int(EMBARGO_DAYS * 365.25 / 252))

    train_mask = panel.index.get_level_values("date") < train_cutoff
    test_mask = panel.index.get_level_values("date") >= test_start

    train = panel.loc[train_mask]
    test = panel.loc[test_mask]

    train = train.dropna(subset=feature_cols + [target_col])
    test = test.dropna(subset=feature_cols + [target_col])

    if train.index.get_level_values("date").nunique() < MIN_TRAIN_DATES:
        raise RuntimeError(
            f"Only {train.index.get_level_values('date').nunique()} training dates — "
            f"need >= {MIN_TRAIN_DATES}."
        )

    model = _make_model(seed=seed)
    model.fit(train[feature_cols].values, train[target_col].values)

    preds = model.predict(test[feature_cols].values)
    test = test.assign(pred=preds)

    ics = []
    for d, grp in test.groupby(level="date"):
        if len(grp) < 4:
            continue
        rho, _ = sps.spearmanr(grp["pred"], grp[target_col])
        if not np.isnan(rho):
            ics.append((d, rho))
    ic_series = pd.Series(
        [r for _, r in ics],
        index=pd.DatetimeIndex([d for d, _ in ics], name="date"),
        name="ic",
    )

    ic_mean = float(ic_series.mean()) if len(ic_series) else 0.0
    ic_std = float(ic_series.std(ddof=1)) if len(ic_series) > 1 else 0.0
    n_naive = len(ic_series)
    n_eff = max(1.0, n_naive / FORWARD_HORIZON)
    ic_t_naive = ic_mean / (ic_std / np.sqrt(n_naive)) if ic_std > 0 and n_naive > 0 else 0.0
    ic_t = ic_mean / (ic_std / np.sqrt(n_eff)) if ic_std > 0 else 0.0

    imp = (train[feature_cols]
           .corrwith(train[target_col], method="spearman")
           .abs().sort_values(ascending=False))
    imp.name = "abs_spearman_with_target"

    return CrossSectionalDiagnostics(
        ic_series=ic_series,
        ic_mean=ic_mean,
        ic_std=ic_std,
        ic_tstat=ic_t,
        ic_tstat_naive=ic_t_naive,
        n_ic_obs=n_naive,
        n_ic_eff=n_eff,
        n_train_dates=int(train.index.get_level_values("date").nunique()),
        n_test_dates=int(test.index.get_level_values("date").nunique()),
        feature_importances=imp,
    )


def forecast_universe(feature_panel: pd.DataFrame,
                       compliant_tickers: list[str],
                       fundamental_provider: FundamentalFeatureProvider | None = None,
                       min_ic_tstat: float = MIN_IC_TSTAT,
                       seed: int = 7) -> pd.DataFrame:

    if not compliant_tickers:
        print("[stage3] No compliant tickers — nothing to forecast.")
        return pd.DataFrame(columns=EMPTY_FORECAST_COLUMNS).rename_axis("ticker")

    # FIX (Bug 1): enforce Shariah screen at the model boundary
    feature_panel = feature_panel.loc[
        feature_panel.index.get_level_values("ticker").isin(compliant_tickers)
    ].copy()

    print("[stage3] Merging forward-looking fundamental features...")
    panel, coverage = _merge_fundamentals(feature_panel, compliant_tickers,
                                           fundamental_provider)

    # FIX (Bug 2): capture raw per-ticker vol BEFORE cross-sectional z-scoring
    raw_vol_by_ticker = (
        panel.groupby(level="ticker")["vol21_ann"].mean().to_dict()
    )

    panel = _add_forward_return(panel)

    feat_cols_present = [c for c in ALL_FEATURE_COLS if c in panel.columns]
    panel = _cross_sectional_transform(panel, feat_cols_present)
    panel["fwd_log_return_cs"] = (
        panel.groupby(level="date")["fwd_log_return"].transform(_vanderwaerden_z)
    )

    for c in FUNDAMENTAL_FEATURE_COLS:
        if c in panel.columns:
            panel[c] = panel[c].fillna(0.0)

    try:
        diag = _walk_forward_diagnostics(panel, feat_cols_present, seed=seed)
    except RuntimeError as e:
        print(f"[stage3] Walk-forward diagnostics unavailable: {e}")
        diag = None

    if diag is not None:
        print(f"[stage3] OOS Information Coefficient (Spearman, per date):")
        print(f"    mean IC             = {diag.ic_mean:+.4f}")
        print(f"    std IC              = {diag.ic_std:.4f}")
        print(f"    n IC obs (daily)    = {diag.n_ic_obs}")
        print(f"    n_eff (overlap-adj) = {diag.n_ic_eff:.1f}")
        print(f"    t-stat (naive)      = {diag.ic_tstat_naive:+.2f}")
        print(f"    t-stat (corrected)  = {diag.ic_tstat:+.2f}   "
              f"<- use this one; threshold {min_ic_tstat:.1f}")
        print(f"    train dates         = {diag.n_train_dates}, "
              f"test dates          = {diag.n_test_dates}")
        print(f"[stage3] Top univariate feature correlations with target:")
        for k, v in diag.feature_importances.head(6).items():
            print(f"    {k:<24s} {v:+.3f}")

        trustworthy = diag.ic_tstat >= min_ic_tstat
    else:
        trustworthy = False

    dates_all = panel.index.get_level_values("date").unique().sort_values()
    latest_date = dates_all[-1]
    label_cutoff = latest_date - timedelta(days=int(FORWARD_HORIZON * 365.25 / 252))

    train_full = panel.loc[
        (panel.index.get_level_values("date") < label_cutoff)
    ].dropna(subset=feat_cols_present + ["fwd_log_return_cs"])

    if train_full.empty:
        print("[stage3] No usable training data for the final model.")
        return pd.DataFrame(columns=EMPTY_FORECAST_COLUMNS).rename_axis("ticker")

    final_model = _make_model(seed=seed)
    final_model.fit(train_full[feat_cols_present].values,
                     train_full["fwd_log_return_cs"].values)

    latest = panel.xs(latest_date, level="date", drop_level=False).copy()
    latest = latest.droplevel("date")

    cs_mean_21d = float(train_full["fwd_log_return"].mean())
    cs_std_21d = float(train_full["fwd_log_return"].std())
    periods_per_year = 252 / FORWARD_HORIZON
    cs_mean_ann = cs_mean_21d * periods_per_year

    if trustworthy:
        cs_scores = final_model.predict(latest[feat_cols_present].values)
        cs_scores = pd.Series(cs_scores, index=latest.index)
        ic_damp = diag.ic_mean if diag is not None else 0.0
        exp_ret_ann = cs_mean_ann + ic_damp * cs_std_21d * cs_scores.values * periods_per_year
        source = "cross_sectional_gbm"
        model_used = "hgb_cross_sectional"
    else:
        cs_scores = pd.Series(0.0, index=latest.index)
        exp_ret_ann = np.full(len(latest), cs_mean_ann)
        source = "fallback_equal_mean"
        model_used = "none"
        print("[stage3] IC t-stat below threshold -> falling back to "
              "cross-sectional mean for every ticker.")

    exp_ret_ann = np.clip(exp_ret_ann, -MAX_ABS_ANNUAL_RETURN, MAX_ABS_ANNUAL_RETURN)

    rows = []
    for i, ticker in enumerate(latest.index):
        cs = float(cs_scores.iloc[i])
        hist_vol = float(raw_vol_by_ticker.get(ticker, np.nan))
        raw_flag = bool(abs(cs) > RAW_FLAG_MULTIPLIER)
        exp_vol = float(np.clip(hist_vol if not np.isnan(hist_vol) else 0.20,
                                 MIN_ANNUAL_VOL, MAX_ANNUAL_VOL))
        rows.append({
            "ticker": ticker,
            "exp_return_ann": float(exp_ret_ann[i]),
            "exp_vol_ann": exp_vol,
            "cs_score": cs,
            "hist_mean_return_ann": cs_mean_ann,
            "forecast_source": source,
            "model_used": model_used,
            "ic_tstat": diag.ic_tstat if diag else np.nan,
            "raw_flag": raw_flag,
            "fundamental_coverage": bool(coverage.get(ticker, False)),
            "n_train_dates": diag.n_train_dates if diag else 0,
            "n_test_dates": diag.n_test_dates if diag else 0,
        })

    out = pd.DataFrame(rows).set_index("ticker")

    if out["raw_flag"].any():
        flagged = out.index[out["raw_flag"]].tolist()
        print(f"[stage3] Flagged extreme cross-sectional scores (|z| > "
              f"{RAW_FLAG_MULTIPLIER}): {flagged}")

    # FIX: assertion before return (was unreachable after `return out`)
    assert set(out.index).issubset(set(compliant_tickers)), (
        f"forecast_universe leaked non-compliant tickers: "
        f"{set(out.index) - set(compliant_tickers)}"
    )

    return out


# -----------------------------------------------------------------------------
# Stage 3b  |  STRESS TESTING
# -----------------------------------------------------------------------------

HISTORICAL_STRESS_SCENARIOS = {
    "2008_gfc":        {"return_mult": 0.05, "vol_add": 0.35},
    "2020_covid":      {"return_mult": 0.30, "vol_add": 0.25},
    "2000_dotcom":     {"return_mult": 0.10, "vol_add": 0.30},
    "2022_rate_shock": {"return_mult": 0.45, "vol_add": 0.20},
}


@dataclass
class StressTestResult:
    scenario: str
    stressed_returns: pd.Series
    stressed_vols: pd.Series
    portfolio_return_under_stress: float


def stress_test_forecasts(forecasts: pd.DataFrame,
                           scenarios: dict | None = None) -> list[StressTestResult]:
    scenarios = scenarios or HISTORICAL_STRESS_SCENARIOS
    out = []
    for name, s in scenarios.items():
        r = forecasts["exp_return_ann"] * s["return_mult"]
        v = (forecasts["exp_vol_ann"] + s["vol_add"]).clip(upper=MAX_ANNUAL_VOL)
        port_r = float(r.mean())
        out.append(StressTestResult(
            scenario=name,
            stressed_returns=r,
            stressed_vols=v,
            portfolio_return_under_stress=port_r,
        ))
    return out


def summarise_stress_tests(results: list[StressTestResult]) -> pd.DataFrame:
    return pd.DataFrame({
        r.scenario: {
            "mean_stressed_return_ann": r.stressed_returns.mean(),
            "worst_ticker_return_ann": r.stressed_returns.min(),
            "mean_stressed_vol_ann": r.stressed_vols.mean(),
            "equal_weight_portfolio_return_ann": r.portfolio_return_under_stress,
        } for r in results
    }).T


# ==============================================================================
# STAGE 4   |  INVESTOR PROFILING
# ==============================================================================

QUESTIONS = [
    {"id": "horizon", "text": "What is your investment time horizon?",
     "options": {"<1 year": 1, "1-3 years": 2, "3-7 years": 3, "7+ years": 4}},
    {"id": "loss_reaction", "text": "If your portfolio fell 20% in a month, what would you do?",
     "options": {"Sell everything immediately": 1, "Sell some to reduce risk": 2,
                 "Hold and wait it out": 3, "Buy more at the lower price": 4}},
    {"id": "income_stability", "text": "How stable is your income / need for liquidity?",
     "options": {"I may need this money soon": 1, "Stable, but I prefer safety": 2,
                 "Stable, comfortable with risk": 3, "Very stable / surplus capital": 4}},
    {"id": "experience", "text": "How would you describe your investing experience?",
     "options": {"None": 1, "Basic": 2, "Experienced": 3, "Very experienced": 4}},
    {"id": "goal", "text": "What is your primary goal?",
     "options": {"Capital preservation": 1, "Income": 2, "Balanced growth": 3, "Maximum growth": 4}},
]


@dataclass
class InvestorProfile:
    raw_score: int
    max_score: int
    risk_category: str
    lambda_risk_aversion: float
    cvar_alpha: float
    max_single_holding: float


def score_questionnaire(answers: dict[str, str]) -> InvestorProfile:
    total, max_total = 0, 0
    for q in QUESTIONS:
        max_total += max(q["options"].values())
        chosen = answers.get(q["id"])
        if chosen not in q["options"]:
            raise ValueError(f"Missing/invalid answer for '{q['id']}': {chosen!r}")
        total += q["options"][chosen]

    pct = total / max_total

    if pct < 0.40:
        category, lam, alpha, cap = "Conservative", 8.0, 0.99, 0.10
    elif pct < 0.60:
        category, lam, alpha, cap = "Moderate", 4.0, 0.97, 0.15
    elif pct < 0.80:
        category, lam, alpha, cap = "Growth", 2.0, 0.95, 0.20
    else:
        category, lam, alpha, cap = "Aggressive", 1.0, 0.90, 0.30

    return InvestorProfile(
        raw_score=total, max_score=max_total, risk_category=category,
        lambda_risk_aversion=lam, cvar_alpha=alpha, max_single_holding=cap,
    )


def run_cli_questionnaire() -> InvestorProfile:
    answers = {}
    for q in QUESTIONS:
        print(f"\n{q['text']}")
        opts = list(q["options"].keys())
        for i, opt in enumerate(opts, 1):
            print(f"  {i}. {opt}")
        choice = int(input("Choose an option number: "))
        answers[q["id"]] = opts[choice - 1]
    return score_questionnaire(answers)


# ==============================================================================
# STAGE 5   |  CONSTRAINED OPTIMISATION (MEAN-CVAR)
# ==============================================================================

ZAKAT_RATE = 0.025
TRANSACTION_COST_BPS = 15
N_SCENARIOS = 5000
RANDOM_SEED = 11


@dataclass
class OptimizationResult:
    weights: pd.Series
    expected_return_gross: float
    expected_return_net_of_costs_and_zakat: float
    cvar: float
    turnover: float
    transaction_cost: float
    zakat_due: float


def _simulate_return_scenarios(forecasts: pd.DataFrame, correlation: pd.DataFrame | None,
                                n_scenarios: int = N_SCENARIOS, seed: int = RANDOM_SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    mu = forecasts["exp_return_ann"].values
    sigma = forecasts["exp_vol_ann"].values
    n_assets = len(mu)

    z = rng.standard_normal((n_scenarios, n_assets))
    if correlation is not None:
        corr = correlation.loc[forecasts.index, forecasts.index].values
        corr = (corr + corr.T) / 2
        eigvals, eigvecs = np.linalg.eigh(corr)
        eigvals = np.clip(eigvals, 1e-8, None)
        corr_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
        L = np.linalg.cholesky(corr_psd)
        z = z @ L.T

    scenarios = mu + z * sigma
    return scenarios


def _portfolio_cvar(weights: np.ndarray, scenario_returns: np.ndarray, alpha: float) -> float:
    port_returns = scenario_returns @ weights
    var_threshold = np.percentile(port_returns, (1 - alpha) * 100)
    tail = port_returns[port_returns <= var_threshold]
    if len(tail) == 0:
        tail = np.array([var_threshold])
    cvar_loss = -tail.mean()
    return float(cvar_loss)


def optimize_portfolio(forecasts: pd.DataFrame,
                        lambda_risk_aversion: float,
                        cvar_alpha: float,
                        max_single_holding: float,
                        current_weights: pd.Series | None = None,
                        correlation: pd.DataFrame | None = None) -> OptimizationResult:
    tickers = forecasts.index.tolist()
    n = len(tickers)
    mu = forecasts["exp_return_ann"].values

    if current_weights is None:
        current_weights = pd.Series(0.0, index=tickers)
    else:
        current_weights = current_weights.reindex(tickers).fillna(0.0)
    w0_current = current_weights.values

    scenarios = _simulate_return_scenarios(forecasts, correlation)

    def objective(w):
        exp_return = mu @ w
        cvar = _portfolio_cvar(w, scenarios, cvar_alpha)
        turnover = np.sum(np.abs(w - w0_current))
        tc = (TRANSACTION_COST_BPS / 10_000) * turnover
        utility = exp_return - lambda_risk_aversion * cvar - tc
        return -utility

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, max_single_holding) for _ in range(n)]
    w_start = np.full(n, 1.0 / n)

    result = minimize(objective, w_start, method="SLSQP", bounds=bounds,
                       constraints=constraints, options={"maxiter": 500, "ftol": 1e-9})

    if not result.success:
        w_final = np.clip(result.x, 0, max_single_holding)
        w_final = w_final / w_final.sum()
    else:
        w_final = result.x
        w_final = np.clip(w_final, 0, max_single_holding)
        w_final = w_final / w_final.sum()

    weights = pd.Series(w_final, index=tickers, name="weight")
    exp_return_gross = float(mu @ w_final)
    cvar_final = _portfolio_cvar(w_final, scenarios, cvar_alpha)
    turnover = float(np.sum(np.abs(w_final - w0_current)))
    tc_final = (TRANSACTION_COST_BPS / 10_000) * turnover
    zakat_due = ZAKAT_RATE
    exp_return_net = exp_return_gross - tc_final - zakat_due

    if cvar_final < 0:
        print(f"[optimization] WARNING: CVaR @ {cvar_alpha:.0%} = {cvar_final:+.2%} is NEGATIVE "
              "(worst tail of scenarios is profitable). Return forecasts are likely "
              "over-optimistic even after shrinkage.")

    return OptimizationResult(
        weights=weights.sort_values(ascending=False),
        expected_return_gross=exp_return_gross,
        expected_return_net_of_costs_and_zakat=exp_return_net,
        cvar=cvar_final,
        turnover=turnover,
        transaction_cost=tc_final,
        zakat_due=zakat_due,
    )


# ==============================================================================
# STAGE 6   |  PIPELINE ORCHESTRATION
# ==============================================================================

DEMO_ANSWERS = {
    "horizon": "7+ years",
    "loss_reaction": "Hold and wait it out",
    "income_stability": "Stable, comfortable with risk",
    "experience": "Experienced",
    "goal": "Balanced growth",
}


def run_pipeline(interactive: bool = False) -> None:
    print("=" * 70)
    print("STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)")
    print("=" * 70)
    dataset = ingest_market_data()
    feature_panel = build_feature_panel(dataset.prices, dataset.volumes)
    print(f"Ingested {dataset.prices.shape[1]} tickers x {dataset.prices.shape[0]} trading days.")
    print(f"Feature panel: {feature_panel.shape[0]} (date,ticker) rows x {feature_panel.shape[1]} indicators.\n")

    print("=" * 70)
    print("STAGE 2 — Shariah Universe Screening")
    print("=" * 70)
    compliant_tickers, audit = screen_universe(dataset.meta, dataset.fundamentals)
    print(f"{len(compliant_tickers)} / {len(dataset.meta)} tickers pass business-activity + "
          f"financial-ratio screens:")
    print(compliant_tickers, "\n")
    if not compliant_tickers:
        print("No tickers passed the Shariah screen -- stopping here.")
        return

    print("=" * 70)
    print("STAGE 3 — Cross-Sectional Return Forecasting")
    print("=" * 70)
    forecasts = forecast_universe(feature_panel, compliant_tickers)
    if forecasts.empty:
        print("No forecasts produced — stopping here.")
        return

    display_cols = ["exp_return_ann", "exp_vol_ann", "cs_score",
                    "forecast_source", "ic_tstat", "raw_flag", "fundamental_coverage"]
    print(forecasts.sort_values("exp_return_ann", ascending=False)[display_cols].round(4), "\n")

    print("=" * 70)
    print("STAGE 3b — Stress Testing")
    print("=" * 70)
    stress_results = stress_test_forecasts(forecasts)
    print("Equal-weighted universe under stress:")
    print(summarise_stress_tests(stress_results).round(4))

    print("=" * 70)
    print("STAGE 4 — Investor Profiling (Robo-Advisor)")
    print("=" * 70)
    profile = run_cli_questionnaire() if interactive else score_questionnaire(DEMO_ANSWERS)
    print(f"Risk category: {profile.risk_category}")
    print(f"  lambda (risk aversion)   = {profile.lambda_risk_aversion}")
    print(f"  CVaR confidence (alpha)  = {profile.cvar_alpha:.0%}")
    print(f"  Max single holding cap   = {profile.max_single_holding:.0%}\n")

    print("=" * 70)
    print("STAGE 5 — Constrained Optimisation (Mean-CVaR)")
    print("=" * 70)
    hist_returns = dataset.prices[forecasts.index].pct_change(fill_method=None).dropna()
    correlation = hist_returns.corr()
    result = optimize_portfolio(
        forecasts,
        lambda_risk_aversion=profile.lambda_risk_aversion,
        cvar_alpha=profile.cvar_alpha,
        max_single_holding=profile.max_single_holding,
        correlation=correlation,
    )

    print("=" * 70)
    print("STAGE 6 — Portfolio Output")
    print("=" * 70)
    final_weights = result.weights[result.weights > 0.005]
    report = final_weights.to_frame("weight")
    report["sector"] = dataset.meta.loc[report.index, "sector"]
    report["exp_return_ann"] = forecasts.loc[report.index, "exp_return_ann"]
    report["exp_vol_ann"] = forecasts.loc[report.index, "exp_vol_ann"]

    bad_in_final = report.index[report["sector"].isin(NON_COMPLIANT_SECTORS)].tolist()
    if bad_in_final:
        print(f"[pipeline] *** COMPLIANCE FAILURE *** non-compliant tickers ended up in "
              f"the final portfolio: {bad_in_final}.")

    print(report.round(4))

    sources = forecasts.loc[final_weights.index, "forecast_source"].value_counts().to_dict()
    n_model = sources.get("cross_sectional_gbm", 0)
    n_fallback = sources.get("fallback_equal_mean", 0)
    src_note = (f"{n_model} from cross-sectional model, {n_fallback} from fallback mean"
                if n_model + n_fallback > 0 else "mixed sources")

    print(f"\nPortfolio-level expected return (gross, annualized): {result.expected_return_gross:.2%}")
    print(f"Portfolio-level expected return (net of TC + zakat) : "
          f"{result.expected_return_net_of_costs_and_zakat:.2%}")
    print(f"    [source: {src_note}; "
          f"IC t-stat = {forecasts['ic_tstat'].iloc[0]:+.2f}]")
    print(f"Portfolio CVaR @ {profile.cvar_alpha:.0%} confidence          : {result.cvar:.2%}")
    print(f"Turnover from current holdings                       : {result.turnover:.2%}")
    print(f"Transaction cost drag                                : {result.transaction_cost:.4%}")
    print(f"Annual zakat obligation (2.5% of eligible wealth)    : {result.zakat_due:.2%}")

    print("\nStressed portfolio returns (final weights, per scenario):")
    w = result.weights
    for r in stress_results:
        port_stressed = float((r.stressed_returns.reindex(w.index).fillna(0) * w).sum())
        print(f"    {r.scenario:<20s} {port_stressed:+.2%}")

    report.to_csv("/tmp/final_portfolio.csv")
    print("\nSaved final portfolio to /tmp/final_portfolio.csv")


if __name__ == "__main__":
    run_pipeline(interactive=False)

STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)
[data_ingestion] Live data pulled via yfinance for 25/25 requested tickers.
Ingested 25 tickers x 1227 trading days.
Feature panel: 30675 (date,ticker) rows x 21 indicators.

STAGE 2 — Shariah Universe Screening
[shariah_screening] Tier 1 excluded 6/25 tickers on business activity: ['1155.KL', '1295.KL', '3182.KL', '4715.KL', '2836.KL', '3255.KL']
13 / 25 tickers pass business-activity + financial-ratio screens:
['6033.KL', '5183.KL', '1961.KL', '4707.KL', '3689.KL', '3026.KL', '7113.KL', '5168.KL', '5285.KL', '8869.KL', '3816.KL', '5211.KL', '3336.KL'] 

STAGE 3 — Cross-Sectional Return Forecasting
[stage3] Merging forward-looking fundamental features...
[stage3] Fundamental fill rate (% of universe):
    eps_revision_90d           0.0%
    eps_revision_30d           0.0%
    revision_breadth           0.0%
    recommendation_drift       0.0%
    surprise_history          23.1%
    est_growth_fy1           100.0%
[stage3] Ticke

/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


[stage3] OOS Information Coefficient (Spearman, per date):
    mean IC             = +0.0087
    std IC              = 0.3248
    n IC obs (daily)    = 466
    n_eff (overlap-adj) = 22.2
    t-stat (naive)      = +0.58
    t-stat (corrected)  = +0.13   <- use this one; threshold 2.0
    train dates         = 518, test dates          = 466
[stage3] Top univariate feature correlations with target:
    atr14                    +0.111
    est_growth_fy1           +0.087
    sma10                    +0.082
    sma200                   +0.082
    ema12                    +0.081
    ema26                    +0.080
[stage3] IC t-stat below threshold -> falling back to cross-sectional mean for every ticker.
         exp_return_ann  exp_vol_ann  cs_score      forecast_source  ic_tstat  \
ticker                                                                          
1961.KL          0.0384       0.2028       0.0  fallback_equal_mean    0.1261   
3026.KL          0.0384       0.1510       0.0  f

# cuayo

In [ ]:
# =============================================================================
# SHARIAH-COMPLIANT ROBO-ADVISOR — OPTION A PIPELINE
# =============================================================================
# DESIGN DECISION (Option A)
# --------------------------
# Three model families were tested for Stage 3:
#   1. Per-asset MLPRegressor on 19 technicals
#   2. Per-asset CNN-LSTM on 19 technicals + 6 fundamental features
#   3. Cross-sectional HistGradientBoosting on cross-sectionally z-scored
#      technicals + fundamentals
# None beat the naive "always predict the training mean" baseline out of
# sample. The best cross-sectional attempt reported a mean IC of +0.009 with
# an overlap-corrected t-stat of +0.13 — i.e. indistinguishable from zero.
#
# Rather than continue tuning model architecture, this pipeline accepts the
# finding: on this universe, at this horizon, with these features, there is
# no demonstrable edge. Stage 3 therefore reports trailing historical
# statistics, NOT forward-looking forecasts. Every downstream stage and
# every output line is labelled accordingly.
#
# Stress testing was reformulated: the previous version multiplied the
# central return estimate by a fraction, which for a low-expected-return
# portfolio produced stress returns that were still positive. That is not
# a stress test. The current version applies a σ-multiple drawdown derived
# from historical peak-to-trough equity moves, calibrated to the
# portfolio's own volatility.
# =============================================================================

# !pip install yfinance -q

from __future__ import annotations
import warnings
from dataclasses import dataclass, field
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# ==============================================================================
# STAGE 1a  |  DATA INGESTION
# ==============================================================================

HORIZON_YEARS = 5
TRADING_DAYS_PER_YEAR = 252

DEFAULT_UNIVERSE = [
    ("1155.KL", "Malayan Banking Bhd (Maybank)", "Conventional Banking"),
    ("1295.KL", "Public Bank Bhd", "Conventional Banking"),
    ("5347.KL", "Tenaga Nasional Bhd", "Utilities"),
    ("6033.KL", "Petronas Gas Bhd", "Energy"),
    ("5183.KL", "Petronas Chemicals Group Bhd", "Materials"),
    ("1961.KL", "IOI Corp Bhd", "Plantation"),
    ("2445.KL", "Kuala Lumpur Kepong Bhd", "Plantation"),
    ("4707.KL", "Nestle Malaysia Bhd", "Consumer Staples"),
    ("3689.KL", "Fraser & Neave Holdings Bhd", "Consumer Staples"),
    ("3026.KL", "Dutch Lady Milk Industries Bhd", "Consumer Staples"),
    ("7113.KL", "Top Glove Corp Bhd", "Health Care Equipment"),
    ("5168.KL", "Hartalega Holdings Bhd", "Health Care Equipment"),
    ("3182.KL", "Genting Bhd", "Gaming & Casinos"),
    ("4715.KL", "Genting Malaysia Bhd", "Gaming & Casinos"),
    ("6888.KL", "Axiata Group Bhd", "Telecommunications"),
    ("6947.KL", "CelcomDigi Bhd (fka Digi.Com)", "Telecommunications"),
    ("6012.KL", "Maxis Bhd", "Telecommunications"),
    ("5285.KL", "SD Guthrie Bhd (fka Sime Darby Plantation)", "Plantation"),
    ("8869.KL", "Press Metal Aluminium Holdings Bhd", "Materials"),
    ("3816.KL", "MISC Bhd", "Shipping/Logistics"),
    ("2836.KL", "Carlsberg Brewery Malaysia Bhd", "Brewery"),
    ("3255.KL", "Heineken Malaysia Bhd", "Brewery"),
    ("4677.KL", "YTL Corp Bhd", "Conglomerate/Utilities"),
    ("5211.KL", "Sunway Bhd", "Property & Construction"),
    ("3336.KL", "IJM Corp Bhd", "Property & Construction"),
]

MIN_VALID_TICKERS = 5


@dataclass
class MarketDataset:
    prices: pd.DataFrame
    volumes: pd.DataFrame
    fundamentals: pd.DataFrame
    meta: pd.DataFrame
    start_date: datetime = field(default=None)
    end_date: datetime = field(default=None)


NON_COMPLIANT_SECTORS = {
    "Conventional Banking",
    "Conventional Insurance",
    "Gaming & Casinos",
    "Brewery",
    "Tobacco",
    "Adult Entertainment",
    "Conventional Leasing",
    "Weapons & Defense",
    "Pork / Non-Halal Food",
}

NON_COMPLIANT_INDUSTRY_KEYWORDS: dict[str, str] = {
    "bank": "Conventional Banking",
    "insurance": "Conventional Insurance",
    "credit services": "Conventional Banking",
    "capital markets": "Conventional Banking",
    "gambling": "Gaming & Casinos",
    "resorts & casinos": "Gaming & Casinos",
    "casino": "Gaming & Casinos",
    "brewers": "Brewery",
    "distillers": "Brewery",
    "wineries": "Brewery",
    "beverages - wineries": "Brewery",
    "tobacco": "Tobacco",
    "aerospace & defense": "Weapons & Defense",
    "adult": "Adult Entertainment",
}


def _assert_vocabulary_consistency() -> None:
    orphans = {v for v in NON_COMPLIANT_INDUSTRY_KEYWORDS.values()
               if v not in NON_COMPLIANT_SECTORS}
    assert not orphans, (
        f"Vocabulary mismatch: {orphans} used as canonical labels but missing "
        "from NON_COMPLIANT_SECTORS."
    )


_assert_vocabulary_consistency()


def _classify_industry(industry: str) -> str:
    ind_lower = (industry or "").lower()
    for kw in sorted(NON_COMPLIANT_INDUSTRY_KEYWORDS, key=len, reverse=True):
        if kw in ind_lower:
            return NON_COMPLIANT_INDUSTRY_KEYWORDS[kw]
    return industry or "Unknown"


_TOTAL_ASSETS_KEYS = ["Total Assets"]
_TOTAL_DEBT_KEYS = ["Total Debt"]
_LONG_TERM_DEBT_KEYS = ["Long Term Debt"]
_CURRENT_DEBT_KEYS = ["Current Debt", "Current Debt And Capital Lease Obligation"]
_CASH_KEYS = ["Cash Cash Equivalents And Short Term Investments", "Cash And Cash Equivalents"]
_RECEIVABLES_KEYS = ["Receivables", "Accounts Receivable"]


def _bs_lookup(balance_sheet: pd.DataFrame, candidates: list[str]) -> float:
    if balance_sheet is None or balance_sheet.empty:
        return np.nan
    col = balance_sheet.columns[0]
    idx_lower = {str(i).strip().lower(): i for i in balance_sheet.index}
    for cand in candidates:
        key = cand.strip().lower()
        if key in idx_lower:
            val = balance_sheet.loc[idx_lower[key], col]
            if pd.notna(val):
                return float(val)
        for lower_name, orig_name in idx_lower.items():
            if key in lower_name:
                val = balance_sheet.loc[orig_name, col]
                if pd.notna(val):
                    return float(val)
    return np.nan


def _fetch_fundamentals(ticker_obj) -> dict:
    balance_sheet = None
    try:
        balance_sheet = ticker_obj.quarterly_balance_sheet
        if balance_sheet is None or balance_sheet.empty:
            balance_sheet = ticker_obj.balance_sheet
    except Exception:
        pass

    total_assets = _bs_lookup(balance_sheet, _TOTAL_ASSETS_KEYS)
    total_debt = _bs_lookup(balance_sheet, _TOTAL_DEBT_KEYS)
    if np.isnan(total_debt):
        ltd = _bs_lookup(balance_sheet, _LONG_TERM_DEBT_KEYS)
        std = _bs_lookup(balance_sheet, _CURRENT_DEBT_KEYS)
        if not (np.isnan(ltd) and np.isnan(std)):
            total_debt = np.nansum([ltd, std])
    cash = _bs_lookup(balance_sheet, _CASH_KEYS)
    receivables = _bs_lookup(balance_sheet, _RECEIVABLES_KEYS)

    market_cap = np.nan
    try:
        market_cap = ticker_obj.fast_info.get("market_cap", np.nan)
    except Exception:
        pass
    if market_cap is None or (isinstance(market_cap, float) and np.isnan(market_cap)):
        try:
            market_cap = ticker_obj.info.get("marketCap", np.nan)
        except Exception:
            market_cap = np.nan

    return {
        "market_cap": market_cap,
        "total_assets": total_assets,
        "total_debt": total_debt,
        "cash_and_interest_securities": cash,
        "receivables": receivables,
    }


def _try_live_ingestion(tickers, start, end) -> MarketDataset | None:
    try:
        import yfinance as yf
    except ImportError:
        print("[data_ingestion] yfinance not installed.")
        return None

    price_series, volume_series, fundamentals_rows = {}, {}, []

    for t in tickers:
        tk = yf.Ticker(t)
        try:
            hist = tk.history(start=start, end=end, auto_adjust=True)
            if hist is None or hist.empty or hist["Close"].dropna().empty:
                print(f"[data_ingestion] No price history for {t}, skipping.")
                continue
            price_series[t] = hist["Close"]
            volume_series[t] = hist["Volume"]
        except Exception as e:
            print(f"[data_ingestion] Price fetch failed for {t}: {e}")
            continue

        try:
            info = tk.info
            row = _fetch_fundamentals(tk)
            row["ticker"] = t
            row["sector"] = _classify_industry(info.get("industry", info.get("sector")))
            fundamentals_rows.append(row)
        except Exception as e:
            print(f"[data_ingestion] Fundamentals fetch failed for {t}: {e}")
            fundamentals_rows.append({
                "ticker": t, "market_cap": np.nan, "total_debt": np.nan,
                "total_assets": np.nan, "cash_and_interest_securities": np.nan,
                "receivables": np.nan, "sector": "Unknown",
            })

    if len(price_series) < MIN_VALID_TICKERS:
        print(f"[data_ingestion] Only {len(price_series)} tickers returned live data "
              f"(< {MIN_VALID_TICKERS}) -> treating as failed.")
        return None

    prices = pd.DataFrame(price_series).sort_index()
    volumes = pd.DataFrame(volume_series).sort_index()
    fundamentals = pd.DataFrame(fundamentals_rows).set_index("ticker").reindex(prices.columns)
    meta = fundamentals[["sector"]].copy()

    n_missing_assets = fundamentals["total_assets"].isna().sum()
    n_missing_core = fundamentals[["total_debt", "cash_and_interest_securities",
                                    "receivables"]].isna().any(axis=1).sum()
    if n_missing_assets or n_missing_core:
        print(f"[data_ingestion] Note: {n_missing_assets}/{len(fundamentals)} tickers "
              f"missing total_assets; {n_missing_core}/{len(fundamentals)} missing "
              "debt/cash/receivables entirely.")

    return MarketDataset(prices, volumes, fundamentals, meta, start, end)


def _synthetic_universe(universe, start, end, seed: int = 42) -> MarketDataset:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end)
    n = len(dates)
    tickers = [u[0] for u in universe]
    names = {u[0]: u[1] for u in universe}
    sectors = {u[0]: u[2] for u in universe}

    prices, volumes, fundamentals_rows = {}, {}, []

    for t in tickers:
        mu = rng.uniform(0.04, 0.12) / TRADING_DAYS_PER_YEAR
        sigma = rng.uniform(0.15, 0.45) / np.sqrt(TRADING_DAYS_PER_YEAR)
        s0 = rng.uniform(1.0, 25.0)
        shocks = rng.normal(mu - 0.5 * sigma ** 2, sigma, n)
        price_path = s0 * np.exp(np.cumsum(shocks))
        prices[t] = price_path

        base_vol = rng.uniform(2e5, 8e6)
        volumes[t] = np.abs(rng.normal(base_vol, base_vol * 0.3, n)).astype(int)

        market_cap = price_path[-1] * rng.uniform(2e8, 6e9)
        is_bank_or_brewer = sectors[t] in ("Conventional Banking", "Brewery")
        debt_ratio = rng.uniform(0.35, 0.55) if is_bank_or_brewer else rng.uniform(0.05, 0.30)
        cash_ratio = rng.uniform(0.35, 0.60) if is_bank_or_brewer else rng.uniform(0.05, 0.28)
        recv_ratio = rng.uniform(0.10, 0.30)

        total_assets = market_cap * rng.uniform(0.8, 1.5)
        fundamentals_rows.append({
            "ticker": t,
            "market_cap": market_cap,
            "total_assets": total_assets,
            "total_debt": debt_ratio * total_assets,
            "cash_and_interest_securities": cash_ratio * total_assets,
            "receivables": recv_ratio * total_assets,
            "sector": sectors[t],
        })

    prices_df = pd.DataFrame(prices, index=dates)
    volumes_df = pd.DataFrame(volumes, index=dates)
    fundamentals_df = pd.DataFrame(fundamentals_rows).set_index("ticker")
    meta_df = pd.DataFrame({"name": names, "sector": sectors})
    return MarketDataset(prices_df, volumes_df, fundamentals_df, meta_df, start, end)


def ingest_market_data(universe=None, years: int = HORIZON_YEARS) -> MarketDataset:
    universe = universe or DEFAULT_UNIVERSE
    end = datetime.today()
    start = end - timedelta(days=int(years * 365.25))
    tickers = [u[0] for u in universe]

    live = _try_live_ingestion(tickers, start, end)
    if live is not None:
        print(f"[data_ingestion] Live data pulled for "
              f"{live.prices.shape[1]}/{len(tickers)} requested tickers.")
        return live

    print("[data_ingestion] No network / yfinance unavailable -> using synthetic "
          f"{years}y dataset for {len(universe)} tickers.")
    return _synthetic_universe(universe, start, end)


# ==============================================================================
# STAGE 1b  |  TECHNICAL INDICATOR ENGINEERING
# ==============================================================================

def _rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _macd(series: pd.Series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line


def _atr(close: pd.Series, window: int = 14) -> pd.Series:
    return close.diff().abs().rolling(window).mean()


def _obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direction = np.sign(close.diff().fillna(0))
    raw_obv = (direction * volume).cumsum()
    roll_mean = raw_obv.rolling(252, min_periods=60).mean()
    roll_std = raw_obv.rolling(252, min_periods=60).std()
    return (raw_obv - roll_mean) / roll_std.replace(0, np.nan)


def compute_indicators_for_ticker(close: pd.Series, volume: pd.Series) -> pd.DataFrame:
    log_ret = np.log(close / close.shift(1))

    sma10 = close.rolling(10).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line, signal_line = _macd(close)

    roll_std21 = log_ret.rolling(21).std() * np.sqrt(252)
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_width = (bb_mid + 2 * bb_std - (bb_mid - 2 * bb_std)) / bb_mid

    feats = pd.DataFrame({
        "close": close,
        "ret_1d": log_ret,
        "ret_5d": np.log(close / close.shift(5)),
        "ret_21d": np.log(close / close.shift(21)),
        "sma10": sma10, "sma50": sma50, "sma200": sma200,
        "ema12": ema12, "ema26": ema26,
        "macd": macd_line, "macd_signal": signal_line,
        "macd_hist": macd_line - signal_line,
        "rsi14": _rsi(close, 14),
        "roc10": close.pct_change(10, fill_method=None) * 100,
        "vol21_ann": roll_std21,
        "bb_width": bb_width,
        "atr14": _atr(close, 14),
        "vol_roc10": volume.pct_change(10, fill_method=None) * 100,
        "obv": _obv(close, volume),
    })
    feats["px_over_sma50"] = close / sma50 - 1
    feats["sma10_over_sma50"] = sma10 / sma50 - 1
    feats = feats.replace([np.inf, -np.inf], np.nan)
    return feats


def build_feature_panel(prices: pd.DataFrame, volumes: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for ticker in prices.columns:
        f = compute_indicators_for_ticker(prices[ticker], volumes[ticker])
        f["ticker"] = ticker
        frames.append(f)
    panel = pd.concat(frames)
    panel = panel.set_index("ticker", append=True)
    panel.index.names = ["date", "ticker"]
    return panel.sort_index()


# ==============================================================================
# STAGE 2  |  SHARIAH UNIVERSE SCREENING
# ==============================================================================

RATIO_THRESHOLDS = {
    "cash_ratio": 0.33,
    "debt_ratio": 0.33,
    "receivables_ratio": 0.50,
}


def business_activity_screen(meta: pd.DataFrame) -> pd.Series:
    return ~meta["sector"].isin(NON_COMPLIANT_SECTORS)


def financial_ratio_screen(fundamentals: pd.DataFrame,
                            denominator: str = "assets") -> pd.DataFrame:
    primary = fundamentals["total_assets"] if denominator == "assets" else fundamentals["market_cap"]
    fallback = fundamentals["market_cap"] if denominator == "assets" else fundamentals["total_assets"]
    denom = primary.where(primary.notna() & (primary != 0), fallback)

    ratios = pd.DataFrame(index=fundamentals.index)
    ratios["denominator_used"] = np.where(
        primary.notna() & (primary != 0), denominator,
        np.where(fallback.notna() & (fallback != 0), f"{denominator}_fallback", "unavailable"),
    )
    required = ["cash_and_interest_securities", "total_debt", "receivables"]
    ratios["data_available"] = fundamentals[required].notna().all(axis=1) & denom.notna() & (denom != 0)

    ratios["cash_ratio"] = fundamentals["cash_and_interest_securities"] / denom
    ratios["debt_ratio"] = fundamentals["total_debt"] / denom
    ratios["receivables_ratio"] = (fundamentals["receivables"]
                                    + fundamentals["cash_and_interest_securities"]) / denom

    ratios["pass_cash"] = ratios["data_available"] & (ratios["cash_ratio"] < RATIO_THRESHOLDS["cash_ratio"])
    ratios["pass_debt"] = ratios["data_available"] & (ratios["debt_ratio"] < RATIO_THRESHOLDS["debt_ratio"])
    ratios["pass_receivables"] = ratios["data_available"] & (ratios["receivables_ratio"] < RATIO_THRESHOLDS["receivables_ratio"])
    ratios["passes_tier2"] = ratios["data_available"] & ratios[["pass_cash", "pass_debt", "pass_receivables"]].all(axis=1)
    return ratios


def screen_universe(meta: pd.DataFrame, fundamentals: pd.DataFrame,
                     denominator: str = "assets") -> tuple[list[str], pd.DataFrame]:
    tier1 = business_activity_screen(meta)
    tier2 = financial_ratio_screen(fundamentals, denominator=denominator)

    audit = tier2.copy()
    audit["sector"] = meta["sector"]
    audit["passes_tier1"] = tier1
    audit["is_shariah_compliant"] = audit["passes_tier1"] & audit["passes_tier2"]

    n_tier1_fail = int((~tier1).sum())
    if n_tier1_fail:
        print(f"[shariah_screening] Tier 1 excluded {n_tier1_fail}/{len(audit)} tickers "
              f"on business activity: {audit.index[~tier1].tolist()}")

    compliant = audit.index[audit["is_shariah_compliant"]].tolist()
    n_missing = int((~audit["data_available"]).sum())
    if n_missing:
        print(f"[shariah_screening] {n_missing}/{len(audit)} tickers excluded due to "
              "missing fundamental data.")
    cols = ["sector", "passes_tier1", "data_available", "denominator_used",
            "cash_ratio", "pass_cash", "debt_ratio", "pass_debt",
            "receivables_ratio", "pass_receivables", "passes_tier2", "is_shariah_compliant"]
    return compliant, audit[cols]


# ==============================================================================
# STAGE 3  |  HISTORICAL RETURN ESTIMATION
# ==============================================================================
# This stage does NOT forecast. It reports per-ticker trailing statistics:
#   hist_return_ann : annualised mean of overlapping 21-day forward log returns
#   hist_vol_ann    : annualised realised vol of daily log returns
#   n_obs           : number of observations used
#
# These are what Stage 5 optimises over. If a future model is demonstrated to
# beat the naive baseline out of sample, add it as a separate module and swap
# the column names back to `exp_*`. Until then, label honestly.
# ==============================================================================

FORWARD_HORIZON = 21
MAX_ABS_ANNUAL_RETURN = 0.60
MAX_ANNUAL_VOL = 0.90
MIN_ANNUAL_VOL = 0.03
MIN_OBS = 100

ESTIMATE_COLUMNS = ["hist_return_ann", "hist_vol_ann", "n_obs", "estimate_source"]


def estimate_returns(feature_panel: pd.DataFrame,
                      compliant_tickers: list[str]) -> pd.DataFrame:
    """Per-ticker trailing historical return and volatility estimates.

    Enforces the Shariah screen at the model boundary, so no excluded ticker
    can leak into the output table.
    """
    if not compliant_tickers:
        print("[stage3] No compliant tickers — nothing to estimate.")
        return pd.DataFrame(columns=ESTIMATE_COLUMNS).rename_axis("ticker")

    panel = feature_panel.loc[
        feature_panel.index.get_level_values("ticker").isin(compliant_tickers)
    ].sort_index().copy()

    # Forward 21-day log returns, per ticker
    panel["fwd_log_return"] = (
        panel.groupby(level="ticker", sort=False)["close"]
             .transform(lambda s: np.log(s.shift(-FORWARD_HORIZON) / s))
    )
    # Realised vol (annualised), per ticker
    panel["realized_vol_ann"] = (
        panel.groupby(level="ticker", sort=False)["ret_1d"]
             .transform(lambda s: s.rolling(21).std() * np.sqrt(TRADING_DAYS_PER_YEAR))
    )

    periods_per_year = TRADING_DAYS_PER_YEAR / FORWARD_HORIZON

    rows = []
    for ticker in compliant_tickers:
        try:
            sub = panel.xs(ticker, level="ticker")
        except KeyError:
            continue
        sub = sub.dropna(subset=["fwd_log_return", "realized_vol_ann"])
        if len(sub) < MIN_OBS:
            print(f"[stage3] {ticker}: only {len(sub)} usable observations "
                  f"(< {MIN_OBS}) — skipping.")
            continue

        hist_ret = float(sub["fwd_log_return"].mean() * periods_per_year)
        hist_vol = float(sub["realized_vol_ann"].mean())
        hist_ret = float(np.clip(hist_ret, -MAX_ABS_ANNUAL_RETURN, MAX_ABS_ANNUAL_RETURN))
        hist_vol = float(np.clip(hist_vol, MIN_ANNUAL_VOL, MAX_ANNUAL_VOL))

        rows.append({
            "ticker": ticker,
            "hist_return_ann": hist_ret,
            "hist_vol_ann": hist_vol,
            "n_obs": len(sub),
            "estimate_source": "historical_5y",
        })

    if not rows:
        print("[stage3] No ticker had enough clean observations.")
        return pd.DataFrame(columns=ESTIMATE_COLUMNS).rename_axis("ticker")

    out = pd.DataFrame(rows).set_index("ticker")

    print(f"[stage3] HISTORICAL RETURN ESTIMATION — no model fitting.")
    print(f"[stage3] Per-ticker means of overlapping 21-day forward returns, "
          f"annualised over {TRADING_DAYS_PER_YEAR} trading days.")
    print(f"[stage3] {len(out)} tickers estimated. Return range: "
          f"[{out['hist_return_ann'].min():+.2%}, "
          f"{out['hist_return_ann'].max():+.2%}].")
    print(f"[stage3] Vol range: [{out['hist_vol_ann'].min():.2%}, "
          f"{out['hist_vol_ann'].max():.2%}].")

    assert set(out.index).issubset(set(compliant_tickers)), (
        f"estimate_returns leaked non-compliant tickers: "
        f"{set(out.index) - set(compliant_tickers)}"
    )

    return out


# ==============================================================================
# STAGE 3b  |  STRESS TESTING
# ==============================================================================
# The previous stress test multiplied the central return estimate by a
# fraction (0.05 for GFC, etc.). For a portfolio whose expected return is
# already low, that produces stress numbers that are still positive — which
# is useless. This version applies a σ-multiple drawdown to the portfolio's
# own volatility. The σ-multiples are calibrated from historical peak-to-
# trough equity drawdowns divided by long-run equity vol (~20%):
#   2022 rate shock: S&P -25%  -> -1.25σ
#   COVID 2020:      S&P -34%  -> -1.70σ
#   dot-com 2000:    S&P -49%  -> -2.45σ
#   GFC 2008:        S&P -55%  -> -2.75σ
# A low-vol portfolio gets a proportionally smaller drawdown, which is the
# right first-order behaviour.
# ==============================================================================

HISTORICAL_STRESS_SCENARIOS = {
    "2022_rate_shock": 1.25,
    "covid_2020":      1.70,
    "dotcom_2000":     2.45,
    "gfc_2008":        2.75,
}


def stress_test_portfolio(weights: pd.Series,
                           vols: pd.Series,
                           correlation: pd.DataFrame,
                           scenarios: dict[str, float] | None = None) -> pd.DataFrame:
    """Portfolio-level stress test using σ-multiple drawdowns.

    Parameters
    ----------
    weights : per-ticker portfolio weights (should sum to ~1)
    vols : per-ticker annualised volatilities (hist_vol_ann)
    correlation : per-ticker correlation matrix of daily returns
    scenarios : dict mapping scenario name -> σ-multiple (positive number;
                applied as a negative shock)
    """
    scenarios = scenarios or HISTORICAL_STRESS_SCENARIOS

    # Align everything on the same ticker index
    w = weights.reindex(vols.index).fillna(0.0)
    v = vols.values
    corr = correlation.reindex(index=vols.index, columns=vols.index).values

    # Portfolio vol from covariance: sigma_p = sqrt(w' (D C D) w)
    D = np.diag(v)
    cov = D @ corr @ D
    port_var = float(w.values @ cov @ w.values)
    port_vol = float(np.sqrt(max(port_var, 0.0)))

    rows = []
    for name, sigma_mult in scenarios.items():
        stressed_return = -sigma_mult * port_vol
        rows.append({
            "scenario": name,
            "sigma_multiple": sigma_mult,
            "stressed_return_ann": stressed_return,
            "portfolio_vol_ann_used": port_vol,
        })

    return pd.DataFrame(rows).set_index("scenario")


# ==============================================================================
# STAGE 4  |  INVESTOR PROFILING
# ==============================================================================

QUESTIONS = [
    {"id": "horizon", "text": "What is your investment time horizon?",
     "options": {"<1 year": 1, "1-3 years": 2, "3-7 years": 3, "7+ years": 4}},
    {"id": "loss_reaction", "text": "If your portfolio fell 20% in a month, what would you do?",
     "options": {"Sell everything immediately": 1, "Sell some to reduce risk": 2,
                 "Hold and wait it out": 3, "Buy more at the lower price": 4}},
    {"id": "income_stability", "text": "How stable is your income / need for liquidity?",
     "options": {"I may need this money soon": 1, "Stable, but I prefer safety": 2,
                 "Stable, comfortable with risk": 3, "Very stable / surplus capital": 4}},
    {"id": "experience", "text": "How would you describe your investing experience?",
     "options": {"None": 1, "Basic": 2, "Experienced": 3, "Very experienced": 4}},
    {"id": "goal", "text": "What is your primary goal?",
     "options": {"Capital preservation": 1, "Income": 2, "Balanced growth": 3, "Maximum growth": 4}},
]


@dataclass
class InvestorProfile:
    raw_score: int
    max_score: int
    risk_category: str
    lambda_risk_aversion: float
    cvar_alpha: float
    max_single_holding: float


def score_questionnaire(answers: dict[str, str]) -> InvestorProfile:
    total, max_total = 0, 0
    for q in QUESTIONS:
        max_total += max(q["options"].values())
        chosen = answers.get(q["id"])
        if chosen not in q["options"]:
            raise ValueError(f"Missing/invalid answer for '{q['id']}': {chosen!r}")
        total += q["options"][chosen]

    pct = total / max_total
    if pct < 0.40:
        category, lam, alpha, cap = "Conservative", 8.0, 0.99, 0.10
    elif pct < 0.60:
        category, lam, alpha, cap = "Moderate", 4.0, 0.97, 0.15
    elif pct < 0.80:
        category, lam, alpha, cap = "Growth", 2.0, 0.95, 0.20
    else:
        category, lam, alpha, cap = "Aggressive", 1.0, 0.90, 0.30

    return InvestorProfile(
        raw_score=total, max_score=max_total, risk_category=category,
        lambda_risk_aversion=lam, cvar_alpha=alpha, max_single_holding=cap,
    )


def run_cli_questionnaire() -> InvestorProfile:
    answers = {}
    for q in QUESTIONS:
        print(f"\n{q['text']}")
        opts = list(q["options"].keys())
        for i, opt in enumerate(opts, 1):
            print(f"  {i}. {opt}")
        choice = int(input("Choose an option number: "))
        answers[q["id"]] = opts[choice - 1]
    return score_questionnaire(answers)


# ==============================================================================
# STAGE 5  |  CONSTRAINED OPTIMISATION (MEAN-CVAR)
# ==============================================================================

ZAKAT_RATE = 0.025
TRANSACTION_COST_BPS = 15
N_SCENARIOS = 5000
RANDOM_SEED = 11


@dataclass
class OptimizationResult:
    weights: pd.Series
    expected_return_gross: float
    expected_return_net_of_costs_and_zakat: float
    cvar: float
    turnover: float
    transaction_cost: float
    zakat_due: float


def _simulate_return_scenarios(estimates: pd.DataFrame,
                                correlation: pd.DataFrame | None,
                                n_scenarios: int = N_SCENARIOS,
                                seed: int = RANDOM_SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    mu = estimates["hist_return_ann"].values
    sigma = estimates["hist_vol_ann"].values
    n_assets = len(mu)

    z = rng.standard_normal((n_scenarios, n_assets))
    if correlation is not None:
        corr = correlation.loc[estimates.index, estimates.index].values
        corr = (corr + corr.T) / 2
        eigvals, eigvecs = np.linalg.eigh(corr)
        eigvals = np.clip(eigvals, 1e-8, None)
        corr_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
        L = np.linalg.cholesky(corr_psd)
        z = z @ L.T

    return mu + z * sigma


def _portfolio_cvar(weights: np.ndarray,
                     scenario_returns: np.ndarray,
                     alpha: float) -> float:
    port_returns = scenario_returns @ weights
    var_threshold = np.percentile(port_returns, (1 - alpha) * 100)
    tail = port_returns[port_returns <= var_threshold]
    if len(tail) == 0:
        tail = np.array([var_threshold])
    return float(-tail.mean())


def optimize_portfolio(estimates: pd.DataFrame,
                        lambda_risk_aversion: float,
                        cvar_alpha: float,
                        max_single_holding: float,
                        current_weights: pd.Series | None = None,
                        correlation: pd.DataFrame | None = None) -> OptimizationResult:
    tickers = estimates.index.tolist()
    n = len(tickers)
    mu = estimates["hist_return_ann"].values

    if current_weights is None:
        current_weights = pd.Series(0.0, index=tickers)
    else:
        current_weights = current_weights.reindex(tickers).fillna(0.0)
    w0_current = current_weights.values

    scenarios = _simulate_return_scenarios(estimates, correlation)

    def objective(w):
        exp_return = mu @ w
        cvar = _portfolio_cvar(w, scenarios, cvar_alpha)
        turnover = np.sum(np.abs(w - w0_current))
        tc = (TRANSACTION_COST_BPS / 10_000) * turnover
        return -(exp_return - lambda_risk_aversion * cvar - tc)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, max_single_holding) for _ in range(n)]

    result = minimize(objective, np.full(n, 1.0 / n), method="SLSQP",
                       bounds=bounds, constraints=constraints,
                       options={"maxiter": 500, "ftol": 1e-9})

    if not result.success:
        w_final = np.clip(result.x, 0, max_single_holding)
        w_final = w_final / w_final.sum()
    else:
        w_final = np.clip(result.x, 0, max_single_holding)
        w_final = w_final / w_final.sum()

    weights = pd.Series(w_final, index=tickers, name="weight")
    exp_return_gross = float(mu @ w_final)
    cvar_final = _portfolio_cvar(w_final, scenarios, cvar_alpha)
    turnover = float(np.sum(np.abs(w_final - w0_current)))
    tc_final = (TRANSACTION_COST_BPS / 10_000) * turnover
    exp_return_net = exp_return_gross - tc_final - ZAKAT_RATE

    if cvar_final < 0:
        print(f"[optimization] WARNING: CVaR @ {cvar_alpha:.0%} = "
              f"{cvar_final:+.2%} is NEGATIVE. Historical return estimates may "
              "still be too optimistic relative to the sampled volatility.")

    return OptimizationResult(
        weights=weights.sort_values(ascending=False),
        expected_return_gross=exp_return_gross,
        expected_return_net_of_costs_and_zakat=exp_return_net,
        cvar=cvar_final,
        turnover=turnover,
        transaction_cost=tc_final,
        zakat_due=ZAKAT_RATE,
    )


# ==============================================================================
# STAGE 6  |  PIPELINE ORCHESTRATION & OUTPUT
# ==============================================================================

DEMO_ANSWERS = {
    "horizon": "7+ years",
    "loss_reaction": "Hold and wait it out",
    "income_stability": "Stable, comfortable with risk",
    "experience": "Experienced",
    "goal": "Balanced growth",
}


def run_pipeline(interactive: bool = False) -> None:
    print("=" * 70)
    print("STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)")
    print("=" * 70)
    dataset = ingest_market_data()
    feature_panel = build_feature_panel(dataset.prices, dataset.volumes)
    print(f"Ingested {dataset.prices.shape[1]} tickers x "
          f"{dataset.prices.shape[0]} trading days.")
    print(f"Feature panel: {feature_panel.shape[0]} (date,ticker) rows x "
          f"{feature_panel.shape[1]} indicators.\n")

    print("=" * 70)
    print("STAGE 2 — Shariah Universe Screening")
    print("=" * 70)
    compliant_tickers, audit = screen_universe(dataset.meta, dataset.fundamentals)
    print(f"{len(compliant_tickers)} / {len(dataset.meta)} tickers pass "
          "business-activity + financial-ratio screens:")
    print(compliant_tickers, "\n")
    if not compliant_tickers:
        print("No tickers passed the Shariah screen — stopping here.")
        return

    print("=" * 70)
    print("STAGE 3 — Historical Return Estimation")
    print("=" * 70)
    estimates = estimate_returns(feature_panel, compliant_tickers)
    if estimates.empty:
        print("No estimates produced — stopping here.")
        return

    display_cols = ["hist_return_ann", "hist_vol_ann", "n_obs", "estimate_source"]
    print()
    print(estimates.sort_values("hist_return_ann", ascending=False)[display_cols].round(4), "\n")

    print("=" * 70)
    print("STAGE 4 — Investor Profiling (Robo-Advisor)")
    print("=" * 70)
    profile = run_cli_questionnaire() if interactive else score_questionnaire(DEMO_ANSWERS)
    print(f"Risk category: {profile.risk_category}")
    print(f"  lambda (risk aversion)   = {profile.lambda_risk_aversion}")
    print(f"  CVaR confidence (alpha)  = {profile.cvar_alpha:.0%}")
    print(f"  Max single holding cap   = {profile.max_single_holding:.0%}\n")

    print("=" * 70)
    print("STAGE 5 — Constrained Optimisation (Mean-CVaR)")
    print("=" * 70)
    hist_returns = dataset.prices[estimates.index].pct_change(fill_method=None).dropna()
    correlation = hist_returns.corr()

    result = optimize_portfolio(
        estimates,
        lambda_risk_aversion=profile.lambda_risk_aversion,
        cvar_alpha=profile.cvar_alpha,
        max_single_holding=profile.max_single_holding,
        correlation=correlation,
    )

    print("=" * 70)
    print("STAGE 6 — Portfolio Output")
    print("=" * 70)
    final_weights = result.weights[result.weights > 0.005]
    report = final_weights.to_frame("weight")
    report["sector"] = dataset.meta.loc[report.index, "sector"]
    report["hist_return_ann"] = estimates.loc[report.index, "hist_return_ann"]
    report["hist_vol_ann"] = estimates.loc[report.index, "hist_vol_ann"]

    bad_in_final = report.index[report["sector"].isin(NON_COMPLIANT_SECTORS)].tolist()
    if bad_in_final:
        print(f"[pipeline] *** COMPLIANCE FAILURE *** non-compliant tickers in "
              f"final portfolio: {bad_in_final}.")

    print(report.round(4))

    # -------- Honest headline --------
    print(f"\nPortfolio-level expected return (gross, annualized): "
          f"{result.expected_return_gross:.2%}")
    print(f"Portfolio-level expected return (net of TC + zakat) : "
          f"{result.expected_return_net_of_costs_and_zakat:.2%}")
    print(f"    [source: trailing 5-year means for {len(final_weights)} holdings "
          "— NOT a forward-looking forecast]")
    print(f"Portfolio CVaR @ {profile.cvar_alpha:.0%} confidence          : "
          f"{result.cvar:.2%}")
    print(f"Turnover from current holdings                       : "
          f"{result.turnover:.2%}")
    print(f"Transaction cost drag                                : "
          f"{result.transaction_cost:.4%}")
    print(f"Annual zakat obligation (2.5% of eligible wealth)    : "
          f"{result.zakat_due:.2%}")

    # -------- Stress test (portfolio-level, σ-multiple) --------
    print("\nStress test (σ-multiple drawdowns applied to portfolio vol):")
    stress = stress_test_portfolio(
        weights=result.weights,
        vols=estimates["hist_vol_ann"],
        correlation=correlation,
    )
    print(stress.round(4))

    report.to_csv("/tmp/final_portfolio.csv")
    print("\nSaved final portfolio to /tmp/final_portfolio.csv")


if __name__ == "__main__":
    run_pipeline(interactive=False)

STAGE 1 — Data Ingestion & Preprocessing (5-Year Horizon)
[data_ingestion] Live data pulled for 25/25 requested tickers.
Ingested 25 tickers x 1227 trading days.
Feature panel: 30675 (date,ticker) rows x 21 indicators.

STAGE 2 — Shariah Universe Screening
[shariah_screening] Tier 1 excluded 6/25 tickers on business activity: ['1155.KL', '1295.KL', '3182.KL', '4715.KL', '2836.KL', '3255.KL']
13 / 25 tickers pass business-activity + financial-ratio screens:
['6033.KL', '5183.KL', '1961.KL', '4707.KL', '3689.KL', '3026.KL', '7113.KL', '5168.KL', '5285.KL', '8869.KL', '3816.KL', '5211.KL', '3336.KL'] 

STAGE 3 — Historical Return Estimation
[stage3] HISTORICAL RETURN ESTIMATION — no model fitting.
[stage3] Per-ticker means of overlapping 21-day forward returns, annualised over 252 trading days.
[stage3] 13 tickers estimated. Return range: [-32.56%, +23.80%].
[stage3] Vol range: [14.79%, 48.94%].

         hist_return_ann  hist_vol_ann  n_obs estimate_source
ticker                         